# Q1 extension of master-2: what to run, in which order
This notebook is `master-file-2` plus three things: a **session selector** (cell "PART 1", variable `SESSION`), a **long-schedule trainer** (Part 8b) and a **Q1 confirmatory analysis** (Part 9F).

| Session | What it trains | Time on a T4 | Why a Q1 reviewer cares |
|---|---|---|---|
| **A** | Restormer-FiLM-control, Restormer-LDMH-full (seeds 0-3, 30 epochs) | ~3.5 h | Does the module help the *strongest* backbone? If yes the paper stops being "a small gain on a weak network". |
| **B** | baseline, film_control, full, scale_only with seeds 4-7 (n = 8 per arm) | ~7 h | n = 4 makes the main gain borderline after multiple-testing correction; n = 8 roughly halves every interval. |
| **C** | `LONG_ARMS` for 100 epochs, 3 seeds | ~9 h (resumes) | Shows the effect is not an artefact of a short 30-epoch schedule. |
| **D** | Restormer-small, -FiLM-control, -LDMH-full, seeds 4-7 | ~5.3 h | Only if A is positive: gives n = 8 for the transfer claim. |
| **E** | nothing (analysis only) | minutes | Re-computes every table/figure + Part 9F. |

**How to run a session.** (1) Put the newest `results_latest.zip` (or its unzipped folder) as a Kaggle Input / Colab upload - Parts 2 and 2b restore it. (2) Set `SESSION` in Part 1. (3) Run All. (4) When it ends, download `results_latest.zip` (Quick Save -> Output). The next session starts from that zip. Everything finished is skipped, so you can always re-run.

**Order I recommend:** A -> look at Part 9F -> B (and D if A is positive) -> C -> E. Session E (or the end of any session) regenerates the claims ledger and Part 9F.


# MASTER-2 notebook - collapse ablation, scale-only control, real-data downstream test, second noise domain, backbone transfer, claims ledger

This notebook **extends your master notebook** (same code, same split, same protocol: 30 epochs, batch 8, seeds 0-3). It adds everything a strong revision needs.
Every number is computed from your runs; nothing is typed in. At the end a **claims ledger** (Part 12) tells you, from the data, *which sentences you are allowed to write in the paper*.

| Part | Experiment | Question it answers | Reviewer comment | Extra GPU time (T4, 26 min / run) |
|---|---|---|---|---|
| A | `full_clamp`, `full_no_bal`, `full_no_div` (+ `full_stress`, `full_clamp_stress`, + a CPU mechanism demo) | Is the collapse real, and are the balance / diversity terms and the sigmoid really what prevents it? | 3 (ablation), 6 | 12 runs = 5.2 h (+ 8 stress runs = 3.5 h) |
| B | `scale_only` (+ `scale_only_g2`, `scale_control`) | Does a *mixture of shapes* add anything beyond a *per-pixel scale* (signal-dependent noise)? | 1, 3, 5 | 4 runs = 1.8 h per arm |
| C | Downstream defect-detection benchmark on **real inspection images** with hundreds of inserted defects, a stated detector, FROC curves | Does restoration quality translate into detection (recall at a fixed false-alarm rate)? | 2, 4 | no training; ~15 min |
| D | Second noise domain (Poisson-Gaussian shot noise) | Does the module help outside the speckle domain? | 1, 5 | 16 runs = 7 h |
| E | Restormer backbone: `Restormer-FiLM-control`, `Restormer-LDMH-full`, `Restormer-LDMH-outnll`, `Restormer-LDMH-deep` | Can the module help a *strong* backbone? | 5 | 4 runs = 1.8 h per arm |
| F | Efficiency (FLOPs, CPU/GPU latency) for every arm | Is the module worth its cost? | 4, 5 | none |
| G | Claims ledger + forest plot + hand-off zip | What may the paper claim? | all | none |

**Pre-registered decision rules** (fixed here, before you look at the data - the ledger applies them mechanically):

1. *Collapse is real* only if `full_clamp_stress` has collapsed seeds (tied betas: min pairwise distance < 0.05, or a dead component: usage < 0.02) and `full_stress` (sigmoid) has none.
2. *Balance penalty needed* only if `full_no_bal` has a dead component and `full` does not. *Diversity penalty needed* only if `full_no_div` has tied betas and `full` does not.
3. *Mixture adds beyond scale* only if `full - scale_only` has a seed-level Welch p < 0.05 **and** a positive 95% CI.
4. *Module beats a capacity-matched control* only if `X - film_control` has seed-level p < 0.05 (the image-level p-values are reported but are not the decision rule - they ignore training-run variance).
5. *Transfers to a strong backbone* only if `Restormer-LDMH-* - Restormer-FiLM-control` has p < 0.05 and a positive CI.
6. *Helps downstream* only if the paired-bootstrap 95% CI of the difference in recall at 1 false alarm per image excludes 0.

If a rule fails, the paper must say so (a negative result is still a result).

**How to run (Kaggle / Colab / local).** Add your data and the newest `results_latest.zip` as inputs, turn on the GPU, edit **Part 1 only**, *Run All*. **One training group per session** (see the session plan in Part 1). After every session download `results_latest.zip` and add it as an input of the next session. First do a **smoke test**: set `SMOKE = True` (tiny synthetic data, 1 epoch) to check that every part runs, then set it back to `False`.

In [2]:
# ======================= PART 1: SETTINGS - EDIT THIS CELL ONLY =======================
SMOKE         = False      # True = tiny synthetic data + 1 epoch: checks that EVERY part runs (minutes, CPU is fine). Then set False.
DATA_PATH     = None       # KLA training data (zip or folder containing GT/ and NoisyLR/); None = search automatically
RESTORE_ZIP   = None       # results_latest.zip of earlier sessions (runs/, tables/, split_manifest.json); None = search automatically
RESULTS_NAME  = "results_pilot"   # keep the SAME name as in the earlier sessions
SEEDS         = [0, 1, 2, 3]
EPOCHS        = 30         # keep 30: it must equal the earlier runs
BATCH_SIZE    = 8
PATIENCE      = 30
SESSION_HOURS = 11.0        # clean stop (progress is saved) after this many hours
ANALYSIS_IMAGES = 400
DRIVE_DIR     = None       # Colab: "/content/drive/MyDrive/semicon_results"
VERBOSE       = True       # True = print every epoch (long logs get truncated on Kaggle; history.csv always has everything)

# ---- SESSION PLAN: what to train in THIS session (ONE group per session; finished runs are skipped) ----------------
#  S1  TRAIN_NOW = ["full_clamp", "full_no_bal", "full_no_div"]                         (12 runs, ~5.3 h)   <- Part A, the three asked arms
#  S2  TRAIN_NOW = ["scale_only", "global_gc", "full_stress", "full_clamp_stress"]      (~6.1 h)           <- Part B + stress test of the collapse claim
#  S3  TRAIN_NOW = [] ; TRAIN_NOW_D2 = ["baseline", "film_control", "full", "scale_only"] (16 runs, ~7 h)  <- Part D (second noise domain)
#  S4  TRAIN_NOW = ["Restormer-FiLM-control", "Restormer-LDMH-full", "Restormer-LDMH-outnll"]  (12 runs, ~5.4 h)  <- Part E
#      optional later: "Restormer-LDMH-deep", "full_outnll", "scale_only_g2", "scale_control"
#  S5  TRAIN_NOW = [] ; TRAIN_NOW_D2 = []   -> only run the analysis parts (C downstream, F efficiency, G ledger)
TRAIN_NOW     = ["full_clamp", "full_no_bal", "full_no_div"]      # KLA domain (speckle + Gaussian), results in runs/
TRAIN_NOW_D2  = []                                                # second domain (Poisson-Gaussian), results in runs_d2/

# ---- second noise domain (Part D): y = D( Poisson(peak * x) / peak + N(0, sigma^2) ) --------------------------------
D2_PEAK, D2_SIGMA = 40.0, 0.04

# ---- downstream defect-detection benchmark (Part C) ------------------------------------------------------------
DOWNSTREAM_DIR  = None     # folder with REAL clean SEM / inspection images (png, jpg, tif, npy). Put the Carinthia-S images here.
DS_DATASET_INFO = dict(name="", doi="", version="", url="", note="")   # FILL IN (reviewer comment 2 asks for DOI / version / provenance)
DS_N_IMAGES     = 150      # images used (>= 100 recommended)
DS_DEFECTS_PER_IMAGE = (2, 4)   # defects inserted per image (min, max) -> ~450 defects with 150 images
DS_SIZE         = 256      # images are centre-cropped to a square and resized to DS_SIZE (the model input is DS_SIZE/2)
DS_ARMS         = ["baseline", "film_control", "full", "scale_only", "Restormer-small"]   # arms with finished runs are used
DS_SEED         = 123
DS_BOOT         = 1000     # bootstrap resamples over images
FAR_TARGETS     = (0.5, 1.0, 2.0)   # false alarms per image at which recall is reported
DET = dict(smooth_sigma=1.0, min_area=6, k_min=1.2, k_max=12.0, n_k=40)   # the (stated) detector, see Part C
# =======================================================================================

# ======================= Q1 EXTENSION: WHICH SESSION ARE YOU RUNNING?  (change only this letter) =======================
#  A  Restormer transfer     : Restormer-FiLM-control + Restormer-LDMH-full, seeds 0-3, 30 epochs       ~3.5 h   <- DO THIS FIRST (decides the Q1 story)
#  B  more seeds, compact    : baseline, film_control, full, scale_only, seeds 4-7 (n becomes 8)         ~7 h     <- tightens every confidence interval
#  C  long schedule          : LONG_ARMS x LONG_SEEDS at LONG_EPOCHS epochs (separate folder runs_long)  ~9 h (resumes across sessions)
#  D  more seeds, Restormer  : Restormer-small, -FiLM-control, -LDMH-full, seeds 4-7 (only if A is positive)  ~5.3 h
#  E  analysis only          : no training; runs every analysis part incl. Part 9F (Holm / power / equivalence)
SESSION = "C"
_SESS = {
    "A": dict(TRAIN_NOW=["Restormer-FiLM-control", "Restormer-LDMH-full"], SEEDS=[0, 1, 2, 3]),
    "B": dict(TRAIN_NOW=["baseline", "film_control", "full", "scale_only"], SEEDS=[4, 5, 6, 7]),
    "C": dict(TRAIN_NOW=[], SEEDS=[0, 1, 2]),
    "D": dict(TRAIN_NOW=["Restormer-small", "Restormer-FiLM-control", "Restormer-LDMH-full"], SEEDS=[4, 5, 6, 7]),
    "E": dict(TRAIN_NOW=[], SEEDS=[0, 1, 2, 3]),
}
TRAIN_NOW, TRAIN_NOW_D2, SEEDS = list(_SESS[SESSION]["TRAIN_NOW"]), [], list(_SESS[SESSION]["SEEDS"])
# long-schedule settings (used only when SESSION == "C"); if session A shows that Restormer transfers, use the Restormer pair instead
LONG_EPOCHS = 100
LONG_SEEDS  = [0, 1, 2]
LONG_ARMS   = ["film_control", "full"]          # alternative: ["Restormer-small", "Restormer-LDMH-full"]

if SMOKE:                  # quick end-to-end check of the whole notebook on tiny synthetic data
    RESULTS_NAME, EPOCHS, SEEDS, ANALYSIS_IMAGES, SESSION_HOURS, RESTORE_ZIP = "results_smoke", 1, [0, 1], 40, 1.0, False
    TRAIN_NOW    = ["baseline", "film_control", "full", "full_clamp", "full_no_bal", "full_no_div", "full_stress", "full_clamp_stress",
                    "scale_only", "scale_control", "full_outnll", "Restormer-small", "Restormer-FiLM-control", "Restormer-LDMH-full",
                    "Restormer-LDMH-outnll", "Restormer-LDMH-deep"]
    TRAIN_NOW_D2 = ["baseline", "film_control", "full", "scale_only"]
    DS_N_IMAGES, DS_SIZE, DS_BOOT, DS_DEFECTS_PER_IMAGE = 12, 64, 100, (1, 2)
    DS_ARMS = ["baseline", "film_control", "full", "scale_only", "Restormer-small"]
    DET = dict(smooth_sigma=1.0, min_area=2, k_min=1.2, k_max=12.0, n_k=20)
print("settings OK:", dict(smoke=SMOKE, train_now=TRAIN_NOW, train_now_d2=TRAIN_NOW_D2, seeds=SEEDS, epochs=EPOCHS, batch=BATCH_SIZE, results=RESULTS_NAME, session_hours=SESSION_HOURS))

MODELS = ['baseline', 'film_control', 'full']

settings OK: {'smoke': False, 'train_now': [], 'train_now_d2': [], 'seeds': [0, 1, 2], 'epochs': 30, 'batch': 8, 'results': 'results_pilot', 'session_hours': 11.0}


## Part 2 - Find the data and prepare the working folders

In [3]:
import os, sys, io, re, glob, json, time, shutil, zipfile, warnings
warnings.filterwarnings("ignore")
IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules
BASE = "/kaggle/working" if IN_KAGGLE else ("/content" if IN_COLAB else os.path.abspath("."))
os.makedirs(BASE, exist_ok=True); os.chdir(BASE)
WORK = os.path.join(BASE, "semicon_work"); os.makedirs(WORK, exist_ok=True)
RESULTS_DIR = os.path.join(BASE, RESULTS_NAME); os.makedirs(RESULTS_DIR, exist_ok=True)

import numpy as np
from scipy import ndimage
def make_smoke_pairs(root, n=48, hr=64, looks=4.0, sigma=0.05, seed=0):
    """tiny synthetic GT / NoisyLR pairs, ONLY for SMOKE=True (checks that the code runs; the numbers mean nothing)"""
    import numpy as np
    from scipy import ndimage
    rng = np.random.RandomState(seed)
    gd, nd = os.path.join(root, "GT"), os.path.join(root, "NoisyLR")
    os.makedirs(gd, exist_ok=True); os.makedirs(nd, exist_ok=True)
    for i in range(n):
        g = ndimage.gaussian_filter(rng.rand(hr, hr), 3); g = (g - g.min()) / (g.max() - g.min() + 1e-9)
        for _ in range(rng.randint(3, 7)):
            cy, cx = rng.randint(0, hr, 2); h, w = rng.randint(hr // 12, hr // 4, 2); g[max(0, cy - h):cy + h, max(0, cx - w):cx + w] = rng.uniform(0.1, 0.95)
        g = np.clip(ndimage.gaussian_filter(g, 0.7), 0, 1).astype(np.float32)
        y = g * rng.gamma(looks, 1.0 / looks, g.shape) + rng.normal(0, sigma, g.shape)
        y = y.reshape(hr // 2, 2, hr // 2, 2).mean((1, 3)).astype(np.float32)
        np.save(os.path.join(gd, f"{i:06d}.npy"), g); np.save(os.path.join(nd, f"{i:06d}.npy"), y)
    return root
if SMOKE:
    shutil.rmtree(RESULTS_DIR, ignore_errors=True); os.makedirs(RESULTS_DIR, exist_ok=True)
    DATA_PATH = make_smoke_pairs(os.path.join(WORK, "smoke_src"))
    print("SMOKE MODE: synthetic data - the numbers mean nothing, only the code path is tested")

if RESTORE_ZIP is None:                                          # look for the results of an earlier session (added as an input)
    for r_ in (["/kaggle/input"] if IN_KAGGLE else ["/content", os.getcwd()]):
        for dp_, dn_, fn_ in os.walk(r_):
            if dp_.startswith(BASE): dn_[:] = []; continue
            if "results_latest.zip" in fn_: RESTORE_ZIP = os.path.join(dp_, "results_latest.zip")
if RESTORE_ZIP:                                                  # continue an earlier session: finished runs are skipped
    zipfile.ZipFile(RESTORE_ZIP).extractall(RESULTS_DIR)
    print("restored earlier results from", RESTORE_ZIP, "->", sorted(os.listdir(os.path.join(RESULTS_DIR, "runs"))) if os.path.isdir(os.path.join(RESULTS_DIR, "runs")) else "")
else:
    print("no earlier results found - starting fresh")

def _clean(name): return "__MACOSX" not in name and not os.path.basename(name).startswith("._")
def _npy(folder): return [f for f in glob.glob(os.path.join(folder, "*.npy")) if _clean(f)]

def find_pair_dirs(root):
    """-> list of (gt_dir, noisy_dir, n_matched) for every GT/NoisyLR sibling pair under root (largest first)"""
    out = []
    for dp, dn, fn in os.walk(root):
        if "__MACOSX" in dp: continue
        low = {d.lower(): d for d in dn}
        if "gt" in low and "noisylr" in low:
            g, n = os.path.join(dp, low["gt"]), os.path.join(dp, low["noisylr"])
            m = len({os.path.basename(p) for p in _npy(g)} & {os.path.basename(p) for p in _npy(n)})
            if m: out.append((g, n, m))
    return sorted(out, key=lambda t: -t[2])

def find_zips():
    roots = ["/kaggle/input"] if IN_KAGGLE else ["/content", "/content/drive/MyDrive", os.getcwd(), os.path.expanduser("~")]
    found = []
    for r in roots:
        if not os.path.isdir(r): continue
        for dp, dn, fn in os.walk(r):
            if dp.count(os.sep) - r.count(os.sep) > 3 or "semicon_work" in dp: dn[:] = []; continue
            found += [os.path.join(dp, f) for f in fn if f.lower().endswith(".zip")]
    return found

def zip_has_pairs(path):
    try: names = [n for n in zipfile.ZipFile(path).namelist() if _clean(n)]
    except Exception: return False
    return any("/gt/" in "/" + n.lower() for n in names) and any("/noisylr/" in "/" + n.lower() for n in names)

DATA_EXTRACT = os.path.join(WORK, "data")
search_roots = ["/kaggle/input"] if IN_KAGGLE else ["/content", os.getcwd()]
cand, DATA_SEARCH_ROOT, src_desc = [], search_roots[0], None
if DATA_PATH and os.path.isdir(DATA_PATH):                      # you gave a folder
    cand = find_pair_dirs(DATA_PATH); DATA_SEARCH_ROOT = DATA_PATH
elif DATA_PATH is None:                                          # a folder that is already unpacked (Kaggle unpacks zips itself)
    for r in search_roots:
        if os.path.isdir(r): cand += [c for c in find_pair_dirs(r) if "semicon_work" not in c[0]]
if cand:
    src_desc = "already-unpacked folder"
else:                                                            # a zip file
    zips = [DATA_PATH] if DATA_PATH else [z for z in find_zips() if zip_has_pairs(z)]
    if not zips: raise FileNotFoundError("No data found. Add your dataset / upload your zip (it must contain GT and NoisyLR folders) and/or set DATA_PATH in Part 1.")
    DATA_ZIP = zips[0]; print("using zip:", DATA_ZIP, "(other candidates:", zips[1:], ")")
    if not os.path.isdir(DATA_EXTRACT) or not find_pair_dirs(DATA_EXTRACT):
        shutil.rmtree(DATA_EXTRACT, ignore_errors=True)
        with zipfile.ZipFile(DATA_ZIP) as z:
            z.extractall(DATA_EXTRACT, members=[m for m in z.namelist() if _clean(m)])    # skip Mac __MACOSX / ._ files
    cand = find_pair_dirs(DATA_EXTRACT); src_desc = "zip"
if not cand: raise FileNotFoundError("Found data but no GT/NoisyLR folder pair with matching .npy file names. Check the folder / zip content.")
GT_DIR, NOISY_DIR, N_PAIRS = cand[0]
print(f"TRAIN PAIRS ({src_desc}): {N_PAIRS} matching files\n  GT    : {GT_DIR}\n  Noisy : {NOISY_DIR}")
if len(cand) > 1: print("  (other GT/NoisyLR pairs found and ignored:", [(c[0], c[2]) for c in cand[1:]], ")")

TEST_NOISY = None                                                # optional test folder: noisy images without ground truth (displayed only)
for dp, dn, fn in os.walk(DATA_EXTRACT if src_desc == "zip" else DATA_SEARCH_ROOT):
    if "__MACOSX" in dp: continue
    for d in dn:
        if d.lower() == "noisylr" and os.path.join(dp, d) != NOISY_DIR and "test" in dp.lower() and _npy(os.path.join(dp, d)):
            TEST_NOISY = os.path.join(dp, d)
print("TEST noisy folder (no ground truth):", TEST_NOISY, f"({len(_npy(TEST_NOISY))} files)" if TEST_NOISY else "(none found - fine)")

DATA_ROOT = os.path.join(WORK, "data_clean"); shutil.rmtree(DATA_ROOT, ignore_errors=True); os.makedirs(DATA_ROOT)
for nm, src in (("GT", GT_DIR), ("NoisyLR", NOISY_DIR)):          # clean view with the layout the code expects: <root>/GT and <root>/NoisyLR
    os.makedirs(os.path.join(DATA_ROOT, nm))
    for f in _npy(src): os.symlink(f, os.path.join(DATA_ROOT, nm, os.path.basename(f)))

print("results folder:", RESULTS_DIR)

no earlier results found - starting fresh
TRAIN PAIRS (already-unpacked folder): 3200 matching files
  GT    : /kaggle/input/datasets/abhi9135445880/knowhow/train/train/GT
  Noisy : /kaggle/input/datasets/abhi9135445880/knowhow/train/train/NoisyLR
TEST noisy folder (no ground truth): /kaggle/input/datasets/abhi9135445880/knowhow/Test_NoisyLR/NoisyLR (400 files)
results folder: /kaggle/working/results_pilot


### 2b  Restore earlier results (also when Kaggle has already unpacked your zip) and report what was found
If `results_latest.zip` was auto-unpacked by Kaggle, the folder with `runs/` and `split_manifest.json` is copied into the results folder. The table shows which arms/seeds are finished (they are skipped) and which will be trained.

In [4]:
_runs_dir = os.path.join(RESULTS_DIR, "runs")
_have_runs = os.path.isdir(_runs_dir) and len(os.listdir(_runs_dir)) > 0
if not _have_runs and not SMOKE:
    _roots = ["/kaggle/input"] if IN_KAGGLE else ["/content", os.getcwd()]
    _src = []
    for _r in _roots:
        _src += [os.path.dirname(q) for q in glob.glob(os.path.join(_r, "**", "split_manifest.json"), recursive=True) if not q.startswith(RESULTS_DIR)]
    if _src:
        shutil.copytree(_src[0], RESULTS_DIR, dirs_exist_ok=True); print("restored from unpacked folder:", _src[0])
    else:
        print("no earlier results found anywhere - everything will be trained from scratch")
print("\nEARLIER RESULTS FOUND IN", RESULTS_DIR)
if os.path.isdir(_runs_dir):
    for _a in sorted(os.listdir(_runs_dir)):
        _s = sorted(d for d in os.listdir(os.path.join(_runs_dir, _a)) if d.startswith("seed") and os.path.exists(os.path.join(_runs_dir, _a, d, "best.pth")))
        print(f"  {_a:28s} finished seeds with checkpoint: {_s}")
else:
    print("  (none)")
print("Arms not listed above are NOT available and must be trained (TRAIN_NOW in Part 1 / the training parts).")

restored from unpacked folder: /kaggle/input/datasets/abhi9135445880/result-8

EARLIER RESULTS FOUND IN /kaggle/working/results_pilot
  NAFNet-small                 finished seeds with checkpoint: ['seed0', 'seed1', 'seed2', 'seed3']
  Restormer-FiLM-control       finished seeds with checkpoint: ['seed0', 'seed1', 'seed2', 'seed3']
  Restormer-LDMH-full          finished seeds with checkpoint: ['seed0', 'seed1', 'seed2', 'seed3']
  Restormer-small              finished seeds with checkpoint: ['seed0', 'seed1', 'seed2', 'seed3']
  SwinIR-small                 finished seeds with checkpoint: ['seed0', 'seed1', 'seed2', 'seed3']
  baseline                     finished seeds with checkpoint: ['seed0', 'seed1', 'seed2', 'seed3', 'seed4', 'seed5', 'seed6', 'seed7']
  film_control                 finished seeds with checkpoint: ['seed0', 'seed1', 'seed2', 'seed3', 'seed4', 'seed5', 'seed6', 'seed7']
  full                         finished seeds with checkpoint: ['seed0', 'seed1', 'seed2', 'se

## Part 3 - ALL CODE: architectures, losses, metrics, data pipeline, statistics, training loop
Everything is written **directly in the cells below** (no helper files, no imports from other files). Run them top to bottom once.

### Where is each of the four models?

| # | Model | Network class (cell) | Loss | Entry in `CONFIGS` (cell 3.5) |
|---|---|---|---|---|
| 1 | `baseline` | `BaselineNet` -> **cell 3.1c** (uses the backbone of cell 3.1a) | Charbonnier | `{'arch': 'baseline', 'loss': 'charbonnier'}` |
| 2 | `global_gc` | `BaselineNet` -> **cell 3.1c** (same network, different loss) | generalized Charbonnier (p = 0.845) | `{'arch': 'baseline', 'loss': 'gen_charbonnier'}` |
| 3 | `film_control` | `DistributionMixtureRestorationNet` -> **cell 3.1c** (uses the LDMH of cell 3.1b) | Charbonnier only (control: capacity without the mixture NLL) | `{'arch': 'ldmh', 'film': True, 'loss': 'charbonnier'}` |
| 4 | `full` | `DistributionMixtureRestorationNet` -> **cell 3.1c** | mixture NLL + diversity + balance penalties | `{'arch': 'ldmh', 'film': True, 'loss': 'mixture', ...}` |

Cell order: 3.0 imports -> 3.1 architecture (a backbone, b LDMH, c the networks, d external baselines) -> 3.2 losses and metrics -> 3.3 data, degradation, distribution fitting, statistics, figures -> 3.4 configuration -> 3.5 training loop and the four model definitions -> 3.6 architecture summary.

### 3.0 Imports and small helpers

In [5]:
import math, random, csv, hashlib, types
import numpy as np, pandas as pd, torch, matplotlib
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from scipy import stats, ndimage, optimize

def _ns(before):
    # collect the names defined since `before` into a namespace, so that M.psnr(...), D.patch_table(...), FG.save(...) work
    return types.SimpleNamespace(**{k: v for k, v in globals().items() if k not in before and not k.startswith("_")})
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

torch 2.11.0+cu130 | cuda available: True


### 3.1a  Architecture - shared backbone (used by ALL four models)
`NAFBlockLite` and `RestorationBackboneLite`: stem -> 4 LR blocks -> PixelShuffle x2 -> 2 HR blocks -> head + bilinear global residual.

In [6]:
"""
Lightweight NAFNet-style backbone (custom, compact re-implementation of the
core NAFNet block design -- NOT the official repo). Operates at the
degraded (LR) input resolution for a few blocks, then upsamples x2 via
PixelShuffle to the GT (HR) resolution, matching the 128->256 (or 256->512)
super-resolution factor in the KLA task.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class SimpleGate(nn.Module):
    """Splits channels in half and multiplies -- NAFNet's activation-free nonlinearity."""
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class NAFBlockLite(nn.Module):
    """Compact NAFNet-style block: depthwise conv + SimpleGate + simplified channel attention."""
    def __init__(self, ch, expand=2):
        super().__init__()
        hidden = ch * expand
        self.norm1 = nn.GroupNorm(1, ch)  # acts as LayerNorm over channels
        self.conv1 = nn.Conv2d(ch, hidden, 1)
        self.dwconv = nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden)
        self.gate = SimpleGate()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(hidden // 2, hidden // 2, 1),
        )
        self.conv2 = nn.Conv2d(hidden // 2, ch, 1)

        self.norm2 = nn.GroupNorm(1, ch)
        self.conv3 = nn.Conv2d(ch, hidden, 1)
        self.gate2 = SimpleGate()
        self.conv4 = nn.Conv2d(hidden // 2, ch, 1)

        self.alpha1 = nn.Parameter(torch.zeros(1, ch, 1, 1))
        self.alpha2 = nn.Parameter(torch.zeros(1, ch, 1, 1))

    def forward(self, x):
        y = self.norm1(x)
        y = self.conv1(y)
        y = self.dwconv(y)
        y = self.gate(y)
        y = y * self.se(y)
        y = self.conv2(y)
        x = x + self.alpha1 * y

        y = self.norm2(x)
        y = self.conv3(y)
        y = self.gate2(y)
        y = self.conv4(y)
        x = x + self.alpha2 * y
        return x


class RestorationBackboneLite(nn.Module):
    """
    stem -> [optional FiLM] -> N LR-resolution NAFBlocks -> PixelShuffle x2
         -> M HR-resolution NAFBlocks -> head, with a global bilinear-upsample residual.
    Accepts optional external (gamma, shift) FiLM tensors (same spatial size as
    the LR feature map) so a noise-statistics head (e.g. LDMH) can condition it.
    """
    def __init__(self, base_ch=32, n_lr_blocks=4, n_hr_blocks=2, scale=2):
        super().__init__()
        self.scale = scale
        self.stem = nn.Conv2d(1, base_ch, 3, padding=1)
        self.lr_blocks = nn.ModuleList([NAFBlockLite(base_ch) for _ in range(n_lr_blocks)])
        self.upsample = nn.Sequential(
            nn.Conv2d(base_ch, base_ch * scale * scale, 3, padding=1),
            nn.PixelShuffle(scale),
        )
        self.hr_blocks = nn.ModuleList([NAFBlockLite(base_ch) for _ in range(n_hr_blocks)])
        self.head = nn.Conv2d(base_ch, 1, 3, padding=1)

    def forward(self, x, film_gamma=None, film_shift=None):
        feat = self.stem(x)
        if film_gamma is not None:
            feat = feat * (1 + film_gamma) + film_shift
        for blk in self.lr_blocks:
            feat = blk(feat)
        feat = self.upsample(feat)
        for blk in self.hr_blocks:
            feat = blk(feat)
        out = self.head(feat)
        x_up = F.interpolate(x, scale_factor=self.scale, mode="bilinear", align_corners=False)
        return x_up + out

### 3.1b  Architecture - Local Distribution Mixture Head (LDMH)  [models 3 and 4]
Per-pixel softmax weights over 3 shared generalized-Gaussian prototypes (beta_k, alpha_k), temperature T = 1.5, sigmoid-bounded beta, mixture NLL, beta-diversity and load-balance penalties, FiLM projector.

In [7]:
"""
Local Distribution Mixture Head (LDMH) -- v2.

Changes vs. the submitted version (each one addresses a finding of the audit):
  * beta_mode: 'sigmoid' (corrected, default) or 'clamp' (the original, buggy
    parameterisation) so the collapse claim can be tested *experimentally*
    instead of asserted.
  * init_scale: the prototype scales are initialised near the empirical residual
    scale (the old softplus(0)=0.69 is ~15x larger than the residual std).
  * NLL no longer uses `(r/scale).clamp(min=eps)`: that clamp is conceptually the same defect
    (zero gradient inside |r| < eps*scale) although its measure is negligible in practice
    (see notebook 02).  Replaced by an additive eps; the NLL is evaluated in float32 outside
    autocast (lgamma/pow are not safe in fp16).
  * `param_groups()` exposes the prototype parameters so weight decay is NOT
    applied to log_beta / log_scale.
  * `beta_hit_bound` diagnostic: reports whether a beta sits against beta_max /
    beta_min (sigmoid saturation), which the original paper did not check.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class LocalDistributionMixtureHead(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, n_components=3, beta_min=0.3, beta_max=2.5,
                 temperature=1.5, beta_mode="sigmoid", init_scale=0.05):
        super().__init__()
        assert beta_mode in ("sigmoid", "clamp")
        self.K = n_components
        self.beta_min, self.beta_max = beta_min, beta_max
        self.T = temperature
        self.beta_mode = beta_mode
        self.encoder = nn.Sequential(
            nn.Conv2d(in_ch, base_ch, 3, padding=1), nn.GroupNorm(8, base_ch), nn.GELU(),
            nn.Conv2d(base_ch, base_ch, 3, padding=1), nn.GroupNorm(8, base_ch), nn.GELU(),
        )
        self.weight_head = nn.Conv2d(base_ch, n_components, 1)

        init_betas = torch.linspace(beta_min + 0.1, beta_max - 0.1, n_components)
        if beta_mode == "sigmoid":
            frac = (init_betas - beta_min) / (beta_max - beta_min)
            self.log_beta = nn.Parameter(torch.logit(frac))
        else:  # original: beta = clamp(exp(l))
            self.log_beta = nn.Parameter(torch.log(init_betas))
        # inverse softplus so that softplus(log_scale) + 1e-3 == init_scale
        s = max(init_scale - 1e-3, 1e-4)
        self.log_scale = nn.Parameter(torch.full((n_components,), math.log(math.expm1(s))))

    # ------------------------------------------------------------------
    def betas(self):
        if self.beta_mode == "sigmoid":
            return self.beta_min + (self.beta_max - self.beta_min) * torch.sigmoid(self.log_beta)
        return torch.exp(self.log_beta).clamp(self.beta_min, self.beta_max)

    def forward(self, x):
        feat = self.encoder(x)
        logits = self.weight_head(feat)
        mix_weights = F.softmax(logits / self.T, dim=1)       # Eq. (3): temperature T
        beta = self.betas()
        scale = F.softplus(self.log_scale) + 1e-3
        return mix_weights, beta, scale, feat

    def proto_params(self):
        return [self.log_beta, self.log_scale]

    @torch.no_grad()
    def get_component_stats(self, mix_weights, beta, scale):
        K = beta.shape[0]
        stats = {"beta": beta.detach().cpu().tolist(), "scale": scale.detach().cpu().tolist()}
        stats["mean_usage"] = mix_weights.mean(dim=(0, 2, 3)).detach().cpu().tolist()
        dom = mix_weights.argmax(dim=1)
        stats["dominant_frac"] = (torch.bincount(dom.flatten(), minlength=K).float() / dom.numel()).cpu().tolist()
        stats["min_beta_pairwise_dist"] = torch.pdist(beta.unsqueeze(1)).min().item() if K > 1 else float("nan")
        stats["beta_hit_bound"] = [bool((b - self.beta_min) < 0.01 or (self.beta_max - b) < 0.01) for b in beta.tolist()]
        return stats


def mixture_gen_gaussian_nll(residual, mix_weights, beta, scale, eps=1e-6):
    """Mixture of zero-mean generalised Gaussians, evaluated in fp32."""
    with torch.autocast(device_type=residual.device.type, enabled=False):
        residual, mix_weights = residual.float(), mix_weights.float()
        beta, scale = beta.float(), scale.float()
        K = beta.shape[0]
        r = residual.abs()
        if r.dim() == 3:
            r = r.unsqueeze(1)
        b = beta.view(1, K, 1, 1)
        s = scale.view(1, K, 1, 1)
        log_norm = torch.log(b) - torch.log(2 * s) - torch.lgamma(1.0 / b)
        log_comp = log_norm - ((r + eps) / s).pow(b)          # no gradient-killing clamp
        log_mix = torch.logsumexp(torch.log(mix_weights + eps) + log_comp, dim=1)
        return -log_mix.mean()


def beta_diversity_penalty(beta, margin=0.3):
    if beta.shape[0] < 2:
        return beta.new_zeros(())
    return F.relu(margin - torch.pdist(beta.unsqueeze(1))).mean()


def load_balance_penalty_entropy(mix_weights):
    K = mix_weights.shape[1]
    u = mix_weights.float().mean(dim=(0, 2, 3)).clamp(min=1e-8)
    entropy = -(u * u.log()).sum()
    return math.log(K) - entropy


class MixtureFiLMProjector(nn.Module):
    def __init__(self, n_components, feat_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(n_components, feat_ch, 3, padding=1), nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch * 2, 3, padding=1),
        )

    def forward(self, mix_weights):
        gamma, shift = self.net(mix_weights).chunk(2, dim=1)
        return gamma, shift

### 3.1c  Architecture - THE FOUR NETWORKS  [`BaselineNet` = models 1 and 2;  `DistributionMixtureRestorationNet` = models 3 and 4]

In [8]:
"""
Model zoo for the ablation ladder (all share the SAME backbone):

  baseline        BaselineNet                                  -- backbone only
  film_control    DistributionMixtureRestorationNet(use_film=True, trained WITHOUT mixture NLL)
                  -> same extra parameters / FLOPs as the full model, but no
                     distribution modelling.  This is the missing capacity-matched control.
  ldmh_loss_only  DistributionMixtureRestorationNet(use_film=False)
                  -> NOTE: backbone never sees the LDMH output, so this is *functionally the
                     baseline* plus a disconnected auxiliary head.  Kept only as a seed-noise
                     sanity check (it must be statistically indistinguishable from baseline).
  full            DistributionMixtureRestorationNet(use_film=True) + mixture NLL (+ penalties)
"""
import torch.nn as nn


class BaselineNet(nn.Module):
    def __init__(self, base_ch=32, n_lr_blocks=4, n_hr_blocks=2, scale=2):
        super().__init__()
        self.backbone = RestorationBackboneLite(base_ch, n_lr_blocks, n_hr_blocks, scale)

    def forward(self, x):
        return self.backbone(x, film_gamma=None, film_shift=None)


class DistributionMixtureRestorationNet(nn.Module):
    def __init__(self, base_ch=32, n_components=3, n_lr_blocks=4, n_hr_blocks=2, scale=2,
                 use_film=True, beta_mode="sigmoid", init_scale=0.05, temperature=1.5):
        super().__init__()
        self.use_film = use_film
        self.ldmh = LocalDistributionMixtureHead(1, base_ch, n_components, beta_mode=beta_mode,
                                                 init_scale=init_scale, temperature=temperature)
        self.film_proj = MixtureFiLMProjector(n_components, base_ch)
        self.backbone = RestorationBackboneLite(base_ch, n_lr_blocks, n_hr_blocks, scale)

    def forward(self, x):
        mix_weights, beta, scale, _ = self.ldmh(x)
        if self.use_film:
            gamma, shift = self.film_proj(mix_weights)
        else:
            gamma, shift = None, None
        return self.backbone(x, gamma, shift), mix_weights, beta, scale

### 3.1d  External baselines (NAFNet / Restormer / SwinIR re-implementations)
Not trained in this notebook - they are registered here only because the training registry refers to them; they are trained later (VS Code, notebook 05).

In [9]:
"""
Compact re-implementations of the external restoration architectures requested by the
reviewer (NAFNet, Restormer, SwinIR), adapted to this task: 1-channel LR input
[B,1,H,W] -> 1-channel x2 output [B,1,2H,2W] with a global bilinear residual and a
PixelShuffle upsampler.

IMPORTANT (state this in the paper): these are faithful-in-design but *re-implemented,
size-reduced* versions written for this repository, NOT the official code/weights.
They are trained from scratch with exactly the same split, augmentation, optimiser,
schedule, budget and checkpoint-selection rule as the proposed model.  For a
camera-ready comparison you may swap in the official repos through BASELINE_REGISTRY
(same call signature), and report both.

Sizes are chosen so that 'small' ~ comparable to the proposed model's budget and
'base' is a stronger, larger reference; parameters / FLOPs are always reported.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- common
class LayerNorm2d(nn.Module):
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(c)); self.b = nn.Parameter(torch.zeros(c)); self.eps = eps

    def forward(self, x):
        mu = x.mean(1, keepdim=True)
        var = (x - mu).pow(2).mean(1, keepdim=True)
        x = (x - mu) / torch.sqrt(var + self.eps)
        return x * self.w.view(1, -1, 1, 1) + self.b.view(1, -1, 1, 1)


class _SRWrap(nn.Module):
    """stem -> body(LR) -> PixelShuffle x2 -> head, + bilinear global residual."""
    def __init__(self, body, width, scale=2):
        super().__init__()
        self.scale = scale
        self.stem = nn.Conv2d(1, width, 3, padding=1)
        self.body = body
        self.up = nn.Sequential(nn.Conv2d(width, width * scale * scale, 3, padding=1), nn.PixelShuffle(scale))
        self.head = nn.Conv2d(width, 1, 3, padding=1)

    def forward(self, x):
        f = self.stem(x)
        f = self.body(f)
        out = self.head(F.gelu(self.up(f)))
        return F.interpolate(x, scale_factor=self.scale, mode="bilinear", align_corners=False) + out


# ----------------------------------------------------------------------------- NAFNet
class _NAFBlock(nn.Module):
    """Official NAFNet block (LayerNorm2d, SimpleGate, Simplified Channel Attention)."""
    def __init__(self, c, dw_expand=2, ffn_expand=2):
        super().__init__()
        dw = c * dw_expand
        self.n1 = LayerNorm2d(c)
        self.c1 = nn.Conv2d(c, dw, 1); self.c2 = nn.Conv2d(dw, dw, 3, padding=1, groups=dw)
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(dw // 2, dw // 2, 1))
        self.c3 = nn.Conv2d(dw // 2, c, 1)
        ffn = c * ffn_expand
        self.n2 = LayerNorm2d(c)
        self.c4 = nn.Conv2d(c, ffn, 1); self.c5 = nn.Conv2d(ffn // 2, c, 1)
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1)); self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    @staticmethod
    def _gate(x):
        a, b = x.chunk(2, 1); return a * b

    def forward(self, x):
        y = self._gate(self.c2(self.c1(self.n1(x))))
        y = self.c3(y * self.sca(y))
        x = x + y * self.beta
        y = self.c5(self._gate(self.c4(self.n2(x))))
        return x + y * self.gamma


class _NAFUNet(nn.Module):
    def __init__(self, w, enc=(1, 1, 2), mid=2, dec=(1, 1, 1)):
        super().__init__()
        self.enc, self.dec, self.down, self.up = nn.ModuleList(), nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c = w
        for n in enc:
            self.enc.append(nn.Sequential(*[_NAFBlock(c) for _ in range(n)]))
            self.down.append(nn.Conv2d(c, 2 * c, 2, 2)); c *= 2
        self.mid = nn.Sequential(*[_NAFBlock(c) for _ in range(mid)])
        for n in dec:
            self.up.append(nn.Sequential(nn.Conv2d(c, c * 2, 1, bias=False), nn.PixelShuffle(2))); c //= 2
            self.dec.append(nn.Sequential(*[_NAFBlock(c) for _ in range(n)]))
        self.levels = len(enc)

    def forward(self, x):
        H, W = x.shape[-2:]
        m = 2 ** self.levels
        ph, pw = (m - H % m) % m, (m - W % m) % m
        x = F.pad(x, (0, pw, 0, ph), mode="reflect") if (ph or pw) else x
        skips = []
        for e, d in zip(self.enc, self.down):
            x = e(x); skips.append(x); x = d(x)
        x = self.mid(x)
        for u, d, s in zip(self.up, self.dec, reversed(skips)):
            x = d(u(x) + s)
        return x[..., :H, :W]


def NAFNetSR(size="small"):
    w, enc, mid, dec = {"small": (16, (1, 1), 2, (1, 1)), "base": (24, (1, 1), 2, (1, 1))}[size]
    return _SRWrap(_NAFUNet(w, enc, mid, dec), w)


# ----------------------------------------------------------------------------- Restormer
class _MDTA(nn.Module):
    def __init__(self, c, heads):
        super().__init__()
        self.h = heads; self.t = nn.Parameter(torch.ones(heads, 1, 1))
        self.qkv = nn.Conv2d(c, c * 3, 1, bias=False)
        self.dw = nn.Conv2d(c * 3, c * 3, 3, padding=1, groups=c * 3, bias=False)
        self.out = nn.Conv2d(c, c, 1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.dw(self.qkv(x)).chunk(3, 1)
        q, k, v = [t.reshape(B, self.h, C // self.h, H * W) for t in (q, k, v)]
        q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        a = (q @ k.transpose(-2, -1)) * self.t
        return self.out((a.softmax(-1) @ v).reshape(B, C, H, W))


class _GDFN(nn.Module):
    def __init__(self, c, e=2.0):
        super().__init__()
        h = int(c * e)
        self.pi = nn.Conv2d(c, h * 2, 1, bias=False)
        self.dw = nn.Conv2d(h * 2, h * 2, 3, padding=1, groups=h * 2, bias=False)
        self.po = nn.Conv2d(h, c, 1, bias=False)

    def forward(self, x):
        a, b = self.dw(self.pi(x)).chunk(2, 1)
        return self.po(F.gelu(a) * b)


class _RBlock(nn.Module):
    def __init__(self, c, heads):
        super().__init__()
        self.n1, self.n2 = LayerNorm2d(c), LayerNorm2d(c)
        self.attn, self.ffn = _MDTA(c, heads), _GDFN(c)

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.ffn(self.n2(x))


def RestormerSR(size="small"):
    w, heads, n = {"small": (32, 2, 6), "base": (48, 4, 8)}[size]
    return _SRWrap(nn.Sequential(*[_RBlock(w, heads) for _ in range(n)]), w)


# ----------------------------------------------------------------------------- SwinIR
def _win_part(x, ws):
    B, H, W, C = x.shape
    x = x.view(B, H // ws, ws, W // ws, ws, C)
    return x.permute(0, 1, 3, 2, 4, 5).reshape(-1, ws * ws, C)


def _win_rev(w, ws, H, W):
    B = w.shape[0] // ((H // ws) * (W // ws))
    x = w.view(B, H // ws, W // ws, ws, ws, -1)
    return x.permute(0, 1, 3, 2, 4, 5).reshape(B, H, W, -1)


class _SwinBlock(nn.Module):
    def __init__(self, c, heads, ws, shift):
        super().__init__()
        self.ws, self.shift, self.h = ws, shift, heads
        self.n1, self.n2 = nn.LayerNorm(c), nn.LayerNorm(c)
        self.qkv, self.proj = nn.Linear(c, 3 * c), nn.Linear(c, c)
        self.mlp = nn.Sequential(nn.Linear(c, 2 * c), nn.GELU(), nn.Linear(2 * c, c))
        self.rpb = nn.Parameter(torch.zeros((2 * ws - 1) ** 2, heads))
        nn.init.trunc_normal_(self.rpb, std=0.02)
        co = torch.stack(torch.meshgrid(torch.arange(ws), torch.arange(ws), indexing="ij")).flatten(1)
        rel = (co[:, :, None] - co[:, None, :]).permute(1, 2, 0) + (ws - 1)
        self.register_buffer("rpi", rel[..., 0] * (2 * ws - 1) + rel[..., 1], persistent=False)

    def _mask(self, H, W, dev):
        ws, s = self.ws, self.shift
        img = torch.zeros(1, H, W, 1, device=dev); cnt = 0
        for hs in (slice(0, -ws), slice(-ws, -s), slice(-s, None)):
            for wsl in (slice(0, -ws), slice(-ws, -s), slice(-s, None)):
                img[:, hs, wsl, :] = cnt; cnt += 1
        mw = _win_part(img, ws).squeeze(-1)
        m = mw[:, None, :] - mw[:, :, None]
        return m.masked_fill(m != 0, -100.0).masked_fill(m == 0, 0.0)

    def forward(self, x, H, W):          # x: [B,H*W,C]
        B, L, C = x.shape
        y = self.n1(x).view(B, H, W, C)
        if self.shift:
            y = torch.roll(y, (-self.shift, -self.shift), (1, 2))
        w = _win_part(y, self.ws)
        N = self.ws * self.ws
        qkv = self.qkv(w).reshape(-1, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0] * (C // self.h) ** -0.5, qkv[1], qkv[2]
        a = q @ k.transpose(-2, -1)
        a = a + self.rpb[self.rpi.view(-1)].view(N, N, -1).permute(2, 0, 1).unsqueeze(0)
        if self.shift:
            m = self._mask(H, W, x.device)
            nW = m.shape[0]
            a = (a.view(-1, nW, self.h, N, N) + m.unsqueeze(1).unsqueeze(0).to(a.dtype)).view(-1, self.h, N, N)
        o = (a.softmax(-1) @ v).transpose(1, 2).reshape(-1, N, C)
        y = _win_rev(self.proj(o), self.ws, H, W)
        if self.shift:
            y = torch.roll(y, (self.shift, self.shift), (1, 2))
        x = x + y.reshape(B, L, C)
        return x + self.mlp(self.n2(x))


class _SwinBody(nn.Module):
    def __init__(self, c, heads, depth, ws):
        super().__init__()
        self.ws = ws
        self.blocks = nn.ModuleList([_SwinBlock(c, heads, ws, 0 if i % 2 == 0 else ws // 2) for i in range(depth)])
        self.conv = nn.Conv2d(c, c, 3, padding=1)

    def forward(self, f):
        B, C, H, W = f.shape
        ph, pw = (self.ws - H % self.ws) % self.ws, (self.ws - W % self.ws) % self.ws
        g = F.pad(f, (0, pw, 0, ph), mode="reflect") if (ph or pw) else f
        Hp, Wp = g.shape[-2:]
        t = g.flatten(2).transpose(1, 2)
        for b in self.blocks:
            t = b(t, Hp, Wp)
        t = t.transpose(1, 2).reshape(B, C, Hp, Wp)[..., :H, :W]
        return f + self.conv(t)          # residual Swin transformer block (RSTB) style


def SwinIRSR(size="small", window=8):
    c, heads, depth = {"small": (32, 2, 6), "base": (48, 4, 8)}[size]
    return _SRWrap(_SwinBody(c, heads, depth, window), c)


BASELINE_REGISTRY = {
    "NAFNet-small": lambda: NAFNetSR("small"), "NAFNet-base": lambda: NAFNetSR("base"),
    "Restormer-small": lambda: RestormerSR("small"), "Restormer-base": lambda: RestormerSR("base"),
    "SwinIR-small": lambda: SwinIRSR("small"), "SwinIR-base": lambda: SwinIRSR("base"),
}

### 3.2a  Losses (Charbonnier, generalized Charbonnier)

In [10]:
"""Fixed (non-adaptive) reconstruction losses used as baselines."""
import torch


def charbonnier_loss(pred, target, eps=1e-3):
    return torch.sqrt((pred - target) ** 2 + eps ** 2).mean()


def gen_charbonnier_loss(pred, target, p=0.845, eps=1e-3):
    return ((pred - target) ** 2 + eps ** 2).pow(p / 2).mean()

### 3.2b  Metrics - PSNR, Gaussian-window SSIM, HP-Err, edge correlation / F1, flat-region error, CNR  (namespace `M`)

In [11]:
_before = set(globals())
"""
Metrics (v2).  All functions are per-image (return a tensor [B]) so that paired statistics
can be computed on the test set.

  * psnr / ssim         : SSIM is the standard Gaussian-window (11x11, sigma 1.5) of Wang et al. [25].
                          (the submitted code used a 7x7 *uniform* window -> not what the paper cites)
  * hp_err              : MAE on the top-q% highest-|residual| pixels (full-scale, per image)
  * edge_preservation   : normalised cross-correlation of Sobel gradient magnitudes (restored vs clean)
  * edge_f1             : F1 of thresholded edges (threshold = percentile of the clean edge map)
  * artifact_energy     : mean |restored - clean| on FLAT regions of the clean image (over-smoothing /
                          ringing is visible at edges, residual noise and texture artefacts in flats)
  * cnr                 : contrast-to-noise ratio of a region of interest vs. its surround
"""
import numpy as np
import torch
import torch.nn.functional as F


def psnr(pred, gt, data_range=1.0):
    mse = ((pred - gt) ** 2).flatten(1).mean(1).clamp(min=1e-10)
    return 10 * torch.log10(data_range ** 2 / mse)


def _gauss(win=11, sigma=1.5, device="cpu"):
    ax = torch.arange(win, dtype=torch.float32, device=device) - (win - 1) / 2
    g = torch.exp(-ax ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    return (g[:, None] * g[None, :])[None, None]


def ssim(pred, gt, data_range=1.0, win=11, sigma=1.5):
    k = _gauss(win, sigma, pred.device)
    C1, C2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    f = lambda x: F.conv2d(x, k)           # 'valid' window, as in skimage / Wang et al.
    mx, my = f(pred), f(gt)
    sxx, syy, sxy = f(pred * pred) - mx ** 2, f(gt * gt) - my ** 2, f(pred * gt) - mx * my
    s = ((2 * mx * my + C1) * (2 * sxy + C2)) / ((mx ** 2 + my ** 2 + C1) * (sxx + syy + C2))
    return s.flatten(1).mean(1)


def hp_err(pred, gt, noisy_lr, q=0.05):
    """MAE on the top-q fraction of pixels where the bilinear-upsampled input is worst."""
    up = F.interpolate(noisy_lr, size=gt.shape[-2:], mode="bilinear", align_corners=False)
    base = (up - gt).abs().flatten(1)
    err = (pred - gt).abs().flatten(1)
    k = max(1, int(q * base.shape[1]))
    idx = torch.topk(base, k, dim=1).indices
    return torch.gather(err, 1, idx).mean(1)


def _sobel_mag(x):
    kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=x.dtype, device=x.device)[None, None]
    gx, gy = F.conv2d(x, kx, padding=1), F.conv2d(x, kx.transpose(-1, -2), padding=1)
    return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)


def edge_preservation(pred, gt):
    a, b = _sobel_mag(pred).flatten(1), _sobel_mag(gt).flatten(1)
    a, b = a - a.mean(1, keepdim=True), b - b.mean(1, keepdim=True)
    return (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-12)


def edge_f1(pred, gt, pct=90):
    mg, mp = _sobel_mag(gt).flatten(1), _sobel_mag(pred).flatten(1)
    th = torch.quantile(mg, pct / 100, dim=1, keepdim=True)
    tp_ = torch.quantile(mp, pct / 100, dim=1, keepdim=True)
    g, p = (mg >= th).float(), (mp >= tp_).float()
    tp = (g * p).sum(1)
    prec, rec = tp / (p.sum(1) + 1e-9), tp / (g.sum(1) + 1e-9)
    return 2 * prec * rec / (prec + rec + 1e-9)


def artifact_energy(pred, gt, flat_pct=30):
    m = _sobel_mag(gt).flatten(1)
    th = torch.quantile(m, flat_pct / 100, dim=1, keepdim=True)
    flat = (m <= th).float()
    e = (pred - gt).abs().flatten(1)
    return (e * flat).sum(1) / (flat.sum(1) + 1e-9)


def cnr(img, roi_mask, surround_dilate=7):
    """img,[B,1,H,W]; roi_mask [B,1,H,W] bool/0-1.  CNR = |mu_roi - mu_bg| / std_bg."""
    out = []
    m = roi_mask.float()
    ring = F.max_pool2d(m, surround_dilate, 1, surround_dilate // 2) - m
    for i in range(img.shape[0]):
        r, b = img[i][m[i] > 0.5], img[i][ring[i] > 0.5]
        if r.numel() < 4 or b.numel() < 4:
            out.append(torch.tensor(float("nan"))); continue
        out.append((r.mean() - b.mean()).abs() / (b.std() + 1e-6))
    return torch.stack(out)


@torch.no_grad()
def full_eval(pred, gt, noisy_lr):
    """Dict of per-image metrics (tensors [B]) for a restored batch."""
    p = pred.clamp(0, 1)
    return {"psnr": psnr(p, gt), "ssim": ssim(p, gt), "hp_err": hp_err(p, gt, noisy_lr),
            "edge_corr": edge_preservation(p, gt), "edge_f1": edge_f1(p, gt),
            "flat_err": artifact_energy(p, gt)}

M = _ns(_before)

### 3.3a  Data - pairing, split manifest (80/10/10), dataset with RAM cache and seeded augmentation, loaders

In [12]:
"""
Data utilities (v2).

Fixes vs. submitted code
  * THREE-way split (train / val / test) stored in a JSON manifest.  Validation is used ONLY
    for early stopping / checkpoint selection; the test split is touched once, at the end.
    (The paper used the same 320 images for selection, tuning and reporting.)
  * One shared definition of the reference residual r = y - D(x) with D = area-average
    (documented, used everywhere; notebooks assert it).
  * Optional in-RAM cache (the old loader re-read .npy files from disk each step).
  * Reproducible augmentation (torch.Generator seeded per run).
"""
import glob
import hashlib
import json
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


# ----------------------------------------------------------------------------- pairing
def list_npy(folder):
    paths = sorted(glob.glob(os.path.join(str(folder), "*.npy")))
    return [p for p in paths if "__MACOSX" not in p and not os.path.basename(p).startswith("._")]


def match_pairs(gt_dir, noisy_dir):
    g = {os.path.basename(p): p for p in list_npy(gt_dir)}
    n = {os.path.basename(p): p for p in list_npy(noisy_dir)}
    common = sorted(set(g) & set(n))
    return [(g[k], n[k]) for k in common]


def load_npy(path):
    return np.load(path).astype(np.float32)


def downsample(img, out_hw):
    """Area-average D(.) -- THE reference downsampler used for every residual in the paper."""
    H, W = img.shape[:2]
    oh, ow = out_hw
    fh, fw = H // oh, W // ow
    return img[:oh * fh, :ow * fw].reshape(oh, fh, ow, fw).mean(axis=(1, 3))


# ----------------------------------------------------------------------------- split manifest
def make_split_manifest(pairs, path, val_frac=0.1, test_frac=0.1, seed=42):
    """Create (or load, if present) a deterministic train/val/test manifest keyed by file name."""
    if os.path.exists(path):
        return json.load(open(path))
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(pairs))
    n_val, n_test = int(len(pairs) * val_frac), int(len(pairs) * test_frac)
    names = [os.path.basename(p[0]) for p in pairs]
    man = {
        "seed": seed, "n_total": len(pairs),
        "test": sorted(names[i] for i in idx[:n_test]),
        "val": sorted(names[i] for i in idx[n_test:n_test + n_val]),
        "train": sorted(names[i] for i in idx[n_test + n_val:]),
    }
    man["sha1"] = hashlib.sha1(json.dumps([man["train"], man["val"], man["test"]]).encode()).hexdigest()
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    json.dump(man, open(path, "w"), indent=1)
    return man


def apply_manifest(pairs, man):
    by = {os.path.basename(p[0]): p for p in pairs}
    out = {k: [by[n] for n in man[k]] for k in ("train", "val", "test")}
    assert not (set(man["train"]) & set(man["val"])) and not (set(man["train"]) & set(man["test"])) \
        and not (set(man["val"]) & set(man["test"])), "split leakage!"
    return out


# ----------------------------------------------------------------------------- dataset
class PairedRestorationDataset(Dataset):
    """returns (noisy_LR, gt_HR, gt_ds, index)  -- all [1,H,W] float32."""

    def __init__(self, pairs, augment=False, cache=True, seed=0):
        self.pairs, self.augment = pairs, augment
        self.gen = torch.Generator().manual_seed(seed)
        self._cache = None
        if cache:
            self._cache = [self._read(i) for i in range(len(pairs))]

    def _read(self, i):
        gt_p, ny_p = self.pairs[i]
        gt, ny = load_npy(gt_p), load_npy(ny_p)
        ds = downsample(gt, ny.shape[:2])
        t = lambda a: torch.from_numpy(np.ascontiguousarray(a)).unsqueeze(0).float()
        return t(ny), t(gt), t(ds)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        ny, gt, ds = self._cache[i] if self._cache is not None else self._read(i)
        if self.augment:
            r = torch.randint(0, 2, (2,), generator=self.gen).tolist()
            k = int(torch.randint(0, 4, (1,), generator=self.gen))
            def tf(x):
                if r[0]: x = torch.flip(x, [-1])
                if r[1]: x = torch.flip(x, [-2])
                return torch.rot90(x, k, [-2, -1]) if k else x
            ny, gt, ds = tf(ny), tf(gt), tf(ds)
        return ny, gt, ds, i


def make_loaders(splits, batch_size=8, seed=0, cache=True, num_workers=0):
    tr = PairedRestorationDataset(splits["train"], augment=True, cache=cache, seed=seed)
    va = PairedRestorationDataset(splits["val"], augment=False, cache=cache)
    te = PairedRestorationDataset(splits["test"], augment=False, cache=cache)
    g = torch.Generator().manual_seed(seed)
    return (DataLoader(tr, batch_size, shuffle=True, drop_last=True, generator=g, num_workers=num_workers),
            DataLoader(va, batch_size, shuffle=False, num_workers=num_workers),
            DataLoader(te, batch_size, shuffle=False, num_workers=num_workers))

### 3.3b  Degradation model (speckle + Gaussian + downsampling) and its moment-matching estimate

In [13]:
"""
Degradation model  y = D( x * s + n ),  s ~ Gamma(L, 1/L) (unit-mean speckle), n ~ N(0, sigma^2),
D = area-average x2 downsampling.

  * `degrade`            : apply it (optionally only one component, for the per-type robustness test)
  * `fit_degradation`    : ESTIMATE (L, sigma) from real (GT, NoisyLR) pairs by moment matching
                           E[r^2 | I] = a*I^2 + b  with a = 1/(4L), b = sigma^2/4, on flat GT regions.
                           Returns the goodness-of-fit so a bad model is visible instead of assumed.
                           (The submitted OOD test hard-coded L=4, sigma=0.05 while claiming they
                           were the "fitted" values.)
  * `make_synthetic_dataset` : tiny synthetic KLA-like dataset used ONLY for smoke-testing the code.
"""
import os
import numpy as np
from scipy import ndimage



def degrade(clean, looks=4.0, sigma=0.05, factor=2, speckle=True, gaussian=True, down=True, seed=0):
    rng = np.random.RandomState(seed)
    x = clean.astype(np.float64)
    if speckle:
        x = x * rng.gamma(shape=looks, scale=1.0 / looks, size=x.shape)
    if gaussian:
        x = x + rng.normal(0, sigma, size=x.shape)
    if down:
        x = downsample(x, (x.shape[0] // factor, x.shape[1] // factor))
    return x.astype(np.float32)


def fit_degradation(pairs, n_images=400, n_bins=25, flat_pct=40, seed=0):
    """pairs: list of (gt_path, noisy_path).  Returns dict(looks, sigma, a, b, r2, bias, ...)."""
    rng = np.random.RandomState(seed)
    sel = rng.choice(len(pairs), min(n_images, len(pairs)), replace=False)
    I, R2, R = [], [], []
    for i in sel:
        gt, ny = load_npy(pairs[i][0]), load_npy(pairs[i][1])
        ds = downsample(gt, ny.shape[:2])
        r = ny - ds
        gm = ndimage.generic_gradient_magnitude(ds, ndimage.sobel)
        flat = gm <= np.percentile(gm, flat_pct)
        I.append(ds[flat]); R2.append(r[flat] ** 2); R.append(r[flat])
    I, R2, R = np.concatenate(I), np.concatenate(R2), np.concatenate(R)
    edges = np.quantile(I, np.linspace(0, 1, n_bins + 1))
    cen, mr2 = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (I >= lo) & (I <= hi)
        if m.sum() > 50:
            cen.append(I[m].mean()); mr2.append(R2[m].mean())
    cen, mr2 = np.array(cen), np.array(mr2)
    A = np.stack([cen ** 2, np.ones_like(cen)], 1)
    (a, b), *_ = np.linalg.lstsq(A, mr2, rcond=None)
    pred = A @ np.array([a, b])
    r2 = 1 - ((mr2 - pred) ** 2).sum() / (((mr2 - mr2.mean()) ** 2).sum() + 1e-12)
    looks = 1.0 / (4 * a) if a > 1e-9 else float("inf")
    sigma = float(np.sqrt(max(4 * b, 0.0)))
    return {"looks": float(looks), "sigma": sigma, "a": float(a), "b": float(b), "r2_signal_dependence": float(r2),
            "mean_residual": float(R.mean()), "n_pixels": int(I.size),
            "note": "moment-matching estimate; check r2_signal_dependence before trusting looks/sigma"}


# ----------------------------------------------------------------------------- smoke-test data
def _synthetic_clean(size, rng):
    x = np.linspace(0, 1, size)
    xx, yy = np.meshgrid(x, x)
    img = 0.45 + 0.2 * np.sin(2 * np.pi * rng.uniform(1, 4) * xx) * np.cos(2 * np.pi * rng.uniform(1, 4) * yy)
    for _ in range(rng.randint(4, 9)):
        cy, cx = rng.randint(0, size, 2); h, w = rng.randint(size // 16, size // 4, 2)
        img[max(0, cy - h):cy + h, max(0, cx - w):cx + w] = rng.uniform(0.1, 0.95)
    return np.clip(ndimage.gaussian_filter(img, 0.7), 0, 1).astype(np.float32)


def make_synthetic_dataset(root, n=48, hr=128, looks=4.0, sigma=0.05, seed=0):
    rng = np.random.RandomState(seed)
    gt_dir, ny_dir = os.path.join(root, "GT"), os.path.join(root, "NoisyLR")
    os.makedirs(gt_dir, exist_ok=True); os.makedirs(ny_dir, exist_ok=True)
    for i in range(n):
        g = _synthetic_clean(hr, rng)
        np.save(os.path.join(gt_dir, f"{i:06d}.npy"), g)
        np.save(os.path.join(ny_dir, f"{i:06d}.npy"), degrade(g, looks, sigma, seed=i))
    return gt_dir, ny_dir

### 3.3c  Distribution fitting, patch table, null models, image-clustered statistics  (namespace `D`, used in Part 5)

In [14]:
_before = set(globals())
"""
Re-analysis of Section IV (statistical motivation).

The submitted analysis shows (i) global GGD beta=0.845, (ii) patch beta mean 1.45 and (iii) negative
correlation between patch beta and (clean-image) variance/edges/entropy, and concludes that the *shape*
of the residual distribution varies spatially.  An obvious alternative was never tested:

    SIGNAL-DEPENDENT SCALE.  With multiplicative speckle, Var(r | I) grows with intensity I.
    Pooling pixels with different scales gives heavy tails globally (low beta) even if every pixel is
    exactly Gaussian given its local scale, and a 32x32 patch that straddles edges mixes scales, so its
    fitted beta is lower than that of a flat patch -- which reproduces correlations with edge density /
    variance WITHOUT any shape heterogeneity.

Everything here is built to test that alternative explicitly:

  patch_table              per-patch beta (fast moment estimator), residual stats, clean-image features
  homogeneous_null         patch betas if shape were constant (global GGD) -> pure sampling noise
  heteroscedastic_null     patch betas if r ~ N(0, a*I^2 + b)  (Gaussian given intensity)
  global_model_comparison  AIC: Gaussian / Laplace / Student-t / GGD vs heteroscedastic-Gaussian/-GGD
  cluster_bootstrap_corr   image-clustered CI for the beta-feature correlations
  partial_corr             correlation after regressing out local intensity (mean of clean patch)
  variance_decomposition   between-image vs within-image variance of patch beta (ICC)
"""
import numpy as np
import pandas as pd
from math import lgamma
from scipy import stats, ndimage, optimize
from scipy.special import gammaln


_BG = np.linspace(0.15, 4.0, 4000)
_RHO = np.exp(2 * gammaln(2 / _BG) - gammaln(1 / _BG) - gammaln(3 / _BG))     # (E|r|)^2 / E[r^2]


def ggd_beta_moment(r_patches):
    """Vectorised moment estimator of beta for zero-mean GGD. r_patches: [N, M]."""
    m1 = np.abs(r_patches).mean(1)
    m2 = (r_patches ** 2).mean(1)
    rho = np.clip(m1 ** 2 / (m2 + 1e-18), _RHO.min() + 1e-6, _RHO.max() - 1e-6)
    return np.interp(rho, _RHO, _BG)


def _patchify(a, p):
    H, W = a.shape
    nr, nc = H // p, W // p
    return a[:nr * p, :nc * p].reshape(nr, p, nc, p).transpose(0, 2, 1, 3).reshape(nr * nc, p * p), nr, nc


def patch_table(pairs, patch=32, max_images=None, seed=0, residual_fn=None):
    """One row per patch. residual_fn(gt_ds, noisy)->residual (default: noisy-gt_ds)."""
    rng = np.random.RandomState(seed)
    idx = np.arange(len(pairs)) if not max_images or max_images >= len(pairs) else rng.choice(len(pairs), max_images, replace=False)
    out = []
    for i in idx:
        gt, ny = load_npy(pairs[i][0]), load_npy(pairs[i][1])
        ds = downsample(gt, ny.shape[:2])
        r = (ny - ds) if residual_fn is None else residual_fn(ds, ny)
        gm = np.hypot(ndimage.sobel(ds, 0), ndimage.sobel(ds, 1))
        R, nr, nc = _patchify(r, patch); G, _, _ = _patchify(ds, patch); E, _, _ = _patchify(gm, patch)
        beta = ggd_beta_moment(R)
        ent = np.array([_entropy(g) for g in G])
        for k in range(R.shape[0]):
            out.append((int(i), k // nc, k % nc, beta[k], R[k].std(), np.abs(R[k]).mean(), G[k].mean(), G[k].var(), E[k].mean(), ent[k]))
    return pd.DataFrame(out, columns=["img", "row", "col", "beta", "resid_std", "resid_mad", "gt_mean", "gt_var", "edge_density", "entropy"])


def _entropy(g, bins=16):
    h, _ = np.histogram(g, bins=bins, range=(0, 1), density=True)
    h = h[h > 0]; p = h / h.sum()
    return float(-(p * np.log2(p)).sum())


# ----------------------------------------------------------------------------- nulls
def homogeneous_null(pairs, global_beta, patch=32, max_images=300, seed=0):
    """Constant shape = global_beta, but patch-specific scale taken from the data: isolates pure
    estimation noise of beta on a patch of patch^2 samples."""
    rng = np.random.RandomState(seed)
    def fn(ds, ny):
        real = ny - ds
        sc = np.full_like(real, 1.0) * real.std()
        return stats.gennorm.rvs(global_beta, scale=sc / np.sqrt(stats.gennorm.var(global_beta)), size=real.shape,
                                 random_state=rng)
    return patch_table(pairs, patch, max_images, seed, residual_fn=fn)


def heteroscedastic_null(pairs, a, b, patch=32, max_images=300, seed=0):
    """r ~ N(0, a*I^2 + b) given the clean intensity I  (per-pixel Gaussian -> NO shape heterogeneity)."""
    rng = np.random.RandomState(seed)
    def fn(ds, ny):
        return rng.normal(0, np.sqrt(np.maximum(a * ds ** 2 + b, 1e-12)))
    return patch_table(pairs, patch, max_images, seed, residual_fn=fn)


# ----------------------------------------------------------------------------- global model comparison
def _aic(ll, k):
    return 2 * k - 2 * ll


def global_model_comparison(pairs, n_images=200, seed=0, a_b=None):
    """AIC table.  Homoscedastic families (as in paper Table I) vs intensity-dependent-scale models."""
    rng = np.random.RandomState(seed)
    sel = rng.choice(len(pairs), min(n_images, len(pairs)), replace=False)
    R, I = [], []
    for i in sel:
        gt, ny = load_npy(pairs[i][0]), load_npy(pairs[i][1])
        ds = downsample(gt, ny.shape[:2]); R.append((ny - ds).ravel()); I.append(ds.ravel())
    r, I = np.concatenate(R), np.concatenate(I)
    n = len(r); rows = []
    s = r.std(); rows.append(("Gaussian", 1, stats.norm.logpdf(r, 0, s).sum()))
    sc = np.abs(r).mean(); rows.append(("Laplace", 1, stats.laplace.logpdf(r, 0, sc).sum()))
    df, _, ts = stats.t.fit(r, floc=0); rows.append(("Student-t", 2, stats.t.logpdf(r, df, 0, ts).sum()))
    sub = r[rng.choice(n, min(n, 400000), replace=False)]
    be, _, al = stats.gennorm.fit(sub, floc=0)
    rows.append(("GGD (global)", 2, stats.gennorm.logpdf(r, be, 0, al).sum()))
    # intensity-dependent scale: sigma^2 = a I^2 + b
    def nll_g(th):
        a, b = np.exp(th); v = a * I ** 2 + b
        return 0.5 * (np.log(2 * np.pi * v) + r ** 2 / v).sum()
    o = optimize.minimize(nll_g, np.log([0.01, 0.002]), method="Nelder-Mead")
    rows.append(("Hetero-Gaussian  N(0,aI^2+b)", 2, -o.fun)); ab = np.exp(o.x)
    def nll_gg(th):
        a, b, be_ = np.exp(th[0]), np.exp(th[1]), 0.3 + 3.7 / (1 + np.exp(-th[2]))
        v = np.sqrt(a * I ** 2 + b)                       # scale
        return -(stats.gennorm.logpdf(r, be_, 0, v * np.sqrt(np.exp(gammaln(1 / be_) - gammaln(3 / be_))))).sum()
    o2 = optimize.minimize(nll_gg, [o.x[0], o.x[1], 0.0], method="Nelder-Mead", options={"maxiter": 400})
    be2 = 0.3 + 3.7 / (1 + np.exp(-o2.x[2]))
    rows.append((f"Hetero-GGD (beta={be2:.2f})", 3, -o2.fun))
    df_ = pd.DataFrame(rows, columns=["model", "k", "loglik"])
    df_["AIC"] = [_aic(l, k) for l, k in zip(df_.loglik, df_.k)]
    df_["dAIC_vs_best"] = df_.AIC - df_.AIC.min()
    return df_.sort_values("AIC").reset_index(drop=True), {"a": float(ab[0]), "b": float(ab[1]), "ggd_beta": float(be), "ggd_alpha": float(al), "n": int(n)}


# ----------------------------------------------------------------------------- inference on correlations
def cluster_bootstrap_corr(df, x, y="beta", n_boot=1000, seed=0):
    """Pearson r with image-clustered bootstrap CI (patches inside an image are NOT independent)."""
    rng = np.random.RandomState(seed)
    groups = {k: g[[x, y]].values for k, g in df.groupby("img")}
    keys = np.array(list(groups))
    r0 = stats.pearsonr(df[x], df[y])[0]
    rs = []
    for _ in range(n_boot):
        samp = np.concatenate([groups[k] for k in rng.choice(keys, len(keys))])
        rs.append(np.corrcoef(samp[:, 0], samp[:, 1])[0, 1])
    return {"r": float(r0), "ci95": (float(np.quantile(rs, .025)), float(np.quantile(rs, .975))), "n_patches": len(df), "n_images": len(keys)}


def partial_corr(df, x, y, control):
    """corr(x,y | control) via residualisation (linear + quadratic in control)."""
    def res(v, c):
        A = np.stack([np.ones_like(c), c, c ** 2], 1)
        return v - A @ np.linalg.lstsq(A, v, rcond=None)[0]
    return float(np.corrcoef(res(df[x].values, df[control].values), res(df[y].values, df[control].values))[0, 1])


def variance_decomposition(df, col="beta"):
    """One-way random-effects ICC: share of patch-beta variance that is BETWEEN images."""
    g = df.groupby("img")[col]
    k = g.size().mean(); n = g.ngroups
    msb = (g.size() * (g.mean() - df[col].mean()) ** 2).sum() / (n - 1)
    msw = ((df[col] - g.transform("mean")) ** 2).sum() / (len(df) - n)
    icc = (msb - msw) / (msb + (k - 1) * msw)
    return {"ICC": float(icc), "var_between": float(max(msb - msw, 0) / k), "var_within": float(msw), "patches_per_image": float(k)}


def compare_observed_vs_null(obs, null_tables, feats=("edge_density", "gt_var", "entropy")):
    rows = []
    for name, t in [("observed", obs)] + list(null_tables.items()):
        r = {"table": name, "beta_mean": t.beta.mean(), "beta_std": t.beta.std()}
        for f in feats:
            r[f"r(beta,{f})"] = stats.pearsonr(t[f], t.beta)[0]
        rows.append(r)
    return pd.DataFrame(rows)

D = _ns(_before)

### 3.3d  Statistics - paired tests, bootstrap CI, Holm correction

In [15]:
"""
Statistical tests for the results tables (reviewer comment 1).

Two complementary levels -- report BOTH:
  (A) seed level  : mean +- std over independent training runs (captures optimisation variance);
                    Welch t-test / Mann-Whitney on the per-seed test means.
  (B) image level : paired comparison of two configurations on the SAME test images, using the
                    seed-averaged per-image metric.  Paired t, Wilcoxon signed-rank and a paired
                    bootstrap CI of the mean difference.  (Captures data-sampling variance.)
Multiple comparisons are controlled with Holm-Bonferroni.  Effect size = Cohen's d_z (paired).
"""
import numpy as np
import pandas as pd
from scipy import stats


def mean_std(x):
    x = np.asarray(x, float)
    return float(x.mean()), float(x.std(ddof=1)) if len(x) > 1 else float("nan")


def bootstrap_ci(d, n_boot=10000, alpha=0.05, seed=0):
    rng = np.random.RandomState(seed)
    d = np.asarray(d, float)
    idx = rng.randint(0, len(d), (n_boot, len(d)))
    m = d[idx].mean(1)
    return float(np.quantile(m, alpha / 2)), float(np.quantile(m, 1 - alpha / 2))


def paired_image_test(a, b, higher_better=True, n_boot=10000, seed=0):
    """a,b: arrays [n_seeds, n_images] OR [n_images].  Tests H0: mean(a-b)=0.  Returns dict."""
    a, b = np.atleast_2d(a).mean(0), np.atleast_2d(b).mean(0)
    d = a - b
    if not higher_better:
        d = -d
    n = len(d)
    t, p_t = stats.ttest_rel(a, b)
    try:
        w, p_w = stats.wilcoxon(d)
    except ValueError:
        w, p_w = np.nan, 1.0
    lo, hi = bootstrap_ci(d, n_boot, seed=seed)
    dz = d.mean() / (d.std(ddof=1) + 1e-12)
    return {"mean_diff": float(d.mean()), "ci95_lo": lo, "ci95_hi": hi, "p_ttest": float(p_t),
            "p_wilcoxon": float(p_w), "cohens_dz": float(dz), "n_images": n,
            "frac_images_improved": float((d > 0).mean())}


def seed_level_test(x, y):
    """x,y: per-seed test means of two configs."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    out = {"mean_x": x.mean(), "mean_y": y.mean(), "diff": x.mean() - y.mean(), "n_x": len(x), "n_y": len(y)}
    if len(x) > 1 and len(y) > 1:
        out["p_welch"] = float(stats.ttest_ind(x, y, equal_var=False).pvalue)
        out["p_mannwhitney"] = float(stats.mannwhitneyu(x, y, alternative="two-sided").pvalue)
    return out


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (same order as input)."""
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    for rank, i in enumerate(order):
        val = (len(p) - rank) * p[i]
        run = max(run, val)
        adj[i] = min(1.0, run)
    return adj


def summarize_configs(summary_df, metric="test_psnr", configs=None):
    """mean +- std (over seeds) for each config -> DataFrame."""
    rows = []
    for c, g in summary_df.groupby("config"):
        if configs and c not in configs: continue
        m, s = mean_std(g[metric])
        rows.append({"config": c, "n_seeds": len(g), "mean": m, "std": s, "min": g[metric].min(),
                     "max": g[metric].max(), "params": int(g["params"].iloc[0])})
    out = pd.DataFrame(rows)
    if configs:
        out["o"] = out["config"].map({c: i for i, c in enumerate(configs)}); out = out.sort_values("o").drop(columns="o")
    return out.reset_index(drop=True)


def compare_to_reference(per_image, ref, configs, higher_better=True):
    """per_image: dict name -> [n_seeds, n_images] array.  Compare every config with `ref`, Holm-corrected."""
    rows = []
    for c in configs:
        if c == ref or c not in per_image or per_image[c].size == 0: continue
        r = paired_image_test(per_image[c], per_image[ref], higher_better)
        r["config"], r["vs"] = c, ref
        rows.append(r)
    df = pd.DataFrame(rows)
    if len(df):
        df["p_wilcoxon_holm"] = holm(df["p_wilcoxon"].values)
        df["p_ttest_holm"] = holm(df["p_ttest"].values)
        df["significant_0.05"] = df["p_wilcoxon_holm"] < 0.05
    return df


def noise_floor(per_image_a, per_image_b):
    """Seed-to-seed spread of the mean metric for a config: what a 'real' effect must exceed."""
    m = np.atleast_2d(per_image_a).mean(1)
    return {"seed_means": m.tolist(), "std_over_seeds": float(m.std(ddof=1)) if len(m) > 1 else float("nan")}


def latex_table(df, cols, fmt="{:.3f}", caption="", label=""):
    head = " & ".join(cols) + r" \\"
    body = "\n".join(" & ".join(fmt.format(v) if isinstance(v, (float, np.floating)) else str(v)
                                for v in row) + r" \\" for row in df[cols].itertuples(index=False))
    return (f"\\begin{{table}}[t]\n\\centering\\caption{{{caption}}}\\label{{{label}}}\n"
            f"\\begin{{tabular}}{{{'l' + 'c' * (len(cols) - 1)}}}\n\\hline\n{head}\n\\hline\n{body}\n\\hline\n"
            f"\\end{{tabular}}\n\\end{{table}}")

### 3.3e  Figure helpers (PNG + PDF + provenance JSON)  (namespace `FG`)

In [16]:
_before = set(globals())
"""
Publication figures (reviewer comment 6: legibility, resolution, caption/figure consistency).

  * 300 dpi PNG + vector PDF, large fonts, colour-blind-safe palette, no overlapping text.
  * PROVENANCE: every figure writes <name>.json with the checkpoint, split, image id(s), seed and the
    exact numbers shown -- the manuscript's captions/numbers must be copied from that file, never typed.
    (In the submitted paper Fig. 5 showed betas from a 3-epoch model while the text quoted another run,
    and Fig. 3's caption described a different plot.)
"""
import json
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11, "legend.fontsize": 9,
                     "figure.dpi": 120, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PALETTE), "pdf.fonttype": 42})


def save(fig, out_dir, name, provenance=None):
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, f"{name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(out_dir, f"{name}.pdf"), bbox_inches="tight")
    if provenance is not None:
        json.dump(provenance, open(os.path.join(out_dir, f"{name}.json"), "w"), indent=1, default=float)
    plt.close(fig)


def bars_with_error(summary, metric_cols, title, out_dir, name, ylabel="", ref=None):
    """summary: DataFrame config,mean,std(,n_seeds).  Error bars = std over seeds."""
    fig, ax = plt.subplots(figsize=(1.1 * len(summary) + 2, 3.8))
    x = np.arange(len(summary))
    ax.bar(x, summary["mean"], yerr=summary["std"].fillna(0), capsize=4, color=PALETTE[0], alpha=.9)
    lo = (summary["mean"] - summary["std"].fillna(0)).min(); hi = (summary["mean"] + summary["std"].fillna(0)).max()
    ax.set_ylim(lo - 0.3 * (hi - lo + 1e-6), hi + 0.2 * (hi - lo + 1e-6))
    ax.set_xticks(x); ax.set_xticklabels(summary["config"], rotation=30, ha="right")
    ax.set_ylabel(ylabel); ax.set_title(title)
    for xi, (m, n) in enumerate(zip(summary["mean"], summary.get("n_seeds", [None] * len(summary)))):
        ax.text(xi, ax.get_ylim()[0], f"n={n}", ha="center", va="bottom", fontsize=8, color="w")
    save(fig, out_dir, name, {"table": summary.to_dict("records"), "metric": ylabel, "error_bars": "std over seeds"})


def training_curves(history_csv, out_dir, name, label="", provenance=None):
    import pandas as pd
    h = pd.read_csv(history_csv)
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
    ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val"); ax[0].set_title("Loss"); ax[0].legend()
    ax[1].plot(h.epoch, h.val_psnr, color=PALETTE[2]); ax[1].set_title("Validation PSNR (dB)")
    ax[2].plot(h.epoch, h.val_ssim, color=PALETTE[4]); ax[2].set_title("Validation SSIM")
    for a in ax: a.set_xlabel("epoch")
    fig.suptitle(label, y=1.02)
    save(fig, out_dir, name, provenance or {"history": history_csv, "best_val_psnr": float(h.val_psnr.max())})


def component_health(history_csv, out_dir, name, beta_min=0.3, beta_max=2.5, provenance=None):
    import pandas as pd
    h = pd.read_csv(history_csv)
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
    for k in range(3):
        ax[0].plot(h.epoch, h[f"beta{k}"], label=f"$\\beta_{k}$"); ax[2].plot(h.epoch, h[f"usage{k}"], label=f"comp {k}")
    ax[0].axhline(beta_min, ls=":", c="gray"); ax[0].axhline(beta_max, ls=":", c="gray"); ax[0].set_title("Learned $\\beta_k$ (dotted: bounds)"); ax[0].legend()
    ax[1].plot(h.epoch, h.min_beta_dist); ax[1].axhline(0.3, ls="--", c="gray"); ax[1].set_title("Min pairwise $\\beta$ distance (dashed: margin)")
    ax[2].set_title("Mean component usage"); ax[2].legend()
    for a in ax: a.set_xlabel("epoch")
    final = {f"beta{k}": float(h[f"beta{k}"].iloc[-1]) for k in range(3)}
    save(fig, out_dir, name, provenance or {"history": history_csv, "final": final})


def mixture_maps(img, mix_w, beta, out_dir, name, provenance):
    """img [H,W]; mix_w [K,H,W]; beta [K].  The 'dominant' panel makes single-component collapse obvious."""
    K = mix_w.shape[0]
    fig, ax = plt.subplots(1, K + 2, figsize=(2.6 * (K + 2), 2.9))
    ax[0].imshow(img, cmap="gray"); ax[0].set_title("Input")
    for k in range(K):
        im = ax[k + 1].imshow(mix_w[k], vmin=0, vmax=1, cmap="viridis"); ax[k + 1].set_title(f"$\\pi_{k}$ ($\\beta$={beta[k]:.2f})")
    dom = mix_w.argmax(0); ax[K + 1].imshow(dom, cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
    fr = np.bincount(dom.ravel(), minlength=K) / dom.size
    ax[K + 1].set_title("Dominant: " + "/".join(f"{f:.0%}" for f in fr))
    for a in ax: a.axis("off")
    fig.colorbar(im, ax=ax[:K + 1], fraction=0.015, pad=0.01)
    provenance = {**provenance, "betas": [float(b) for b in beta], "dominant_fraction": fr.tolist()}
    save(fig, out_dir, name, provenance)


def qualitative_grid(items, out_dir, name, provenance, zoom=None):
    """items: list of (title, 2-D array); first is clean GT. Adds absolute-error maps and an optional zoom box."""
    gt = items[0][1]
    n = len(items)
    fig, ax = plt.subplots(2, n, figsize=(2.6 * n, 5.4))
    for j, (t, a) in enumerate(items):
        ax[0, j].imshow(a, cmap="gray", vmin=0, vmax=1); ax[0, j].set_title(t, fontsize=9)
        e = np.abs(a - gt); ax[1, j].imshow(e, cmap="inferno", vmin=0, vmax=0.25)
        ax[1, j].set_title(f"|err|  mean={e.mean():.3f}" if j else "|error| (0-0.25)", fontsize=9)
        if zoom is not None:
            y0, y1, x0, x1 = zoom
            ax[0, j].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="cyan", lw=1))
        ax[0, j].axis("off"); ax[1, j].axis("off")
    save(fig, out_dir, name, provenance)
    if zoom is not None:
        y0, y1, x0, x1 = zoom
        fig, ax = plt.subplots(1, n, figsize=(2.6 * n, 2.9))
        for j, (t, a) in enumerate(items):
            ax[j].imshow(a[y0:y1, x0:x1], cmap="gray", vmin=0, vmax=1, interpolation="nearest"); ax[j].set_title(t, fontsize=9); ax[j].axis("off")
        save(fig, out_dir, name + "_zoom", provenance)


def scatter_beta(df, feats, out_dir, name, provenance, n_max=20000):
    """hexbin of patch beta vs structural features; caption numbers come from `provenance`."""
    fig, ax = plt.subplots(1, len(feats), figsize=(3.8 * len(feats), 3.4))
    d = df.sample(min(len(df), n_max), random_state=0)
    for a, f in zip(np.atleast_1d(ax), feats):
        hb = a.hexbin(d[f], d.beta, gridsize=40, cmap="viridis", mincnt=1, bins="log")
        a.set_xlabel(f.replace("_", " ")); a.set_ylabel("patch $\\beta$")
        r = np.corrcoef(df[f], df.beta)[0, 1]; a.set_title(f"r = {r:.3f}")
    save(fig, out_dir, name, provenance)

FG = _ns(_before)

### 3.4  Configuration (namespace `C`): paths, optimiser constants, split, helpers

In [17]:
C = types.SimpleNamespace()
C.MODE = "full" if EPOCHS >= 100 else "pilot"                      # only used in labels / bootstrap sizes
C.EPOCHS, C.SEEDS, C.BATCH_SIZE, C.PATIENCE, C.CACHE = EPOCHS, list(SEEDS), BATCH_SIZE, PATIENCE, True      # from Part 1
C.LR, C.WEIGHT_DECAY, C.GRAD_CLIP, C.SPLIT_SEED = 2e-4, 1e-4, 1.0, 42
C.DATA_ROOT, C.OUT = DATA_ROOT, RESULTS_DIR
C.RUNS, C.FIGS, C.TABLES = (os.path.join(RESULTS_DIR, d) for d in ("runs", "figures", "tables"))
for _d in (C.RUNS, C.FIGS, C.TABLES): os.makedirs(_d, exist_ok=True)
FIG_DIR = C.FIGS

def get_pairs():
    return match_pairs(os.path.join(DATA_ROOT, "GT"), os.path.join(DATA_ROOT, "NoisyLR"))
def get_splits():
    # -> (splits dict train/val/test -> pairs, manifest). The manifest (keyed by file NAME) is saved in <results>/split_manifest.json and re-used if it exists.
    pairs = get_pairs()
    man = make_split_manifest(pairs, os.path.join(C.OUT, "split_manifest.json"), 0.1, 0.1, C.SPLIT_SEED)
    return apply_manifest(pairs, man), man
def device(): return "cuda" if torch.cuda.is_available() else "cpu"
def banner(): print(f"MODE={C.MODE}  epochs={C.EPOCHS}  seeds={C.SEEDS}  device={device()}  out={C.OUT}")
C.get_pairs, C.get_splits, C.device, C.banner = get_pairs, get_splits, device, banner

def save_now(tag=""):
    # zip the whole results folder (runs, figures, tables, split manifest) so nothing is lost if the session ends
    zip_base = os.path.join(BASE, "results_latest")
    shutil.make_archive(zip_base, "zip", RESULTS_DIR)
    msg = f"saved {zip_base}.zip ({os.path.getsize(zip_base + '.zip')/1e6:.1f} MB)"
    if DRIVE_DIR:
        os.makedirs(DRIVE_DIR, exist_ok=True); shutil.copy(zip_base + ".zip", os.path.join(DRIVE_DIR, "results_latest.zip")); msg += f" + copied to {DRIVE_DIR}"
    print(msg, tag)
def savefig(fig, name, provenance=None):
    # PNG + PDF + a provenance JSON next to every figure (notebook 09 refuses figures without one); the figure stays open so it can be shown
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight"); fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")
    prov = {"figure": name, "generated_by": "master notebook", "epochs": EPOCHS, "seeds": SEEDS, "n_pairs": N_PAIRS}
    prov.update(provenance or {})
    json.dump(prov, open(os.path.join(FIG_DIR, name + ".json"), "w"), indent=1, default=float)
banner()

MODE=pilot  epochs=30  seeds=[0, 1, 2]  device=cuda  out=/kaggle/working/results_pilot


### 3.5  Training loop and THE FOUR MODEL DEFINITIONS (`CONFIGS`)
Search this cell for `CONFIGS`: `baseline`, `global_gc`, `film_control`, `full` are defined there together with the loss each one uses (`compute_loss`), the optimiser groups (no weight decay on the prototypes), early stopping, resume, and the single evaluation on the test split.

In [18]:
"""
Experiment registry + single-run trainer.

Every configuration is trained with the SAME protocol (split manifest, augmentation stream,
optimiser, schedule, epochs, patience, checkpoint rule).  A run is identified by
(config_name, seed) and is skipped when `results.json` already exists, so long jobs on a laptop
GPU can be interrupted and resumed.

Outputs per run  (runs/<config>/seed<seed>/):
    best.pth                best-validation-PSNR weights (+ config)
    history.csv             per-epoch train/val loss, val PSNR/SSIM, LR, component statistics
    test_per_image.csv      per-image TEST metrics at the best-val checkpoint   <- used for paired stats
    results.json            summary (best epoch, val & test means, params, time)
"""
import csv
import json
import os
import random
import time
import numpy as np
import torch


# ----------------------------------------------------------------------------- registry
# arch: 'baseline' | 'ldmh' | 'ext:<name>' ; loss: 'charbonnier' | 'gen_charbonnier' | 'mixture'
CONFIGS = {
    # --- the ladder in the paper ------------------------------------------------------
    "baseline":        dict(arch="baseline", loss="charbonnier"),
    "global_gc":       dict(arch="baseline", loss="gen_charbonnier"),
    "ldmh_loss_only":  dict(arch="ldmh", film=False, loss="mixture", div=0.1, bal=0.3),
    "full":            dict(arch="ldmh", film=True, loss="mixture", div=0.1, bal=0.3),
    # --- NEW controls requested by the audit ------------------------------------------
    "film_control":    dict(arch="ldmh", film=True, loss="charbonnier"),            # same capacity, no NLL
    "full_no_penalty": dict(arch="ldmh", film=True, loss="mixture", div=0.0, bal=0.0),   # = old Table V loss
    "full_clamp":      dict(arch="ldmh", film=True, loss="mixture", div=0.1, bal=0.3, beta_mode="clamp"),
    "full_no_bal":     dict(arch="ldmh", film=True, loss="mixture", div=0.1, bal=0.0),
    "full_no_div":     dict(arch="ldmh", film=True, loss="mixture", div=0.0, bal=0.3),
    # --- external architectures (same protocol) ---------------------------------------
    **{k: dict(arch=f"ext:{k}", loss="charbonnier") for k in BASELINE_REGISTRY},
}
LADDER = ["baseline", "global_gc", "film_control", "ldmh_loss_only", "full"]
ABLATION = ["baseline", "global_gc", "film_control", "ldmh_loss_only", "full",
            "full_no_penalty", "full_clamp", "full_no_bal", "full_no_div"]
EXTERNAL = ["baseline", "full", "NAFNet-small", "Restormer-small", "SwinIR-small",
            "NAFNet-base", "Restormer-base", "SwinIR-base"]


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_model(cfg):
    a = cfg["arch"]
    if a == "baseline":
        return BaselineNet()
    if a == "ldmh":
        return DistributionMixtureRestorationNet(use_film=cfg.get("film", True),
                                                 beta_mode=cfg.get("beta_mode", "sigmoid"),
                                                 init_scale=cfg.get("init_scale", 0.05))
    if a.startswith("ext:"):
        return BASELINE_REGISTRY[a[4:]]()
    raise ValueError(a)


def is_ldmh(cfg):
    return cfg["arch"] in ("ldmh", "scale")


def compute_loss(cfg, out, gt, gt_ds, noisy):
    pred = out[0] if is_ldmh(cfg) else out
    kind = cfg["loss"]
    if kind == "charbonnier":
        loss = charbonnier_loss(pred, gt)
    elif kind == "gen_charbonnier":
        loss = gen_charbonnier_loss(pred, gt, p=cfg.get("global_beta", 0.845))
    elif kind == "mixture":
        _, w, beta, scale = out
        nll = mixture_gen_gaussian_nll(noisy - gt_ds, w, beta, scale)
        loss = (charbonnier_loss(pred, gt) + cfg.get("nll", 1.0) * nll
                + cfg.get("div", 0.0) * beta_diversity_penalty(beta, 0.3)
                + cfg.get("bal", 0.0) * load_balance_penalty_entropy(w))
    else:
        raise ValueError(kind)
    return loss, pred


def _param_groups(model, wd):
    proto = [p for n, p in model.named_parameters() if n.endswith("log_beta") or n.endswith("log_scale")]
    ids = {id(p) for p in proto}
    rest = [p for p in model.parameters() if id(p) not in ids]
    g = [{"params": rest, "weight_decay": wd}]
    if proto:
        g.append({"params": proto, "weight_decay": 0.0})
    return g


@torch.no_grad()
def evaluate(model, cfg, loader, device, per_image=False, with_loss=False):
    model.eval()
    rows, losses = [], []
    for noisy, gt, gt_ds, idx in loader:
        noisy, gt, gt_ds = noisy.to(device), gt.to(device), gt_ds.to(device)
        out = model(noisy)
        if with_loss:
            losses.append(compute_loss(cfg, out, gt, gt_ds, noisy)[0].item())
        pred = out[0] if is_ldmh(cfg) else out
        m = M.full_eval(pred, gt, noisy)
        for j in range(noisy.shape[0]):
            rows.append({"idx": int(idx[j]), **{k: float(v[j]) for k, v in m.items()}})
    mean = {k: float(np.mean([r[k] for r in rows])) for k in rows[0] if k != "idx"}
    if with_loss:
        mean["loss"] = float(np.mean(losses))
    return (mean, rows) if per_image else mean


def run_one(name, seed, splits, run_root="runs", epochs=200, batch_size=8, lr=2e-4, wd=1e-4,
            clip=1.0, patience=30, device=None, cache=True, verbose=True, cfg_override=None, resume=True, deadline=None):
    cfg = dict(CONFIGS[name]); cfg.update(cfg_override or {})
    out_dir = os.path.join(run_root, name, f"seed{seed}")
    res_path = os.path.join(out_dir, "results.json")
    if os.path.exists(res_path):
        if verbose: print(f"[skip] {name} seed{seed} (done)")
        return json.load(open(res_path))
    os.makedirs(out_dir, exist_ok=True)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device == "cuda"
    if device == "cpu":
        torch.set_num_threads(max(1, os.cpu_count() or 1))

    set_seed(seed)
    model = build_model(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    train_l, val_l, test_l = make_loaders(splits, batch_size, seed=seed, cache=cache)
    opt = torch.optim.AdamW(_param_groups(model, wd, cfg.get("proto_lr_mult", 1.0), lr), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    fields = ["epoch", "train_loss", "val_loss", "val_psnr", "val_ssim", "lr", "time_sec"]
    if has_stats(cfg):
        fields += [f"beta{k}" for k in range(3)] + [f"usage{k}" for k in range(3)] + ["min_beta_dist", "any_beta_at_bound"]
    hist_path, last_path = os.path.join(out_dir, "history.csv"), os.path.join(out_dir, "last.pth")
    best, best_ep, bad, t_start, prior_sec, start_ep = -1e9, -1, 0, time.time(), 0.0, 1
    if resume and os.path.exists(last_path) and os.path.exists(hist_path):
        # continue an interrupted run (e.g. a cloud session that timed out) from the end of its last finished epoch
        ck = torch.load(last_path, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); best, best_ep, bad = ck["best"], ck["best_ep"], ck["bad"]
        prior_sec, start_ep = ck.get("elapsed_sec", 0.0), ck["epoch"] + 1
        f = open(hist_path, "a", newline=""); wr = csv.DictWriter(f, fieldnames=fields)
        if verbose: print(f"[resume] {name} seed{seed}: continuing from epoch {start_ep}/{epochs}", flush=True)
    else:
        f = open(hist_path, "w", newline="")
        wr = csv.DictWriter(f, fieldnames=fields); wr.writeheader()
    ep = start_ep - 1
    for ep in range(start_ep, epochs + 1):
        t0 = time.time(); model.train(); tl = []
        for noisy, gt, gt_ds, _ in train_l:
            noisy, gt, gt_ds = noisy.to(device), gt.to(device), gt_ds.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=use_amp):
                loss, _ = compute_loss(cfg, model(noisy), gt, gt_ds, noisy)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update(); tl.append(loss.item())
            if verbose and ep == 1 and len(tl) in (10, 50) or (verbose and ep == 1 and len(tl) % 100 == 0):
                dt = time.time() - t0
                print(f"   [{name}|s{seed}] epoch 1: {len(tl)}/{len(train_l)} batches, {dt:.0f}s "
                      f"-> projected {dt / len(tl) * len(train_l) / 60:.1f} min/epoch", flush=True)
        sched.step()

        vm = evaluate(model, cfg, val_l, device, with_loss=True)
        row = {"epoch": ep, "train_loss": np.mean(tl), "val_loss": vm["loss"], "val_psnr": vm["psnr"],
               "val_ssim": vm["ssim"], "lr": opt.param_groups[0]["lr"], "time_sec": time.time() - t0}
        if has_stats(cfg):
            with torch.no_grad():
                noisy = next(iter(val_l))[0].to(device)
                _, w, b, s = model(noisy)
                cs = model.ldmh.get_component_stats(w, b, s)
            for k in range(3):
                row[f"beta{k}"], row[f"usage{k}"] = cs["beta"][k], cs["mean_usage"][k]
            row["min_beta_dist"], row["any_beta_at_bound"] = cs["min_beta_pairwise_dist"], int(any(cs["beta_hit_bound"]))
        wr.writerow(row); f.flush()
        if verbose:
            et = time.time() - t0; eta = et * (epochs - ep) / 60
            print(f"[{name}|s{seed}] ep {ep}/{epochs} train_loss={np.mean(tl):.5f} val_psnr={vm['psnr']:.3f} "
                  f"val_ssim={vm['ssim']:.4f} | {et / 60:.1f} min/epoch, ~{eta:.0f} min left for this run", flush=True)
        if vm["psnr"] > best:
            best, best_ep, bad = vm["psnr"], ep, 0
            torch.save({"model": model.state_dict(), "cfg": cfg, "epoch": ep, "seed": seed}, os.path.join(out_dir, "best.pth"))
        else:
            bad += 1
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "best": best, "best_ep": best_ep, "bad": bad,
                    "elapsed_sec": prior_sec + time.time() - t_start}, last_path)
        if deadline is not None and time.time() > deadline and ep < epochs:
            f.close()
            raise TimeoutError(f"time budget reached after epoch {ep}; progress saved, run again to resume")
        if bad >= patience:
            break
    f.close()

    ck = torch.load(os.path.join(out_dir, "best.pth"), map_location=device)
    model.load_state_dict(ck["model"])
    val_m = evaluate(model, cfg, val_l, device)
    test_m, rows = evaluate(model, cfg, test_l, device, per_image=True)     # TEST touched exactly once
    with open(os.path.join(out_dir, "test_per_image.csv"), "w", newline="") as g:
        w = csv.DictWriter(g, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    res = {"name": name, "seed": seed, "best_epoch": best_ep, "epochs_run": ep, "params": n_params,
           "val": val_m, "test": test_m, "train_minutes": (prior_sec + time.time() - t_start) / 60,
           "device": device, "cfg": cfg}
    json.dump(res, open(res_path, "w"), indent=1)
    if os.path.exists(last_path): os.remove(last_path)        # optimiser state no longer needed once the run is finished
    return res


def load_results(run_root, names=None):
    """-> (summary DataFrame, dict[name] -> per-image DataFrame list)"""
    import pandas as pd, glob
    rows = []
    for p in glob.glob(os.path.join(run_root, "*", "seed*", "results.json")):
        r = json.load(open(p))
        if names and r["name"] not in names: continue
        rows.append({"config": r["name"], "seed": r["seed"], "params": r["params"], "best_epoch": r["best_epoch"],
                     "minutes": r["train_minutes"], **{f"val_{k}": v for k, v in r["val"].items()},
                     **{f"test_{k}": v for k, v in r["test"].items()}})
    return pd.DataFrame(rows).sort_values(["config", "seed"]).reset_index(drop=True)


def load_per_image(run_root, name, metric="psnr"):
    """-> array [n_seeds, n_test_images] for `metric`, aligned on idx."""
    import pandas as pd, glob
    mats = []
    for p in sorted(glob.glob(os.path.join(run_root, name, "seed*", "test_per_image.csv"))):
        d = pd.read_csv(p).sort_values("idx"); mats.append(d[metric].values)
    return np.stack(mats) if mats else np.empty((0, 0))

### 3.7 Extensions (new): scale-only control, Restormer-LDMH, new losses, stress arms, effect helpers

In [19]:
# ============================ 3.7  EXTENSIONS (new in this notebook) ============================
"""
Everything the new experiments need, defined AFTER the master code so that the master cells above stay byte-identical.

  * is_ldmh / has_stats        : tuple-returning networks vs. networks with an LDMH head (beta / usage statistics)
  * _param_groups(.., mult)    : optional learning-rate multiplier for the prototype parameters (the 'stress' arms)
  * RestormerLDMHNet           : the same LDMH + FiLM module on the Restormer-small backbone (+ optional FiLM at every block)
  * ScaleOnlyNet               : SCALE-ONLY CONTROL. Same encoder / FiLM / backbone as `full`, but the head predicts ONE per-pixel
                                 scale (no mixture, no learned shape) and is trained with a heteroscedastic generalised-Gaussian NLL
                                 with a FIXED shape (beta=1.53 = best global model of Table I, or beta=2 = Hetero-Gaussian).
                                 This is exactly the 'signal-dependent noise scale' explanation of the data (Section IV).
  * losses                     : 'scale_nll', and 'mixture_out' (mixture NLL + a noise-model-weighted loss on the restoration error)
  * CONFIGS                    : all new arms
"""
import zlib

def is_ldmh(cfg):  return cfg["arch"] in ("ldmh", "scale")      # the network returns a tuple (prediction, ...)
def has_stats(cfg): return cfg["arch"] == "ldmh"                 # the network has an LDMH head with beta / usage statistics

def _param_groups(model, wd, mult=1.0, lr=None):
    proto = [p for n, p in model.named_parameters() if n.endswith("log_beta") or n.endswith("log_scale")]
    ids = {id(p) for p in proto}
    rest = [p for p in model.parameters() if id(p) not in ids]
    g = [{"params": rest, "weight_decay": wd}]
    if proto:
        d = {"params": proto, "weight_decay": 0.0}
        if mult != 1.0 and lr is not None: d["lr"] = lr * mult
        g.append(d)
    return g


class RestormerLDMHNet(nn.Module):
    """LDMH + FiLM attached to the Restormer-small body (stem -> [FiLM] -> 6 Restormer blocks -> PixelShuffle x2 -> head + bilinear residual).
    film_every=True additionally modulates the features after every block (a 'deeper' conditioning, more parameters)."""
    def __init__(self, width=32, heads=2, n_blocks=6, n_components=3, scale=2, use_film=True, film_every=False,
                 beta_mode="sigmoid", init_scale=0.05, temperature=1.5):
        super().__init__()
        self.use_film, self.film_every, self.up_factor = use_film, film_every, scale
        self.ldmh = LocalDistributionMixtureHead(1, width, n_components, beta_mode=beta_mode, init_scale=init_scale, temperature=temperature)
        self.film_proj = MixtureFiLMProjector(n_components, width)
        if film_every:
            self.film_mid = nn.ModuleList([MixtureFiLMProjector(n_components, width) for _ in range(n_blocks - 1)])
        self.stem = nn.Conv2d(1, width, 3, padding=1)
        self.body = nn.ModuleList([_RBlock(width, heads) for _ in range(n_blocks)])     # same state-dict keys as nn.Sequential
        self.up = nn.Sequential(nn.Conv2d(width, width * scale * scale, 3, padding=1), nn.PixelShuffle(scale))
        self.head = nn.Conv2d(width, 1, 3, padding=1)

    def forward(self, x):
        w, beta, sc, _ = self.ldmh(x)
        f = self.stem(x)
        if self.use_film:
            g, s = self.film_proj(w); f = f * (1 + g) + s
        for i, blk in enumerate(self.body):
            f = blk(f)
            if self.film_every and i < len(self.body) - 1:
                g, s = self.film_mid[i](w); f = f * (1 + g) + s
        out = self.head(F.gelu(self.up(f)))
        x_up = F.interpolate(x, scale_factor=self.up_factor, mode="bilinear", align_corners=False)
        return x_up + out, w, beta, sc


class ScaleOnlyNet(nn.Module):
    """Scale-only control: encoder -> ONE per-pixel noise scale sigma(u) -> FiLM(sigma/sigma0) -> NAFNet-style backbone."""
    def __init__(self, base_ch=32, init_scale=0.05, n_lr_blocks=4, n_hr_blocks=2, scale=2):
        super().__init__()
        self.sigma0 = init_scale
        self.enc = nn.Sequential(nn.Conv2d(1, base_ch, 3, padding=1), nn.GroupNorm(8, base_ch), nn.GELU(),
                                 nn.Conv2d(base_ch, base_ch, 3, padding=1), nn.GroupNorm(8, base_ch), nn.GELU())
        self.scale_head = nn.Conv2d(base_ch, 1, 1)
        nn.init.constant_(self.scale_head.bias, math.log(math.expm1(max(init_scale - 1e-3, 1e-4))))
        self.film_proj = MixtureFiLMProjector(1, base_ch)
        self.backbone = RestorationBackboneLite(base_ch, n_lr_blocks, n_hr_blocks, scale)

    def forward(self, x):
        sig = F.softplus(self.scale_head(self.enc(x))) + 1e-3          # [B,1,H,W], the generalised-Gaussian scale parameter
        g, s = self.film_proj(sig / self.sigma0)
        return self.backbone(x, g, s), sig


def build_model(cfg):
    a = cfg["arch"]
    if a == "baseline":
        return BaselineNet()
    if a == "scale":
        return ScaleOnlyNet(init_scale=cfg.get("init_scale", 0.05))
    if a == "ldmh" and cfg.get("backbone") == "restormer":
        return RestormerLDMHNet(use_film=cfg.get("film", True), film_every=cfg.get("film_every", False),
                                beta_mode=cfg.get("beta_mode", "sigmoid"), init_scale=cfg.get("init_scale", 0.05))
    if a == "ldmh":
        return DistributionMixtureRestorationNet(use_film=cfg.get("film", True), beta_mode=cfg.get("beta_mode", "sigmoid"),
                                                 init_scale=cfg.get("init_scale", 0.05))
    if a.startswith("ext:"):
        return BASELINE_REGISTRY[a[4:]]()
    raise ValueError(a)


def ggd_nll_map(r_abs, scale, beta):
    """per-pixel NLL of a zero-mean generalised Gaussian (fp32): (|r|/alpha)^beta - log beta + log(2 alpha) + lgamma(1/beta)"""
    return (r_abs / scale).pow(beta) - math.log(beta) + torch.log(2 * scale) + math.lgamma(1.0 / beta)


def compute_loss(cfg, out, gt, gt_ds, noisy):
    pred = out[0] if is_ldmh(cfg) else out
    kind = cfg["loss"]
    if kind == "charbonnier":
        loss = charbonnier_loss(pred, gt)
    elif kind == "gen_charbonnier":
        loss = gen_charbonnier_loss(pred, gt, p=cfg.get("global_beta", 0.845))
    elif kind in ("mixture", "mixture_out"):
        _, w, beta, scale = out
        nll = mixture_gen_gaussian_nll(noisy - gt_ds, w, beta, scale)
        loss = (charbonnier_loss(pred, gt) + cfg.get("nll", 1.0) * nll
                + cfg.get("div", 0.0) * beta_diversity_penalty(beta, 0.3)
                + cfg.get("bal", 0.0) * load_balance_penalty_entropy(w))
        if kind == "mixture_out":      # + noise-model-weighted loss on the restoration error at LR resolution (stop-gradient on the noise model)
            d = F.avg_pool2d(pred.float(), 2) - gt_ds
            loss = loss + cfg.get("out_nll", 0.2) * mixture_gen_gaussian_nll(d, w.detach(), beta.detach(), scale.detach())
    elif kind == "scale_nll":
        _, sig = out
        with torch.autocast(device_type=gt.device.type, enabled=False):
            nll = ggd_nll_map((noisy - gt_ds).abs().float(), sig.float(), cfg.get("scale_beta", 1.53)).mean()
        loss = charbonnier_loss(pred, gt) + cfg.get("nll", 1.0) * nll
    else:
        raise ValueError(kind)
    return loss, pred


_MIX = dict(arch="ldmh", film=True, loss="mixture", div=0.1, bal=0.3)
CONFIGS.update({
    # ---- Part A: collapse ablation (full_clamp / full_no_bal / full_no_div already exist in the master CONFIGS) -------------
    "full_stress":       dict(_MIX, proto_lr_mult=50.0),                          # sigmoid beta, prototype lr x50: stress test
    "full_clamp_stress": dict(_MIX, beta_mode="clamp", proto_lr_mult=50.0),       # ORIGINAL hard clamp under the same stress
    # ---- Part B: scale-only control ------------------------------------------------------------------------------------
    "scale_only":        dict(arch="scale", loss="scale_nll", scale_beta=1.53),   # Hetero-GGD, beta fixed at the Table-I value
    "scale_only_g2":     dict(arch="scale", loss="scale_nll", scale_beta=2.0),    # Hetero-Gaussian
    "scale_control":     dict(arch="scale", loss="charbonnier"),                  # same net, no NLL (capacity control for scale_only)
    # ---- Part E: strong backbone -------------------------------------------------------------------------------------
    "Restormer-FiLM-control": dict(arch="ldmh", backbone="restormer", film=True, loss="charbonnier"),
    "Restormer-LDMH-full":    dict(_MIX, backbone="restormer"),
    "Restormer-LDMH-outnll":  dict(_MIX, backbone="restormer", loss="mixture_out", out_nll=0.2),
    "Restormer-LDMH-deep":    dict(_MIX, backbone="restormer", film_every=True),
    "full_outnll":            dict(_MIX, loss="mixture_out", out_nll=0.2),            # compact backbone + noise-model-weighted output loss
})
ARMS_A  = ["full", "full_clamp", "full_no_bal", "full_no_div", "full_stress", "full_clamp_stress"]
ARMS_B  = ["baseline", "film_control", "scale_control", "scale_only", "scale_only_g2", "full"]
ARMS_D2 = ["baseline", "film_control", "scale_only", "full"]
ARMS_E  = ["Restormer-small", "Restormer-FiLM-control", "Restormer-LDMH-full", "Restormer-LDMH-outnll", "Restormer-LDMH-deep"]

def finished_seeds(run_root, name):
    d = os.path.join(run_root, name)
    if not os.path.isdir(d): return []
    return sorted(int(s[4:]) for s in os.listdir(d) if s.startswith("seed") and os.path.exists(os.path.join(d, s, "results.json")))

def welch(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    d = x.mean() - y.mean()
    if len(x) < 2 or len(y) < 2: return d, np.nan, np.nan, np.nan
    vx, vy = x.var(ddof=1) / len(x), y.var(ddof=1) / len(y); se = np.sqrt(vx + vy)
    dfw = se ** 4 / (vx ** 2 / (len(x) - 1) + vy ** 2 / (len(y) - 1)); h = stats.t.ppf(.975, dfw) * se
    return d, d - h, d + h, float(stats.ttest_ind(x, y, equal_var=False).pvalue)

ALL_EFFECTS = {}
def effect(a, b, run_root=None, metric="psnr", label=None, store=True):
    """seed-level (Welch) and image-level (paired) effect of arm a minus arm b on the TEST split."""
    run_root = run_root or C.RUNS
    if not finished_seeds(run_root, a) or not finished_seeds(run_root, b): return None
    ra, rb = load_results(run_root, [a]), load_results(run_root, [b])
    x, y = ra[f"test_{metric}"].values, rb[f"test_{metric}"].values
    d, lo, hi, p = welch(x, y)
    pa, pb = load_per_image(run_root, a, metric), load_per_image(run_root, b, metric)
    img = paired_image_test(pa, pb, higher_better=(metric != "hp_err")) if pa.size and pb.size and pa.shape[-1] == pb.shape[-1] else {}
    r = dict(label=label or f"{a} - {b}", a=a, b=b, domain=os.path.basename(run_root), n_a=len(x), n_b=len(y), diff=d, ci_lo=lo, ci_hi=hi, p_welch=p,
             img_diff=img.get("mean_diff", np.nan), img_lo=img.get("ci95_lo", np.nan), img_hi=img.get("ci95_hi", np.nan), p_wilcoxon=img.get("p_wilcoxon", np.nan))
    if store: ALL_EFFECTS[(os.path.basename(run_root), a, b)] = r
    return r

def effect_table(pairs, run_root=None, metric="psnr"):
    rows = [effect(a, b, run_root, metric) for a, b in pairs]
    df = pd.DataFrame([r for r in rows if r is not None])
    if len(df): print(df[["label", "n_a", "n_b", "diff", "ci_lo", "ci_hi", "p_welch", "img_diff", "img_lo", "img_hi"]].round(4).to_string(index=False))
    return df
print("extensions ready | arms:", len(CONFIGS))

extensions ready | arms: 25


### 3.8 Architecture table - every arm built and forward-tested before any GPU time is spent

In [20]:
dev = C.device()
ALL_ARMS = ["baseline", "global_gc", "film_control", "full", "full_clamp", "full_no_bal", "full_no_div", "full_stress", "full_clamp_stress",
            "scale_only", "scale_only_g2", "scale_control", "full_outnll", "NAFNet-small", "Restormer-small", "SwinIR-small",
            "Restormer-FiLM-control", "Restormer-LDMH-full", "Restormer-LDMH-outnll", "Restormer-LDMH-deep"]
rows = []
for _n in ALL_ARMS:
    _m = build_model(dict(CONFIGS[_n]))
    rows.append(dict(arm=_n, network=type(_m).__name__, params=sum(p.numel() for p in _m.parameters()), loss=CONFIGS[_n]["loss"],
                     extra={k: v for k, v in CONFIGS[_n].items() if k not in ("arch", "loss", "film")}))
ARCH_TABLE = pd.DataFrame(rows); print(ARCH_TABLE.to_string(index=False))
ARCH_TABLE.to_csv(os.path.join(C.TABLES, "architectures.csv"), index=False)
# smoke-forward every arm on a random input to catch shape errors before any GPU time is spent
for _n in ALL_ARMS:
    _m = build_model(dict(CONFIGS[_n])).eval(); _x = torch.rand(2, 1, 32, 32)
    with torch.no_grad(): _o = _m(_x)
    _p = _o[0] if isinstance(_o, (tuple, list)) else _o
    assert _p.shape == (2, 1, 64, 64), (_n, _p.shape)
print("\nall", len(ALL_ARMS), "arms build and give a [B,1,2H,2W] output")

                   arm                           network  params            loss                                                                 extra
              baseline                       BaselineNet   86945     charbonnier                                                                    {}
             global_gc                       BaselineNet   86945 gen_charbonnier                                                                    {}
          film_control DistributionMixtureRestorationNet  116138     charbonnier                                                                    {}
                  full DistributionMixtureRestorationNet  116138         mixture                                              {'div': 0.1, 'bal': 0.3}
            full_clamp DistributionMixtureRestorationNet  116138         mixture                        {'div': 0.1, 'bal': 0.3, 'beta_mode': 'clamp'}
           full_no_bal DistributionMixtureRestorationNet  116138         mixture              

## Part 4 - Data audit and visualisation  (= notebook 00, plus extra figures)

## 1. Split manifest

In [21]:
splits, man = C.get_splits()
print({k: len(v) for k, v in splits.items()}, "| manifest sha1:", man["sha1"][:12])
assert not (set(man["train"]) & set(man["val"])) and not (set(man["train"]) & set(man["test"])) and not (set(man["val"]) & set(man["test"]))
print("OK: train/val/test are disjoint. Manifest saved to", os.path.join(C.OUT, "split_manifest.json"))
env = dict(torch=torch.__version__, cuda=torch.cuda.is_available(),
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           mode=C.MODE, epochs=C.EPOCHS, seeds=C.SEEDS, batch_size=C.BATCH_SIZE, lr=C.LR, split_sha1=man["sha1"])
json.dump(env, open(os.path.join(C.TABLES, "environment.json"), "w"), indent=1); print(env)

{'train': 2560, 'val': 320, 'test': 320} | manifest sha1: 31d7258b8c36
OK: train/val/test are disjoint. Manifest saved to /kaggle/working/results_pilot/split_manifest.json
{'torch': '2.11.0+cu130', 'cuda': True, 'gpu': 'Tesla T4', 'mode': 'pilot', 'epochs': 30, 'seeds': [0, 1, 2], 'batch_size': 8, 'lr': 0.0002, 'split_sha1': '31d7258b8c36e2fd1262b33776880d7ec6ed4ce2'}


## 2. Value ranges and the reference residual

In [22]:
rng = np.random.RandomState(0)
sample = [splits["train"][i] for i in rng.choice(len(splits["train"]), min(200, len(splits["train"])), replace=False)]
rows = []
for g, n in sample:
    gt, ny = load_npy(g), load_npy(n); ds = downsample(gt, ny.shape[:2])
    rows.append(dict(gt_min=gt.min(), gt_max=gt.max(), noisy_min=ny.min(), noisy_max=ny.max(),
                     frac_noisy_below0=(ny < 0).mean(), frac_noisy_above1=(ny > 1).mean(),
                     res_mean=(ny - ds).mean(), res_std=(ny - ds).std()))
audit = pd.DataFrame(rows)
print(audit.describe().loc[["mean", "min", "max"]].round(4).to_string())
print("\nGT shape:", gt.shape, " noisy shape:", ny.shape, "-> scale factor", gt.shape[0] // ny.shape[0])
print("NOTE: degraded values leave [0,1]; metrics clamp predictions to [0,1] and use data_range=1 (as the paper states).")
audit.describe().round(5).to_csv(os.path.join(C.TABLES, "data_audit.csv"))

      gt_min  gt_max  noisy_min  noisy_max  frac_noisy_below0  frac_noisy_above1  res_mean  res_std
mean     0.0     1.0    -0.0056     1.3896             0.0033             0.0358   -0.0000   0.0832
min      0.0     1.0    -0.1268     0.7221             0.0000             0.0000   -0.0024   0.0114
max      0.0     1.0     0.1115     1.9384             0.2163             0.3748    0.0018   0.1636

GT shape: (256, 256)  noisy shape: (128, 128) -> scale factor 2
NOTE: degraded values leave [0,1]; metrics clamp predictions to [0,1] and use data_range=1 (as the paper states).


In [23]:
# Is the reference downsampler D(.) the one the data provider used?  If not, 'residual' contains structured edge error.
# Diagnostic: residual std vs. integer misregistration of D(x) relative to y.  The minimum should be at (0,0).
import torch.nn.functional as F
from scipy import ndimage
def resid_std(shift, mode="area"):
    out = []
    for g, n in sample[:60]:
        gt, ny = load_npy(g), load_npy(n)
        if mode == "area": ds = downsample(gt, ny.shape[:2])
        elif mode == "bilinear": ds = F.interpolate(torch.from_numpy(gt)[None, None], size=ny.shape[:2], mode="bilinear", align_corners=False)[0, 0].numpy()
        else: ds = ndimage.zoom(gt, ny.shape[0] / gt.shape[0], order=1)      # what the 'validates the statistical claim' notebooks used
        ds = np.roll(ds, shift, axis=(0, 1)); out.append((ny - ds)[2:-2, 2:-2].std())
    return float(np.mean(out))
grid = pd.DataFrame({f"dx={dx}": [resid_std((dy, dx)) for dy in (-1, 0, 1)] for dx in (-1, 0, 1)}, index=[f"dy={d}" for d in (-1, 0, 1)])
print("residual std vs misregistration (area-average D):\n", grid.round(5).to_string())
modes = {m: resid_std((0, 0), m) for m in ("area", "bilinear", "zoom_order1")}
print("\nresidual std by choice of D:", {k: round(v, 5) for k, v in modes.items()})
best = np.unravel_index(grid.values.argmin(), grid.shape)
print("minimum at", grid.index[best[0]], grid.columns[best[1]], "->", "OK (aligned)" if best == (1, 1) else "WARNING: reference D is mis-registered/different; report it")
grid.to_csv(os.path.join(C.TABLES, "residual_definition_check.csv"))

residual std vs misregistration (area-average D):
          dx=-1     dx=0     dx=1
dy=-1  0.12072  0.10846  0.12249
dy=0   0.10959  0.08334  0.11022
dy=1   0.12201  0.10820  0.12035

residual std by choice of D: {'area': 0.08334, 'bilinear': 0.08334, 'zoom_order1': 0.07964}
minimum at dy=0 dx=0 -> OK (aligned)


## 3. What do the images look like?  (state it honestly in the paper)
What do the images look like? Look at the contact sheet and describe the data accurately in the paper.

In [24]:
import types
def _mk(names): return types.SimpleNamespace(**{k: globals()[k] for k in names})
M  = _mk(["psnr", "ssim", "hp_err", "edge_preservation", "edge_f1", "artifact_energy", "cnr", "full_eval"])
D  = _mk(["ggd_beta_moment", "patch_table", "homogeneous_null", "heteroscedastic_null", "global_gc" if False else "global_model_comparison",
          "cluster_bootstrap_corr", "partial_corr", "variance_decomposition", "compare_observed_vs_null"])
FG = _mk(["PALETTE", "save", "bars_with_error", "training_curves", "component_health", "mixture_maps", "qualitative_grid", "scatter_beta"])
print("M, D, FG repaired:", hasattr(FG, "save"), hasattr(M, "psnr"), hasattr(D, "patch_table"))

M, D, FG repaired: True True True


In [25]:
idx = np.random.RandomState(1).choice(len(splits["train"]), min(16, len(splits["train"])), replace=False)
fig, ax = plt.subplots(4, 4, figsize=(8, 8))
names = []
for a, i in zip(ax.ravel(), idx):
    g, _ = splits["train"][i]; a.imshow(load_npy(g), cmap="gray", vmin=0, vmax=1); a.axis("off"); names.append(os.path.basename(g))
FG.save(fig, C.FIGS, "fig_data_contact_sheet", {"split": "train", "files": names, "selection": "random, seed=1"})
print("saved contact sheet:", names[:4], "...")

saved contact sheet: ['001772.npy', '003121.npy', '000570.npy', '002105.npy'] ...


## 4. Estimate the degradation parameters from TRAIN pairs only
Moment matching `E[r^2 | I] = a I^2 + b` on flat regions (a = 1/(4 L), b = sigma^2/4). `r2_signal_dependence` tells you whether the model form fits at all.

In [26]:
fit = fit_degradation(splits["train"], n_images=min(400, len(splits["train"])))
json.dump(fit, open(os.path.join(C.TABLES, "degradation_fit.json"), "w"), indent=1)
print(json.dumps(fit, indent=1))
print(f"\nUse looks={fit['looks']:.2f}, sigma={fit['sigma']:.4f} for ALL synthetic degradations (robustness / OOD / downstream).")
if fit["r2_signal_dependence"] < 0.8:
    print("WARNING: weak signal-dependence fit - the multiplicative-speckle + Gaussian model may not describe this data; say so in the paper.")

{
 "looks": 9.259709358215332,
 "sigma": 0.04252737760543823,
 "a": 0.02699868753552437,
 "b": 0.00045214444980956614,
 "r2_signal_dependence": 0.9977861642837524,
 "mean_residual": 8.677723235450685e-05,
 "n_pixels": 2621600,
 "note": "moment-matching estimate; check r2_signal_dependence before trusting looks/sigma"
}

Use looks=9.26, sigma=0.0425 for ALL synthetic degradations (robustness / OOD / downstream).


### Extra figures: pairs, value distributions, and whether the noise scale depends on the signal

In [27]:
from scipy import stats as _st
splits, man = C.get_splits(); sizes = {k: len(v) for k, v in splits.items()}
if sizes["train"] < 500: print("WARNING: fewer than 500 training images - check that this is the full data set.")
rng = np.random.RandomState(0)
sample = [splits["train"][i] for i in rng.choice(sizes["train"], min(200, sizes["train"]), replace=False)]

# figure: noisy input | clean ground truth | residual y - D(x)
idx = rng.choice(sizes["train"], 4, replace=False)
fig, ax = plt.subplots(4, 3, figsize=(9, 12))
for r_, i in enumerate(idx):
    g, n = splits["train"][i]; gt, ny = load_npy(g), load_npy(n); res = ny - downsample(gt, ny.shape[:2])
    ax[r_, 0].imshow(ny, cmap="gray", vmin=0, vmax=1); ax[r_, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    ax[r_, 2].imshow(res, cmap="coolwarm", vmin=-0.3, vmax=0.3); ax[r_, 0].set_ylabel(os.path.basename(g), fontsize=8)
    for a in ax[r_]: a.set_xticks([]); a.set_yticks([])
for a, t in zip(ax[0], ["noisy low-res input", "clean ground truth", "residual y - D(x)"]): a.set_title(t, fontsize=10)
fig.suptitle("Training pairs (random, seed 0)", y=0.995); fig.tight_layout(); savefig(fig, "fig_pairs", {"split": "train", "files": [os.path.basename(splits["train"][i][0]) for i in idx], "selection": "random, seed 0"}); plt.show()

# figure: value distributions and residual histogram (linear and log)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
vals_gt = np.concatenate([load_npy(g).ravel()[::13] for g, _ in sample[:60]]); vals_ny = np.concatenate([load_npy(n).ravel()[::7] for _, n in sample[:60]])
ax[0].hist(vals_gt, 80, alpha=.6, density=True, label="GT"); ax[0].hist(vals_ny, 80, alpha=.6, density=True, label="noisy"); ax[0].set_title("pixel values"); ax[0].legend()
res_all = np.concatenate([(load_npy(n) - downsample(load_npy(g), load_npy(n).shape[:2])).ravel()[::5] for g, n in sample[:60]])
xs = np.linspace(res_all.min(), res_all.max(), 400)
ax[1].hist(res_all, 200, density=True, alpha=.6); ax[1].plot(xs, _st.norm.pdf(xs, 0, res_all.std()), "r", lw=1, label="Gaussian, same SD"); ax[1].set_title("residual (linear)"); ax[1].legend()
ax[2].hist(res_all, 200, density=True, alpha=.6); ax[2].plot(xs, _st.norm.pdf(xs, 0, res_all.std()), "r", lw=1); ax[2].set_yscale("log"); ax[2].set_ylim(1e-3, None); ax[2].set_title("residual (log y): tails")
fig.tight_layout(); savefig(fig, "fig_distributions", {"split": "train", "n_images": 60, "pooled_excess_kurtosis": float(_st.kurtosis(res_all))}); plt.show()
print(f"pooled residual excess kurtosis = {_st.kurtosis(res_all):.2f}   (0 = Gaussian; >0 = heavy tails when pixels are pooled)")

# figure: is the noise scale signal-dependent?  Var(r | local intensity) = a*I^2 + b
I_all, V_all = [], []
for g, n in sample[:100]:
    gt, ny = load_npy(g), load_npy(n); ds = downsample(gt, ny.shape[:2]); I_all.append(ds.ravel()[::3]); V_all.append(((ny - ds) ** 2).ravel()[::3])
I_all, V_all = np.concatenate(I_all), np.concatenate(V_all)
bins = np.linspace(0.02, 0.98, 25); which = np.digitize(I_all, bins)
keep = [k for k in range(1, len(bins)) if (which == k).sum() > 200]
bx = np.array([I_all[which == k].mean() for k in keep]); by = np.array([V_all[which == k].mean() for k in keep])
A = np.vstack([bx ** 2, np.ones_like(bx)]).T; (a_, b_), *_ = np.linalg.lstsq(A, by, rcond=None)
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.plot(bx, by, "o", label="measured Var(r | local intensity)"); ax.plot(bx, a_ * bx ** 2 + b_, "r-", label=f"a*I^2 + b   (a={a_:.4f}, b={b_:.5f})")
ax.set_xlabel("clean local intensity I"); ax.set_ylabel("residual variance"); ax.legend(); ax.set_title("Does the noise scale depend on the signal?")
fig.tight_layout(); savefig(fig, "fig_noise_vs_intensity", {"a": float(a_), "b": float(b_), "split": "train", "n_images": 100}); plt.show()

if TEST_NOISY:
    fs = sorted(_npy(TEST_NOISY))[:4]
    fig, ax = plt.subplots(1, len(fs), figsize=(3 * len(fs), 3))
    for a, f in zip(np.atleast_1d(ax), fs): a.imshow(load_npy(f), cmap="gray"); a.set_title(os.path.basename(f), fontsize=8); a.axis("off")
    fig.suptitle("Test folder (noisy only - no ground truth, so no metric can be computed)"); fig.tight_layout(); savefig(fig, "fig_test_noisy", {"files": [os.path.basename(f) for f in fs]}); plt.show()
save_now("(after data audit)")

pooled residual excess kurtosis = 3.33   (0 = Gaussian; >0 = heavy tails when pixels are pooled)
saved /kaggle/working/results_latest.zip (77.0 MB) (after data audit)


## Part 5 - Statistical analysis of the residual distribution  (= notebook 01)
Decision gate 1: does signal-dependent noise explain the heavy tails and the beta-structure correlations? Uses the training split only.

In [28]:
from scipy import stats
splits, man = C.get_splits(); tr = splits["train"]     # analysis uses the TRAIN split only (no val/test leakage)
LR = load_npy(tr[0][1]).shape[0]; P = 32 if LR >= 128 else 16
n_img = ANALYSIS_IMAGES                  # set in Part 1
print("patch size", P, "| images used:", min(n_img, len(tr)))

patch size 32 | images used: 400


## 1. Global model comparison (replaces Table I)
Homoscedastic families as in the paper, **plus intensity-dependent-scale models**. If a heteroscedastic model beats the global GGD, the 'heavy-tailed' premise must be reframed.

In [29]:
cmp_, ab = D.global_model_comparison(tr, n_images=min(200, len(tr)))
print(cmp_.round(1).to_string(index=False)); print(ab)
cmp_.to_csv(os.path.join(C.TABLES, "tableI_global_model_comparison.csv"), index=False)
best = cmp_.iloc[0]["model"]
print("\nBest model by AIC:", best, "->", "heteroscedastic structure matters" if best.startswith("Hetero") else "global GGD preferred")

                       model  k    loglik        AIC  dAIC_vs_best
      Hetero-GGD (beta=1.53)  3 4562984.7 -9125963.4           0.0
Hetero-Gaussian  N(0,aI^2+b)  2 4528255.4 -9056506.7       69456.7
                GGD (global)  2 3675453.8 -7350903.7     1775059.7
                     Laplace  1 3651217.5 -7302433.0     1823530.4
                   Student-t  2 3580603.7 -7161203.4     1964760.0
                    Gaussian  1 3284801.5 -6569600.9     2556362.5
{'a': 0.03406723900807617, 'b': 5.1708286740514505e-05, 'ggd_beta': 0.7978815716514408, 'ggd_alpha': 0.04100733841282936, 'n': 3276800}

Best model by AIC: Hetero-GGD (beta=1.53) -> heteroscedastic structure matters


## 2. Patch-level table
Fast moment estimator of beta (checked against the MLE below), residual statistics and *clean-image* features per 32x32 patch.

In [30]:
tab = D.patch_table(tr, P, max_images=n_img)
print(len(tab), "patches |", f"beta mean={tab.beta.mean():.3f} std={tab.beta.std():.3f}")
chk = []
for g, n in tr[:min(20, len(tr))]:
    gt, ny = load_npy(g), load_npy(n); r = ny - downsample(gt, ny.shape[:2])
    for i in range(0, ny.shape[0] - P + 1, P):
        for j in range(0, ny.shape[1] - P + 1, P):
            p = r[i:i+P, j:j+P].ravel(); chk.append((D.ggd_beta_moment(p[None])[0], stats.gennorm.fit(p, floc=0)[0]))
chk = np.array(chk)
print(f"moment vs MLE beta over {len(chk)} patches: Spearman rho = {stats.spearmanr(chk[:,0], chk[:,1])[0]:.3f}, mean moment={chk[:,0].mean():.3f}, mean MLE={chk[:,1].mean():.3f}")

6400 patches | beta mean=1.450 std=0.445
moment vs MLE beta over 320 patches: Spearman rho = 0.996, mean moment=1.579, mean MLE=1.569


## 3. What does the heterogeneity look like when tested properly?

In [31]:
vd = D.variance_decomposition(tab)
print("variance of patch beta  BETWEEN images:", round(vd["var_between"], 4), "| WITHIN images:", round(vd["var_within"], 4), "| ICC =", round(vd["ICC"], 3))
feats = ["edge_density", "gt_var", "entropy"]
obs = {f: D.cluster_bootstrap_corr(tab, f, n_boot=300 if C.MODE != "full" else 1000) for f in feats}
part = {f: D.partial_corr(tab, f, "beta", "gt_mean") for f in feats}
for f in feats:
    print(f"r(beta, {f:13s}) = {obs[f]['r']:+.3f}  image-clustered 95% CI {tuple(round(v,3) for v in obs[f]['ci95'])}   |  partial r | local intensity = {part[f]:+.3f}")

variance of patch beta  BETWEEN images: 0.1022 | WITHIN images: 0.096 | ICC = 0.516
r(beta, edge_density ) = -0.373  image-clustered 95% CI (-0.431, -0.319)   |  partial r | local intensity = -0.470
r(beta, gt_var       ) = -0.503  image-clustered 95% CI (-0.539, -0.472)   |  partial r | local intensity = -0.678
r(beta, entropy      ) = -0.420  image-clustered 95% CI (-0.467, -0.374)   |  partial r | local intensity = -0.634


## 4. Null models
**Homogeneous-shape null**: every pixel from the *same* GGD (global beta) - what pure estimation noise gives.  
**Heteroscedastic-Gaussian null**: `r ~ N(0, a I^2 + b)` given the clean intensity - every pixel exactly Gaussian, *no shape heterogeneity*.

In [32]:
n_null = min(n_img, 300)
null_h = D.homogeneous_null(tr, ab["ggd_beta"], patch=P, max_images=n_null)
null_g = D.heteroscedastic_null(tr, ab["a"], ab["b"], patch=P, max_images=n_null)
table = D.compare_observed_vs_null(tab, {"homogeneous GGD null": null_h, "hetero-Gaussian null": null_g}, feats)
print(table.round(3).to_string(index=False)); table.to_csv(os.path.join(C.TABLES, "section4_observed_vs_null.csv"), index=False)

               table  beta_mean  beta_std  r(beta,edge_density)  r(beta,gt_var)  r(beta,entropy)
            observed      1.450     0.445                -0.373          -0.503           -0.420
homogeneous GGD null      0.805     0.053                -0.044          -0.030           -0.028
hetero-Gaussian null      1.441     0.452                -0.480          -0.562           -0.496


In [33]:
# ---- decision rule ----
rows = []
for f in feats:
    r_obs, (lo, hi) = obs[f]["r"], obs[f]["ci95"]
    r_null = table.loc[table.table == "hetero-Gaussian null", f"r(beta,{f})"].item()
    rows.append(dict(feature=f, r_observed=r_obs, ci_lo=lo, ci_hi=hi, r_hetero_null=r_null, null_inside_CI=bool(lo <= r_null <= hi)))
dec = pd.DataFrame(rows)
sd_obs, sd_null = tab.beta.std(), null_g.beta.std()
excess_sd = float(np.sqrt(max(sd_obs ** 2 - sd_null ** 2, 0)))
mean_gap = float(tab.beta.mean() - null_g.beta.mean())
print(dec.round(3).to_string(index=False))
print(f"\npatch-beta SD: observed {sd_obs:.3f} vs hetero-Gaussian null {sd_null:.3f}  -> EXCESS SD attributable to genuine shape variation = {excess_sd:.3f}")
print(f"mean patch beta: observed {tab.beta.mean():.3f} vs null {null_g.beta.mean():.3f}  (gap {mean_gap:+.3f})")
dec["observed_exceeds_null"] = dec.ci_hi < dec.r_hetero_null          # observed MORE negative than the null (CI entirely beyond it)
explained = (not dec.observed_exceeds_null.any()) and excess_sd < 0.5 * sd_null
verdict = ("NOT SUPPORTED: signal-dependent (heteroscedastic) Gaussian noise reproduces the heavy-tailed pooled fit, the patch-beta spread and the "
           "feature correlations. Reframe Section IV as 'signal-dependent noise scale', and test the method against a scale-only model."
           if explained else
           "PARTLY/FULLY SUPPORTED: observed heterogeneity exceeds the heteroscedastic-Gaussian null on at least one criterion; "
           "report the excess and keep the shape-heterogeneity motivation, scoped to the quantified excess.")
print("\nVERDICT:", verdict)
json.dump({"global_best_model": best, "ab": ab, "ICC": vd, "observed": obs, "partial_corr_vs_local_intensity": part,
           "excess_sd": excess_sd, "mean_gap": mean_gap, "heterogeneity_of_shape_supported": (not explained), "verdict": verdict},
          open(os.path.join(C.TABLES, "section4_summary.json"), "w"), indent=1, default=float)

     feature  r_observed  ci_lo  ci_hi  r_hetero_null  null_inside_CI
edge_density      -0.373 -0.431 -0.319         -0.480           False
      gt_var      -0.503 -0.539 -0.472         -0.562           False
     entropy      -0.420 -0.467 -0.374         -0.496           False

patch-beta SD: observed 0.445 vs hetero-Gaussian null 0.452  -> EXCESS SD attributable to genuine shape variation = 0.000
mean patch beta: observed 1.450 vs null 1.441  (gap +0.009)

VERDICT: NOT SUPPORTED: signal-dependent (heteroscedastic) Gaussian noise reproduces the heavy-tailed pooled fit, the patch-beta spread and the feature correlations. Reframe Section IV as 'signal-dependent noise scale', and test the method against a scale-only model.


## 5. Figures (caption numbers come from the saved JSON - do not retype them)

In [34]:
prov = {"split": "train", "n_patches": int(len(tab)), "patch": P, "estimator": "moment (checked vs MLE)",
        "r_observed": {f: obs[f]["r"] for f in feats}}
FG.scatter_beta(tab, feats, C.FIGS, "fig_beta_vs_structure_observed", prov)
FG.scatter_beta(null_g, feats, C.FIGS, "fig_beta_vs_structure_hetero_null", {**prov, "table": "heteroscedastic-Gaussian null"})
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.hist(tab.beta, bins=50, alpha=.6, density=True, label="observed"); ax.hist(null_g.beta, bins=50, alpha=.6, density=True, label="hetero-Gaussian null")
ax.hist(null_h.beta, bins=50, alpha=.6, density=True, label="homogeneous GGD null"); ax.set_xlabel("patch beta"); ax.legend(); ax.set_title("Is patch-beta spread beyond nulls?")
FG.save(fig, C.FIGS, "fig_patch_beta_vs_nulls", {"sd_obs": sd_obs, "sd_null_hetero": sd_null, "excess_sd": excess_sd})
print("figures saved to", C.FIGS)

figures saved to /kaggle/working/results_pilot/figures


## Part 6 - LDMH unit tests  (= notebook 02)
Each check prints PASS / FAIL; nothing is hidden.

In [35]:
import torch.nn.functional as F
def check(name, ok, extra=""): print(("PASS " if ok else "FAIL ") + name, extra)
RESULTS = {}

In [36]:
RESULTS = {}
n = lambda m: sum(p.numel() for p in m.parameters())
check("baseline parameter count == 86,945", n(BaselineNet()) == 86945, n(BaselineNet()))
check("full parameter count == 116,138", n(DistributionMixtureRestorationNet()) == 116138, n(DistributionMixtureRestorationNet()))

PASS baseline parameter count == 86,945 86945
PASS full parameter count == 116,138 116138


In [37]:
# 2. clamp: tied betas beyond the ceiling -> zero gradient from the diversity penalty (paper Section VI-A)
l = torch.tensor([5.0, 5.2], requires_grad=True)
b = torch.exp(l).clamp(0.3, 2.5); beta_diversity_penalty(b).backward()
check("clamp: tied betas are bitwise identical", bool(b[0] == b[1]), b.tolist())
check("clamp: diversity-penalty gradient is exactly zero", bool((l.grad == 0).all()), l.grad.tolist())
l2 = torch.tensor([5.0, 5.2], requires_grad=True)
b2 = 0.3 + 2.2 * torch.sigmoid(l2); beta_diversity_penalty(b2).backward()
check("sigmoid: same configuration still receives gradient", bool((l2.grad != 0).all()), l2.grad.tolist())

PASS clamp: tied betas are bitwise identical [2.5, 2.5]
PASS clamp: diversity-penalty gradient is exactly zero [0.0, 0.0]
PASS sigmoid: same configuration still receives gradient [0.0146256722509861, -0.01200353167951107]


In [38]:
# 3. float32 saturation of the sigmoid: where does the gradient become EXACTLY zero?
sat = None
for z in np.arange(5, 40, 0.5):
    t = torch.tensor([float(z)], requires_grad=True); (0.3 + 2.2 * torch.sigmoid(t)).sum().backward()
    if t.grad.item() == 0.0: sat = float(z); break
print("float32: d beta / d l becomes exactly 0 at pre-activation l >=", sat)
bound = lambda l: 0.3 + 2.2 / (1 + np.exp(-l))
print("beta at that point:", bound(sat), "-> the paper's 'never exactly zero' holds mathematically but NOT in float32; say 'practically vanishing'.")
RESULTS["sigmoid_float32_saturation_l"] = sat

float32: d beta / d l becomes exactly 0 at pre-activation l >= 17.0
beta at that point: 2.4999999089213736 -> the paper's 'never exactly zero' holds mathematically but NOT in float32; say 'practically vanishing'.


In [39]:
# 4. the old NLL's clamp(min=eps) on r/scale: conceptually the same defect, but how large is the dead zone in practice?
def nll_old(res, w, beta, scale, eps=1e-6):
    K = beta.shape[0]; r = res.abs()
    b_, s_ = beta.view(1, K, 1, 1), scale.view(1, K, 1, 1)
    lc = torch.log(b_) - torch.log(2 * s_) - torch.lgamma(1.0 / b_) - (r / s_).clamp(min=eps).pow(b_)
    return -torch.logsumexp(torch.log(w + eps) + lc, 1).mean()
def grad_beta(fn, r0):
    res = torch.full((1, 1, 8, 8), r0); w = torch.full((1, 3, 8, 8), 1 / 3)
    be = torch.tensor([0.5, 1.0, 2.0], requires_grad=True); sc = torch.tensor([0.05] * 3, requires_grad=True)
    fn(res, w, be, sc).backward(); return be.grad.clone(), sc.grad.clone()
# isolate the |r/s|^beta term's gradient: difference between a tiny and a typical residual
for nm, fn in [("old (clamp)", nll_old), ("v2 (additive eps)", mixture_gen_gaussian_nll)]:
    g_tiny, _ = grad_beta(fn, 1e-9); g_zero, _ = grad_beta(fn, 0.0)
    print(f"{nm:18s} d NLL/d beta at |r|=1e-9: {g_tiny.tolist()}   |r|=0: {g_zero.tolist()}")
frac = float((np.abs(np.random.RandomState(0).normal(0, 0.085, 10 ** 6)) < 1e-6 * 0.05).mean())
print(f"fraction of N(0,0.085) residuals inside the old dead zone (|r| < eps*scale): {frac:.2e}")
check("old-NLL dead zone is negligible in practice (<1e-5 of pixels): v2 change is a cleanliness fix, NOT a source of gains", frac < 1e-5)

old (clamp)        d NLL/d beta at |r|=1e-9: [-0.7042274475097656, -0.16088926792144775, -0.003916993737220764]   |r|=0: [-0.7042274475097656, -0.16088926792144775, -0.003916993737220764]
v2 (additive eps)  d NLL/d beta at |r|=1e-9: [-0.7088098526000977, -0.16107071936130524, -0.003919616341590881]   |r|=0: [-0.7088073492050171, -0.16107060015201569, -0.0039196014404296875]
fraction of N(0,0.085) residuals inside the old dead zone (|r| < eps*scale): 0.00e+00
PASS old-NLL dead zone is negligible in practice (<1e-5 of pixels): v2 change is a cleanliness fix, NOT a source of gains 


In [40]:
# 5. temperature is applied in v2
torch.manual_seed(0)
x = torch.randn(1, 1, 16, 16)
def ent(T):
    torch.manual_seed(1); m = LocalDistributionMixtureHead(temperature=T)
    w = m(x)[0]; return float(-(w * (w + 1e-9).log()).sum(1).mean())
e1, e15 = ent(1.0), ent(1.5)
check("T=1.5 gives flatter mixture weights than T=1.0 (temperature is applied)", e15 > e1, f"entropy {e1:.4f} -> {e15:.4f}")

PASS T=1.5 gives flatter mixture weights than T=1.0 (temperature is applied) entropy 1.0645 -> 1.0829


## 6. Synthetic recovery of a known partition
Truth: blob regions where residuals are heavy-tailed (beta 0.6) vs near-Gaussian (beta 1.9). The head sees only the **observable** noisy image (texture amplitude differs by region), not the residual. Repeated over 5 seeds. Thresholds fixed in advance: pixel agreement > 0.90 and each recovered beta within 25% of truth.

In [41]:
from scipy import ndimage, stats
def make_field(seed, S=96):
    rng = np.random.RandomState(seed)
    mask = ndimage.gaussian_filter(rng.randn(S, S), 8) > 0
    resid = np.where(mask, stats.gennorm.rvs(0.6, scale=0.03, size=(S, S), random_state=rng), stats.gennorm.rvs(1.9, scale=0.05, size=(S, S), random_state=rng))
    tex = rng.randn(S, S) * np.where(mask, 0.25, 0.05)                           # observable structure differs by region
    return mask, resid.astype(np.float32), (0.5 + tex).astype(np.float32)
steps = 40 if C.MODE == "smoke" else 400
rows = []
for seed in range(5):
    mask, resid, img = make_field(seed)
    torch.manual_seed(seed)
    m = LocalDistributionMixtureHead(base_ch=16, n_components=2, init_scale=0.04)
    opt = torch.optim.Adam(m.parameters(), lr=5e-3)
    xi, rr = torch.from_numpy(img)[None, None], torch.from_numpy(resid)[None, None]
    for _ in range(steps):
        w, be, sc, _ = m(xi); loss = mixture_gen_gaussian_nll(rr, w, be, sc); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): w, be, sc, _ = m(xi)
    heavy = int(be.argmin()); pred = (w[0, heavy] > 0.5).numpy()
    agree = max((pred == mask).mean(), (pred != mask).mean())                  # label-switch invariant
    # context: a NON-learned detector of the same observable cue (local std of the input, Otsu threshold)
    from skimage.filters import threshold_otsu
    ls = np.sqrt(np.maximum(ndimage.uniform_filter(img ** 2, 5) - ndimage.uniform_filter(img, 5) ** 2, 0)); base_pred = ls > threshold_otsu(ls)
    base_agree = max((base_pred == mask).mean(), (base_pred != mask).mean())
    rows.append(dict(seed=seed, agreement=agree, local_std_baseline=base_agree, beta_lo=float(be.min()), beta_hi=float(be.max()), scale_lo=float(sc[be.argmin()]), scale_hi=float(sc[be.argmax()])))
rec = pd.DataFrame(rows); print(rec.round(3).to_string(index=False))
print("truth: beta 0.6 (scale 0.03) vs 1.9 (scale 0.05)")
ok_a = bool((rec.agreement > 0.90).all()); ok_o = bool(rec.agreement.mean() > 0.85)
ok_b = bool(((rec.beta_lo - 0.6).abs() / 0.6 < 0.25).all() and ((rec.beta_hi - 1.9).abs() / 1.9 < 0.25).all())
check("strict criterion (pre-registered): agreement > 0.90 in ALL seeds", ok_a, f"mean={rec.agreement.mean():.3f}")
check("original notebook-04 criterion: mean agreement > 0.85 (the submitted run scored 0.674)", ok_o)
check("prototype betas recovered within 25% of truth in all seeds", ok_b)
print(f"context: non-learned local-std baseline reaches {rec.local_std_baseline.mean():.3f}; LDMH {rec.agreement.mean():.3f}. "
      f"If LDMH is not clearly above this baseline, its spatial routing adds nothing beyond a simple local-variance cue.")
RESULTS.update(recovery_agreement_mean=float(rec.agreement.mean()), local_std_baseline_mean=float(rec.local_std_baseline.mean()),
               recovery_strict_pass=ok_a, recovery_original_criterion_pass=ok_o, beta_recovery_pass=ok_b)
if not (ok_a and ok_b): print("-> Report this honestly. Do NOT claim that LDMH 'learns local distribution structure' beyond what this shows.")
rec.to_csv(os.path.join(C.TABLES, "ldmh_synthetic_recovery.csv"), index=False)
json.dump(RESULTS, open(os.path.join(C.TABLES, "ldmh_unit_tests.json"), "w"), indent=1, default=float)

 seed  agreement  local_std_baseline  beta_lo  beta_hi  scale_lo  scale_hi
    0      0.878               0.970    0.493    2.245     0.027     0.054
    1      0.716               0.957    0.500    2.300     0.028     0.053
    2      0.841               0.970    0.492    2.119     0.027     0.051
    3      0.841               0.963    0.490    2.212     0.028     0.054
    4      0.712               0.970    0.496    2.229     0.028     0.054
truth: beta 0.6 (scale 0.03) vs 1.9 (scale 0.05)
FAIL strict criterion (pre-registered): agreement > 0.90 in ALL seeds mean=0.797
FAIL original notebook-04 criterion: mean agreement > 0.85 (the submitted run scored 0.674) 
PASS prototype betas recovered within 25% of truth in all seeds 
context: non-learned local-std baseline reaches 0.966; LDMH 0.797. If LDMH is not clearly above this baseline, its spatial routing adds nothing beyond a simple local-variance cue.
-> Report this honestly. Do NOT claim that LDMH 'learns local distribution structu

## 6b. Mechanism demo (CPU, ~1 minute): can the diversity / balance terms rescue a clamped parameterisation?
Original failure of the paper: two prototype betas were pushed beyond the clamp ceiling and became **bitwise identical**; the diversity penalty then has zero gradient. Here the *same situation* is created on a synthetic residual field with three known regions (beta 0.6 / 1.4 / 2.0) and the head is trained with the full loss (NLL + diversity + balance). Five seeds per parameterisation. This is a controlled demonstration of the mechanism; the GPU runs in Parts A (`*_stress`) test it inside the full network.

In [42]:
def make_field3(seed, S=96):
    rng = np.random.RandomState(seed)
    lab = np.digitize(ndimage.gaussian_filter(rng.randn(S, S), 8), [-0.15, 0.15])            # 3 smooth regions
    betas, scales = [0.6, 1.4, 2.0], [0.03, 0.04, 0.05]
    resid = np.zeros((S, S), np.float32)
    for k in range(3):
        m = lab == k; resid[m] = stats.gennorm.rvs(betas[k], scale=scales[k], size=m.sum(), random_state=rng)
    tex = rng.randn(S, S) * np.array([0.05, 0.15, 0.25])[lab]                               # observable cue differs by region
    return resid, (0.5 + tex).astype(np.float32)

def collapse_demo(mode, seed, steps=300, div=0.1, bal=0.3):
    resid, img = make_field3(seed); torch.manual_seed(seed)
    m = LocalDistributionMixtureHead(base_ch=16, n_components=3, beta_mode=mode, init_scale=0.04)
    with torch.no_grad():                                   # 'two prototypes pushed beyond the ceiling' (the original failure state)
        if mode == "clamp": m.log_beta.copy_(torch.log(torch.tensor([3.4, 3.1, 1.0])))      # exp(l) > 2.5 -> clamped to exactly 2.5 (bitwise tied)
        else:               m.log_beta.copy_(torch.tensor([6.0, 5.0, 0.0]))                  # sigmoid: beta ~ 2.49 / 2.48 (same 'pushed to the ceiling')
    b0 = m.betas().detach().clone()
    opt = torch.optim.Adam(m.parameters(), lr=5e-3)
    xi, rr = torch.from_numpy(img)[None, None], torch.from_numpy(resid)[None, None]
    for _ in range(steps):
        w, be, sc, _ = m(xi)
        loss = mixture_gen_gaussian_nll(rr, w, be, sc) + div * beta_diversity_penalty(be, 0.3) + bal * load_balance_penalty_entropy(w)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): w, be, sc, _ = m(xi)
    d0 = float(torch.pdist(b0.unsqueeze(1)).min()); d1 = float(torch.pdist(be.unsqueeze(1)).min())
    return dict(mode=mode, seed=seed, beta_start=[round(float(v), 3) for v in b0], beta_end=[round(float(v), 3) for v in be],
                min_dist_start=d0, min_dist_end=d1, min_usage=float(w.mean((0, 2, 3)).min()), tied_at_end=bool(d1 < 0.05))

MECH = pd.DataFrame([collapse_demo(md, sd) for md in ("clamp", "sigmoid") for sd in range(5)])
print(MECH[["mode", "seed", "beta_start", "beta_end", "min_dist_start", "min_dist_end", "min_usage", "tied_at_end"]].round(3).to_string(index=False))
MECH_SUMMARY = MECH.groupby("mode").agg(n=("seed", "size"), tied_at_end=("tied_at_end", "sum"), mean_min_dist_end=("min_dist_end", "mean")).round(3)
print("\n", MECH_SUMMARY.to_string())
print("\n=> clamp: tied betas stay tied (zero gradient, nothing can separate them); sigmoid: the same start is pulled apart again (min distance > 0.05); the demo shows the mechanism, not that the full network always avoids collapse (see the *_stress runs).")
MECH.to_csv(os.path.join(C.TABLES, "collapse_mechanism_demo.csv"), index=False)

   mode  seed          beta_start              beta_end  min_dist_start  min_dist_end  min_usage  tied_at_end
  clamp     0     [2.5, 2.5, 1.0]     [2.5, 2.5, 2.198]           0.000         0.000      0.294         True
  clamp     1     [2.5, 2.5, 1.0]     [2.5, 2.5, 2.197]           0.000         0.000      0.320         True
  clamp     2     [2.5, 2.5, 1.0]     [2.5, 2.5, 2.195]           0.000         0.000      0.306         True
  clamp     3     [2.5, 2.5, 1.0]     [2.5, 2.5, 2.194]           0.000         0.000      0.295         True
  clamp     4     [2.5, 2.5, 1.0]     [2.5, 2.5, 2.188]           0.000         0.000      0.303         True
sigmoid     0 [2.495, 2.485, 1.4]   [2.498, 2.399, 1.5]           0.009         0.100      0.294        False
sigmoid     1 [2.495, 2.485, 1.4] [2.498, 2.406, 1.612]           0.009         0.093      0.318        False
sigmoid     2 [2.495, 2.485, 1.4] [2.498, 2.398, 1.592]           0.009         0.101      0.316        False
sigmoid   

## Part 7 - Second noise domain (Poisson-Gaussian shot noise)
Same clean images, same split manifest, a **different noise family**: `y = D( Poisson(peak * x) / peak + N(0, sigma^2) )`. The variance now grows **linearly** with intensity (`x/peak + sigma^2`) instead of quadratically (speckle). If the module only helps because it learns "speckle", it should not help here; if it learns signal-dependent noise statistics in general, it should.

In [43]:
def degrade_poisson(clean, peak=40.0, sigma=0.04, factor=2, seed=0):
    rng = np.random.RandomState(seed)
    x = rng.poisson(np.clip(clean.astype(np.float64), 0, None) * peak) / peak + rng.normal(0, sigma, size=clean.shape)
    return downsample(x, (clean.shape[0] // factor, clean.shape[1] // factor)).astype(np.float32)

D2_ROOT = os.path.join(WORK, "data_d2" + ("_smoke" if SMOKE else ""))
def build_domain2(pairs, root, peak, sigma):
    gt_dir, ny_dir = os.path.join(root, "GT"), os.path.join(root, "NoisyLR"); stamp = os.path.join(root, "stamp.json")
    want = dict(peak=peak, sigma=sigma, n=len(pairs))
    if os.path.exists(stamp) and json.load(open(stamp)) == want: return gt_dir, ny_dir
    shutil.rmtree(root, ignore_errors=True); os.makedirs(gt_dir); os.makedirs(ny_dir)
    for gp, _ in pairs:
        nm = os.path.basename(gp); os.symlink(gp, os.path.join(gt_dir, nm))
        np.save(os.path.join(ny_dir, nm), degrade_poisson(load_npy(gp), peak, sigma, seed=zlib.crc32(nm.encode()) % (2 ** 31)))
    json.dump(want, open(stamp, "w")); return gt_dir, ny_dir

splits, man = C.get_splits()
_gt2, _ny2 = build_domain2(C.get_pairs(), D2_ROOT, D2_PEAK, D2_SIGMA)
splits_d2 = apply_manifest(match_pairs(_gt2, _ny2), man)                  # SAME file-name split as the main domain
RUNS_D2 = os.path.join(C.OUT, "runs_d2"); os.makedirs(RUNS_D2, exist_ok=True)
print("domain-2 split:", {k: len(v) for k, v in splits_d2.items()}, "| same manifest sha1:", man["sha1"][:12])

# how does the noise variance depend on the signal in each domain?  E[r^2 | I] = c0 + c1 I + c2 I^2
def var_vs_intensity(spl, n_img=120):
    rng_ = np.random.RandomState(0); sel = rng_.choice(len(spl["train"]), min(n_img, len(spl["train"])), replace=False); I_, V_ = [], []
    for i in sel:
        g, n = spl["train"][i]; gt, ny = load_npy(g), load_npy(n); ds = downsample(gt, ny.shape[:2]); I_.append(ds.ravel()[::3]); V_.append(((ny - ds) ** 2).ravel()[::3])
    I_, V_ = np.concatenate(I_), np.concatenate(V_); bins = np.linspace(0.02, 0.98, 25); wh = np.digitize(I_, bins)
    ks = [k for k in range(1, len(bins)) if (wh == k).sum() > 100]
    bx = np.array([I_[wh == k].mean() for k in ks]); by = np.array([V_[wh == k].mean() for k in ks])
    A = np.stack([np.ones_like(bx), bx, bx ** 2], 1); c = np.linalg.lstsq(A, by, rcond=None)[0]
    return bx, by, c
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4)); NOISE_FIT = {}
for a_, (nm, spl) in zip(ax, [("domain 1: speckle + Gaussian (KLA)", splits), (f"domain 2: Poisson-Gaussian (peak={D2_PEAK:g})", splits_d2)]):
    bx, by, c = var_vs_intensity(spl); NOISE_FIT[nm] = dict(c0=float(c[0]), c1_linear=float(c[1]), c2_quadratic=float(c[2]))
    a_.plot(bx, by, "o", ms=4); a_.plot(bx, c[0] + c[1] * bx + c[2] * bx ** 2, "r-", label=f"c1={c[1]:.4f}, c2={c[2]:.4f}"); a_.set_title(nm, fontsize=9)
    a_.set_xlabel("clean local intensity I"); a_.set_ylabel("Var(r | I)"); a_.legend()
fig.tight_layout(); savefig(fig, "fig_noise_domains", {"fit": NOISE_FIT, "d2_peak": D2_PEAK, "d2_sigma": D2_SIGMA}); plt.show()
json.dump(NOISE_FIT, open(os.path.join(C.TABLES, "noise_domains_fit.json"), "w"), indent=1); print(NOISE_FIT)

domain-2 split: {'train': 2560, 'val': 320, 'test': 320} | same manifest sha1: 31d7258b8c36
{'domain 1: speckle + Gaussian (KLA)': {'c0': -0.0005328879924491048, 'c1_linear': 0.008570630103349686, 'c2_quadratic': 0.018007084727287292}, 'domain 2: Poisson-Gaussian (peak=40)': {'c0': 0.000396051473217085, 'c1_linear': 0.006247401237487793, 'c2_quadratic': 1.545510531286709e-05}}


## Part 8 - Training driver (one group per session)
`train_arms(names, run_root, spl)` trains every arm in `names` for every seed, skips finished runs, saves `results_latest.zip` after each run and stops cleanly at `SESSION_HOURS`. Domain 1 (KLA) goes to `runs/`, domain 2 (Poisson-Gaussian) to `runs_d2/`. Set `TRAIN_NOW` / `TRAIN_NOW_D2` in Part 1 (session plan there).

In [44]:
dev = C.device()
SESSION_END = time.time() + SESSION_HOURS * 3600
def train_arms(names, run_root, spl, tag=""):
    _t_all = time.time(); _n_total = len(names) * len(SEEDS); _n_done = 0
    for name in names:
        if name not in CONFIGS: print("unknown arm", name); continue
        print("=" * 90, f"\nTRAINING {name} {tag}  seeds={SEEDS} epochs={EPOCHS} device={dev}\n", "=" * 90, sep="")
        for seed in SEEDS:
            try:
                r = run_one(name, seed, spl, run_root=run_root, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=C.LR, wd=C.WEIGHT_DECAY, clip=C.GRAD_CLIP,
                            patience=PATIENCE, device=dev, cache=C.CACHE, verbose=True, deadline=SESSION_END)
            except TimeoutError as e:
                save_now("(time budget)"); print(f"\nSTOPPED CLEANLY: {e}. Download results_latest.zip, start a new session with it as input, Run All: it resumes here."); return False
            except KeyboardInterrupt:
                save_now("(interrupted)"); print("interrupted - progress saved"); return False
            print(f"==> {name}{tag} seed{seed} | best_epoch={r['best_epoch']} val_psnr={r['val']['psnr']:.3f} test_psnr={r['test']['psnr']:.3f} | {r['train_minutes']:.1f} min", flush=True)
            save_now(f"(after {name}{tag} seed{seed})")
            _n_done += 1; _el = (time.time() - _t_all) / 60
            print(f"   PROGRESS {_n_done}/{_n_total} runs done | {_el:.0f} min elapsed | about {_el / _n_done * (_n_total - _n_done):.0f} min left for this list", flush=True)
    return True

if dev == "cpu" and not SMOKE and (TRAIN_NOW or TRAIN_NOW_D2):
    print("WARNING: CPU only - switch the accelerator to a GPU before training (a run takes ~26 min on a T4).")
ok1 = train_arms(TRAIN_NOW, C.RUNS, splits, "") if TRAIN_NOW else True
ok2 = train_arms(TRAIN_NOW_D2, RUNS_D2, splits_d2, " [domain2]") if (TRAIN_NOW_D2 and ok1) else True
print("\nfinished runs (domain 1):", {n: finished_seeds(C.RUNS, n) for n in sorted(os.listdir(C.RUNS))})
print("finished runs (domain 2):", {n: finished_seeds(RUNS_D2, n) for n in sorted(os.listdir(RUNS_D2))})
save_now("(end of training cell)")


finished runs (domain 1): {'NAFNet-small': [0, 1, 2, 3], 'Restormer-FiLM-control': [0, 1, 2, 3], 'Restormer-LDMH-full': [0, 1, 2, 3], 'Restormer-small': [0, 1, 2, 3], 'SwinIR-small': [0, 1, 2, 3], 'baseline': [0, 1, 2, 3, 4, 5, 6, 7], 'film_control': [0, 1, 2, 3, 4, 5, 6, 7], 'full': [0, 1, 2, 3, 4, 5, 6, 7], 'full_clamp': [0, 1, 2, 3], 'full_no_bal': [0, 1, 2, 3], 'full_no_div': [0, 1, 2, 3], 'global_gc': [0, 1], 'scale_only': [4, 5, 6, 7]}
finished runs (domain 2): {'baseline': []}
saved /kaggle/working/results_latest.zip (77.0 MB) (end of training cell)


## Part 8b - Long schedule (session C)
Trains `LONG_ARMS` for `LONG_EPOCHS` epochs in a separate folder. Tests whether the 30-epoch protocol under-states the effect (a reviewer will ask).

In [45]:
# ======================= PART 8b - LONG SCHEDULE (SESSION = "C") =======================
# Same code, same split, same seeds-as-units; only the number of epochs changes (cosine schedule over LONG_EPOCHS, no early stopping).
# Results go to a SEPARATE folder (runs_long) so the 30-epoch runs are never mixed up with them. Finished runs are skipped; an interrupted
# run resumes from its last epoch; a clean stop happens at SESSION_HOURS (download results_latest.zip, start a new session, Run All).
RUNS_LONG = os.path.join(C.OUT, "runs_long"); os.makedirs(RUNS_LONG, exist_ok=True)

def train_long(names, seeds, epochs):
    n_total, n_done, t0 = len(names) * len(seeds), 0, time.time()
    for name in names:
        for seed in seeds:
            if seed in finished_seeds(RUNS_LONG, name):
                n_done += 1; print(f"[skip] {name} seed{seed} (done)"); continue
            try:
                r = run_one(name, seed, splits, run_root=RUNS_LONG, epochs=epochs, batch_size=BATCH_SIZE, lr=C.LR, wd=C.WEIGHT_DECAY, clip=C.GRAD_CLIP,
                            patience=epochs, device=dev, cache=C.CACHE, verbose=VERBOSE, deadline=SESSION_END)
            except TimeoutError as e:
                save_now("(long: time budget)"); print(f"STOPPED CLEANLY: {e}. Download results_latest.zip and Run All in a new session: it resumes here."); return False
            n_done += 1
            print(f"==> LONG {name} seed{seed} | epochs={epochs} best_epoch={r['best_epoch']} val_psnr={r['val']['psnr']:.3f} test_psnr={r['test']['psnr']:.3f} | {r['train_minutes']:.0f} min", flush=True)
            save_now(f"(after long {name} seed{seed})")
            print(f"   PROGRESS {n_done}/{n_total} | {(time.time()-t0)/60:.0f} min elapsed", flush=True)
    return True

if SESSION == "C" or SMOKE:
    _ep = 1 if SMOKE else LONG_EPOCHS
    if dev == "cpu" and not SMOKE: print("WARNING: CPU only - switch the accelerator to a GPU first.")
    train_long(LONG_ARMS if not SMOKE else LONG_ARMS[:1], LONG_SEEDS if not SMOKE else LONG_SEEDS[:1], _ep)
else:
    print("long-schedule training runs only when SESSION = 'C' (current:", SESSION, ")")


   [film_control|s0] epoch 1: 10/320 batches, 8s -> projected 4.1 min/epoch
   [film_control|s0] epoch 1: 50/320 batches, 14s -> projected 1.5 min/epoch
   [film_control|s0] epoch 1: 100/320 batches, 21s -> projected 1.1 min/epoch
   [film_control|s0] epoch 1: 200/320 batches, 36s -> projected 1.0 min/epoch
   [film_control|s0] epoch 1: 300/320 batches, 51s -> projected 0.9 min/epoch
[film_control|s0] ep 1/100 train_loss=0.04111 val_psnr=26.686 val_ssim=0.7045 | 1.0 min/epoch, ~96 min left for this run
[film_control|s0] ep 2/100 train_loss=0.03684 val_psnr=26.987 val_ssim=0.7200 | 0.9 min/epoch, ~86 min left for this run
[film_control|s0] ep 3/100 train_loss=0.03604 val_psnr=27.057 val_ssim=0.7217 | 0.9 min/epoch, ~85 min left for this run
[film_control|s0] ep 4/100 train_loss=0.03563 val_psnr=27.151 val_ssim=0.7265 | 0.9 min/epoch, ~84 min left for this run
[film_control|s0] ep 5/100 train_loss=0.03532 val_psnr=27.226 val_ssim=0.7306 | 0.9 min/epoch, ~83 min left for this run
[film_co

## Part 9A - Collapse ablation: `full_clamp`, `full_no_bal`, `full_no_div` (+ stress arms)
A run is called **collapsed** when, at the end of training, two betas are tied (`min_beta_dist < 0.05`) or a component is dead (`min usage < 0.02`). The table reports, per arm, how many seeds collapsed, the final betas, and the effect on PSNR.

In [91]:
def final_state(run_root, name):
    rows = []
    for sd in finished_seeds(run_root, name):
        h = pd.read_csv(os.path.join(run_root, name, f"seed{sd}", "history.csv"))
        if "beta0" not in h.columns: continue
        l = h.iloc[-1]; us = [float(l[f"usage{k}"]) for k in range(3)]; bs = [float(l[f"beta{k}"]) for k in range(3)]
        rows.append(dict(arm=name, seed=sd, betas=[round(b, 3) for b in bs], usage=[round(u, 3) for u in us], min_beta_dist=float(l.min_beta_dist), min_usage=min(us),
                         at_bound=int(l.any_beta_at_bound), collapsed=bool(l.min_beta_dist < 0.05 or min(us) < 0.02)))
    return rows

COLL = pd.DataFrame([r for a in ARMS_A for r in final_state(C.RUNS, a)])
if len(COLL):
    print(COLL.to_string(index=False))
    COLL_SUM = COLL.groupby("arm").agg(n_seeds=("seed", "size"), n_collapsed=("collapsed", "sum"), n_at_bound=("at_bound", "sum"),
                                       min_beta_dist_mean=("min_beta_dist", "mean"), min_usage_mean=("min_usage", "mean")).reindex([a for a in ARMS_A if a in set(COLL.arm)])
    print("\n", COLL_SUM.round(3).to_string()); COLL.to_csv(os.path.join(C.TABLES, "collapse_per_seed.csv"), index=False); COLL_SUM.to_csv(os.path.join(C.TABLES, "collapse_summary.csv"))
else:
    COLL_SUM = pd.DataFrame(); print("no finished LDMH runs yet (train the arms of Part A first)")

# trajectories of the betas and the usage, seed 0 (and the other seeds faintly)
arms_here = [a for a in ARMS_A if finished_seeds(C.RUNS, a)]
if arms_here:
    fig, ax = plt.subplots(2, len(arms_here), figsize=(3.2 * len(arms_here), 5.2), squeeze=False)
    for j, a in enumerate(arms_here):
        for sd in finished_seeds(C.RUNS, a):
            h = pd.read_csv(os.path.join(C.RUNS, a, f"seed{sd}", "history.csv"))
            if "beta0" not in h.columns: continue
            for k in range(3):
                ax[0, j].plot(h.epoch, h[f"beta{k}"], color=f"C{k}", alpha=1.0 if sd == finished_seeds(C.RUNS, a)[0] else 0.25, lw=1.2)
                ax[1, j].plot(h.epoch, h[f"usage{k}"], color=f"C{k}", alpha=1.0 if sd == finished_seeds(C.RUNS, a)[0] else 0.25, lw=1.2)
        ax[0, j].set_title(a, fontsize=9); ax[0, j].set_ylim(0.2, 2.6); ax[1, j].set_ylim(0, 1); ax[1, j].set_xlabel("epoch")
    ax[0, 0].set_ylabel("component beta"); ax[1, 0].set_ylabel("mean usage"); fig.tight_layout()
    savefig(fig, "fig_collapse_trajectories", {"arms": arms_here}); plt.show()

print("\nEffect on test PSNR (seed-level Welch, image-level paired):")
EFF_A = effect_table([("full_clamp", "full"), ("full_no_bal", "full"), ("full_no_div", "full"), ("full_clamp_stress", "full_stress"), ("full", "film_control"), ("full", "baseline")], C.RUNS)

        arm  seed                 betas                 usage  min_beta_dist  min_usage  at_bound  collapsed
       full     0  [0.504, 1.554, 2.38] [0.194, 0.443, 0.363]       0.826295   0.194100         0      False
       full     1 [0.503, 1.548, 2.385] [0.189, 0.444, 0.367]       0.836758   0.188848         0      False
       full     2 [0.504, 1.558, 2.377] [0.193, 0.443, 0.364]       0.818649   0.192945         0      False
       full     3 [0.504, 1.553, 2.386] [0.192, 0.446, 0.362]       0.833232   0.192020         0      False
       full     4 [0.503, 1.551, 2.384] [0.189, 0.447, 0.365]       0.832841   0.188665         0      False
       full     5  [0.503, 1.55, 2.386]  [0.192, 0.447, 0.36]       0.836540   0.192424         0      False
       full     6 [0.503, 1.557, 2.381]  [0.19, 0.445, 0.365]       0.824512   0.190111         0      False
       full     7 [0.505, 1.545, 2.381] [0.191, 0.444, 0.364]       0.835753   0.191362         0      False
 full_clamp     0  

## Part 9B - Scale-only control: does the *mixture of shapes* add anything beyond a *per-pixel scale*?
`scale_only` has the same encoder, FiLM and backbone as `full`, but predicts only one per-pixel scale (heteroscedastic GGD, shape fixed at the best global value beta = 1.53). If `full` is not better than `scale_only`, the module's gain is the signal-dependent noise scale and **not** a learned distribution shape.

In [92]:
print("Arms (mean +- std over seeds, test PSNR / SSIM):")
RES1 = load_results(C.RUNS)
tab = RES1[RES1.config.isin(ARMS_B + ARMS_E + ["full_outnll"])].groupby("config").agg(n=("seed", "size"), params=("params", "first"), psnr=("test_psnr", "mean"), psnr_sd=("test_psnr", "std"),
        ssim=("test_ssim", "mean"), minutes=("minutes", "mean")).round(4)
print(tab.to_string()); tab.to_csv(os.path.join(C.TABLES, "arms_summary_domain1.csv"))
print("\nPairwise effects (arm a - arm b):")
EFF_B = effect_table([("full", "scale_only"), ("full", "scale_only_g2"), ("scale_only", "scale_control"), ("scale_only", "film_control"), ("scale_only", "baseline"),
                      ("scale_control", "baseline"), ("full", "film_control"), ("full_outnll", "full")], C.RUNS)

Arms (mean +- std over seeds, test PSNR / SSIM):
                        n  params     psnr  psnr_sd    ssim  minutes
config                                                              
Restormer-FiLM-control  4  141110  27.9589   0.0298  0.7404  26.8224
Restormer-LDMH-full     4  141110  27.9515   0.0163  0.7405  26.8836
Restormer-small         4  111917  27.9226   0.0143  0.7399  25.3412
baseline                8   86945  27.6620   0.0422  0.7293  24.6320
film_control            8  116138  27.6865   0.0359  0.7295  26.4065
full                    8  116138  27.7731   0.0203  0.7334  26.5356
scale_only              4  115490  27.6736   0.0290  0.7310  26.5045

Pairwise effects (arm a - arm b):
                    label  n_a  n_b    diff   ci_lo  ci_hi  p_welch  img_diff  img_lo  img_hi
        full - scale_only    8    4  0.0995  0.0567 0.1424   0.0023    0.0995  0.0862  0.1142
scale_only - film_control    4    8 -0.0129 -0.0579 0.0320   0.5225   -0.0129 -0.0285  0.0021
    scale_onl

## Part 9D - Second noise domain (Poisson-Gaussian)

In [93]:
try:
    RES2 = load_results(RUNS_D2)
except (KeyError, ValueError):      # no domain-2 runs on disk yet
    RES2 = pd.DataFrame()

if len(RES2) and "config" in RES2.columns:
    tab2 = RES2.groupby("config").agg(
        n=("seed", "size"),
        psnr=("test_psnr", "mean"),
        psnr_sd=("test_psnr", "std"),
        ssim=("test_ssim", "mean"),
    ).round(4)
    print(tab2.to_string())
    tab2.to_csv(os.path.join(C.TABLES, "arms_summary_domain2.csv"))
    print("\nEffects in domain 2:")
    EFF_D = effect_table(
        [("full", "baseline"), ("full", "film_control"), ("full", "scale_only"),
         ("scale_only", "film_control"), ("film_control", "baseline")],
        RUNS_D2,
    )
else:
    RES2 = pd.DataFrame()
    EFF_D = pd.DataFrame()
    print("no domain-2 runs yet (set TRAIN_NOW_D2 in Part 1)")

no domain-2 runs yet (set TRAIN_NOW_D2 in Part 1)


## Part 9E - Strong backbone: can the module help Restormer?

In [94]:
print("Restormer-based arms (domain 1):")
EFF_E = effect_table([("Restormer-FiLM-control", "Restormer-small"), ("Restormer-LDMH-full", "Restormer-FiLM-control"), ("Restormer-LDMH-outnll", "Restormer-FiLM-control"),
                      ("Restormer-LDMH-deep", "Restormer-FiLM-control"), ("Restormer-LDMH-full", "Restormer-small"), ("Restormer-small", "full")], C.RUNS)

Restormer-based arms (domain 1):
                                       label  n_a  n_b    diff   ci_lo  ci_hi  p_welch  img_diff  img_lo  img_hi
    Restormer-FiLM-control - Restormer-small    4    4  0.0363 -0.0083 0.0808   0.0880    0.0363  0.0305  0.0425
Restormer-LDMH-full - Restormer-FiLM-control    4    4 -0.0074 -0.0520 0.0372   0.6824   -0.0074 -0.0124 -0.0028
       Restormer-LDMH-full - Restormer-small    4    4  0.0289  0.0023 0.0555   0.0378    0.0289  0.0241  0.0339
                      Restormer-small - full    4    8  0.1495  0.1263 0.1726   0.0000    0.1495  0.1329  0.1669


### Forest plot of all effects

In [95]:
# forest plot of all stored effects (seed-level 95% CI)
E = pd.DataFrame(list(ALL_EFFECTS.values()))
if len(E):
    E = E[E.n_a >= 2].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(7.5, 0.32 * len(E) + 1.5)); y = np.arange(len(E))[::-1]
    ax.errorbar(E["diff"], y, xerr=[E["diff"] - E["ci_lo"], E["ci_hi"] - E["diff"]], fmt="o", color="C0", capsize=2)
    ax.axvline(0, color="k", lw=0.8); ax.set_yticks(y); ax.set_yticklabels([f"{r.label} [{r.domain}]" for r in E.itertuples()], fontsize=7)
    ax.set_xlabel("difference in test PSNR (dB), seed-level 95% CI"); fig.tight_layout(); savefig(fig, "fig_forest_effects", {"n_effects": int(len(E))}); plt.show()
    E.to_csv(os.path.join(C.TABLES, "all_effects.csv"), index=False)

## Part 9F - Q1 confirmatory analysis
Primary family (3 contrasts, Holm), secondary family, descriptive rows; minimum detectable effect; equivalence (TOST) for the "no effect" claims; paste-ready sentences. **This is the table that goes into the paper.**

In [96]:
# ======================= PART 9F - Q1 CONFIRMATORY ANALYSIS (run after the training sessions) =======================
# What a Q1 reviewer asks for, computed from YOUR runs (nothing typed in):
#   1. A SMALL pre-declared PRIMARY family (3 contrasts, declared here before sessions A/B/D are run), Holm-corrected; SECONDARY
#      contrasts are Holm-corrected within their own tier; everything else is descriptive. (Not 20 loose p-values.)
#   2. The unit of replication is the SEED (Welch t, 95% CI); n = number of finished seeds per arm (grows when you run session B / D).
#   3. Minimum detectable effect (80% power) for every contrast: tells the reader how small an effect n seeds could have found.
#   4. EQUIVALENCE (TOST, margin +-0.05 dB) for the 'no effect' claims (clamp / no-balance / no-diversity): "not significant" is not
#      evidence of no effect; equivalence is.
#   5. Standardised effect (Hedges g) and a ready-to-paste sentence for each contrast.
import itertools
from scipy import stats as _st

EQ_MARGIN_DB = 0.05          # equivalence margin (dB) - state it in the paper

def _arr(root, arm, metric="test_psnr"):
    if not os.path.isdir(os.path.join(root, arm)): return np.array([])
    r = load_results(root, [arm])
    return r.sort_values("seed")[metric].values.astype(float) if len(r) else np.array([])

def _welch_full(x, y, conf=0.95):
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2: return None
    vx, vy = x.var(ddof=1) / nx, y.var(ddof=1) / ny; se = np.sqrt(vx + vy); d = x.mean() - y.mean()
    dfw = se ** 4 / (vx ** 2 / (nx - 1) + vy ** 2 / (ny - 1))
    h = _st.t.ppf(0.5 + conf / 2, dfw) * se
    p = 2 * _st.t.sf(abs(d / se), dfw)
    mde = (_st.t.ppf(0.975, dfw) + _st.t.ppf(0.80, dfw)) * se           # 80% power, two-sided alpha = .05
    sp = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2))
    J = 1 - 3 / (4 * (nx + ny) - 9); g = d / sp * J if sp > 0 else np.nan
    # TOST: both one-sided tests against +-EQ_MARGIN_DB  ->  equivalent if the 90% CI lies inside the margin
    h90 = _st.t.ppf(0.95, dfw) * se
    return dict(n_a=nx, n_b=ny, diff=d, ci_lo=d - h, ci_hi=d + h, p=p, mde80=mde, hedges_g=g,
                eq_lo=d - h90, eq_hi=d + h90, equivalent=bool(d - h90 > -EQ_MARGIN_DB and d + h90 < EQ_MARGIN_DB))

def _holm(ps):
    ps = np.asarray(ps, float); m = len(ps); order = np.argsort(ps); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * ps[i]); adj[i] = min(1.0, run)
    return adj

_RUNS_LONG = os.path.join(C.OUT, "runs_long")
# (label, arm_a, arm_b, run_root, tier)   tier: primary | secondary | descriptive
FAMILY = [
    ("KLA  full - FiLM control (capacity-matched)",   "full", "film_control", C.RUNS, "primary"),
    ("KLA  full - baseline",                           "full", "baseline",     C.RUNS, "secondary"),
    ("KLA  full - scale-only (shape mixture vs scale)", "full", "scale_only",   C.RUNS, "primary"),
    ("D2   full - FiLM control (Poisson-Gaussian)",    "full", "film_control", RUNS_D2, "secondary"),
    ("D2   full - scale-only (Poisson-Gaussian)",      "full", "scale_only",   RUNS_D2, "secondary"),
    ("Restormer  LDMH - FiLM control (transfer)",      "Restormer-LDMH-full", "Restormer-FiLM-control", C.RUNS, "primary"),
    ("Restormer  LDMH - Restormer-small",              "Restormer-LDMH-full", "Restormer-small",        C.RUNS, "secondary"),
    ("LONG schedule  full - FiLM control",             "full", "film_control", _RUNS_LONG, "secondary"),
    ("LONG schedule  Restormer LDMH - Restormer-small", "Restormer-LDMH-full", "Restormer-small", _RUNS_LONG, "secondary"),
    # descriptive (no hypothesis test is claimed)
    ("KLA  Restormer-small - full (external backbone)", "Restormer-small", "full", C.RUNS, "descriptive"),
    ("KLA  SwinIR-small - full",                       "SwinIR-small", "full", C.RUNS, "descriptive"),
    ("KLA  NAFNet-small - full",                       "NAFNet-small", "full", C.RUNS, "descriptive"),
    ("KLA  hard clamp - full (equivalence)",           "full_clamp", "full", C.RUNS, "descriptive"),
    ("KLA  no balance penalty - full (equivalence)",   "full_no_bal", "full", C.RUNS, "descriptive"),
    ("KLA  no diversity penalty - full (equivalence)", "full_no_div", "full", C.RUNS, "descriptive"),
]
rows = []
for lab, a, b, root, tier in FAMILY:
    x, y = _arr(root, a), _arr(root, b)
    r = _welch_full(x, y) if len(x) and len(y) else None
    if r is None: continue
    rows.append(dict(contrast=lab, tier=tier, **r))
Q1 = pd.DataFrame(rows)
if len(Q1):
    Q1["p_holm"] = np.nan
    for _t in ("primary", "secondary"):
        m = (Q1.tier == _t).values
        if m.sum(): Q1.loc[m, "p_holm"] = _holm(Q1.loc[m, "p"].values)
    Q1["sig_holm"] = (Q1.p_holm < 0.05) & (Q1.ci_lo > 0)
    show = Q1[["contrast", "tier", "n_a", "n_b", "diff", "ci_lo", "ci_hi", "p", "p_holm", "mde80", "hedges_g", "equivalent"]].round(4)
    print("PRIMARY / SECONDARY / DESCRIPTIVE contrasts (Holm within tier), seed-level Welch, test PSNR (dB):\n")
    print(show.to_string(index=False))
    Q1.to_csv(os.path.join(C.TABLES, "table_q1_confirmatory.csv"), index=False)

    print("\n--- paste-ready sentences (edit the wording, keep the numbers) ---")
    for _, r in Q1.iterrows():
        if r.tier != "descriptive":
            verdict = ("reliable after Holm correction" if r.sig_holm else "not reliable after Holm correction")
            print(f"* [{r.tier}] {r.contrast}: {r['diff']:+.3f} dB (95% CI {r.ci_lo:+.3f} to {r.ci_hi:+.3f}; n = {int(r.n_a)} vs {int(r.n_b)} seeds; Welch p = {r.p:.3g}, "
                  f"Holm p = {r.p_holm:.3g}; Hedges g = {r.hedges_g:.2f}) - {verdict}; with this n an effect below {r.mde80:.3f} dB could not be detected at 80% power.")
        elif "equivalence" in r.contrast:
            print(f"* {r.contrast}: {r['diff']:+.4f} dB, 90% CI {r.eq_lo:+.3f} to {r.eq_hi:+.3f} -> "
                  f"{'EQUIVALENT within +-%.2f dB (TOST)' % EQ_MARGIN_DB if r.equivalent else 'equivalence NOT shown at +-%.2f dB' % EQ_MARGIN_DB}.")

    # forest plot (confirmatory family + descriptive rows), seed-level 95% CI, equivalence band shaded
    fig, ax = plt.subplots(figsize=(6.4, 0.34 * len(Q1) + 1.2))
    ypos = np.arange(len(Q1))[::-1]
    ax.axvspan(-EQ_MARGIN_DB, EQ_MARGIN_DB, color="#dddddd", alpha=.6, lw=0, label=f"equivalence band +-{EQ_MARGIN_DB} dB")
    for y_, (_, r) in zip(ypos, Q1.iterrows()):
        col = "#1f77b4" if (r.tier != "descriptive" and r.sig_holm) else ("#888888" if r.tier != "descriptive" else "#c77f1a")
        ax.plot([r.ci_lo, r.ci_hi], [y_, y_], color=col, lw=1.6); ax.plot(r["diff"], y_, "o", color=col, ms=4)
    ax.axvline(0, color="k", lw=.7); ax.set_yticks(ypos); ax.set_yticklabels(Q1.contrast, fontsize=7)
    ax.set_xlabel("difference in test PSNR (dB), seed-level 95% CI   (blue: reliable after Holm, grey: not, orange: descriptive)", fontsize=7)
    ax.legend(fontsize=7, loc="lower right", frameon=False); plt.tight_layout()
    savefig(fig, "fig_q1_confirmatory", {"family": [f[0] for f in FAMILY], "margin_db": EQ_MARGIN_DB}); plt.show()
else:
    print("no finished runs found for the confirmatory family yet")

# --- long-schedule summary (only if session C was run) --------------------------------------------------------------
if os.path.isdir(_RUNS_LONG) and len(os.listdir(_RUNS_LONG)):
    RL = load_results(_RUNS_LONG)
    print("\nLONG schedule (%d epochs): test PSNR mean +- sd over seeds" % LONG_EPOCHS)
    print(RL.groupby("config").agg(n=("seed", "size"), psnr=("test_psnr", "mean"), psnr_sd=("test_psnr", "std"), ssim=("test_ssim", "mean"),
                                    best_epoch=("best_epoch", "mean")).round(4).to_string())


PRIMARY / SECONDARY / DESCRIPTIVE contrasts (Holm within tier), seed-level Welch, test PSNR (dB):

                                       contrast        tier  n_a  n_b    diff   ci_lo   ci_hi      p  p_holm  mde80  hedges_g  equivalent
    KLA  full - FiLM control (capacity-matched)     primary    8    8  0.0866  0.0546  0.1187 0.0001  0.0003 0.0448    2.8106       False
                           KLA  full - baseline   secondary    8    8  0.1112  0.0744  0.1480 0.0001  0.0001 0.0514    3.1766       False
KLA  full - scale-only (shape mixture vs scale)     primary    8    4  0.0995  0.0567  0.1424 0.0023  0.0046 0.0579    3.9503       False
      Restormer  LDMH - FiLM control (transfer)     primary    4    4 -0.0074 -0.0520  0.0372 0.6824  0.6824 0.0603   -0.2680        True
              Restormer  LDMH - Restormer-small   secondary    4    4  0.0289  0.0023  0.0555 0.0378  0.0378 0.0364    1.6399        True
KLA  Restormer-small - full (external backbone) descriptive    4    8  0.

## Part 10 - Downstream defect-detection benchmark (real inspection images, hundreds of defects, stated detector)
**Protocol (everything is saved in `downstream_manifest.json`).**

1. *Images*: real clean inspection / SEM images from `DOWNSTREAM_DIR` (not used in training), centre-cropped, resized to `DS_SIZE`, robustly intensity-normalised.
2. *Defects*: `DS_DEFECTS_PER_IMAGE` synthetic defects per image with **known masks** (`particle`, `bridge`, `void`; random contrast 0.15-0.35, adaptive sign), inserted into a "test die". The "reference die" is the same image without defects.
3. *Degradation*: both dies are degraded **independently** with the *fitted* degradation (looks, sigma of Part 4) and restored by each arm.
4. *Detector (die-to-die)*: `d = restored_test - restored_reference` -> Gaussian smoothing (`smooth_sigma`) -> `z = |d| / (1.4826 MAD(d))` -> threshold `k` -> connected components with area >= `min_area` px. A defect is **detected** if a component overlaps its mask (dilated by 2 px); every other component is a **false alarm**.
5. *Metric*: FROC - recall at a fixed number of false alarms per image (`FAR_TARGETS`), mean recall over FAR in [0.05, 2], with a **paired bootstrap over images** (shared resamples across arms).

If `DOWNSTREAM_DIR` is empty the notebook falls back to the *ground-truth images of the KLA test split* and says so: **that is not a real-SEM test** and the ledger will refuse to call it one.

In [97]:
from PIL import Image
EXT = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".npy")
SKIP = ("mask", "label", "annot", "segment", "_gt", "groundtruth", "defect")
def list_images(folder):
    out = []
    for dp, dn, fn in os.walk(folder):
        if "__MACOSX" in dp: continue
        for f in fn:
            if f.lower().endswith(EXT) and not f.startswith("._") and not any(s in f.lower() for s in SKIP): out.append(os.path.join(dp, f))
    return sorted(out)

def load_gray(path):
    if path.lower().endswith(".npy"): a = np.load(path).astype(np.float32)
    else:
        im = Image.open(path); a = np.asarray(im).astype(np.float32)
        if a.ndim == 3: a = a[..., :3].mean(-1)
        a = a / (65535.0 if a.max() > 255 else 255.0)
    while a.ndim > 2: a = a[..., 0]
    return a

def prep(a, size):
    h, w = a.shape; s = min(h, w); y0, x0 = (h - s) // 2, (w - s) // 2; a = a[y0:y0 + s, x0:x0 + s]
    a = np.asarray(Image.fromarray(a).resize((size, size), Image.LANCZOS), np.float32)        # antialiased
    lo, hi = np.percentile(a, [0.5, 99.5]); return np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)

DS_REAL = bool(DOWNSTREAM_DIR) and os.path.isdir(DOWNSTREAM_DIR) and len(list_images(DOWNSTREAM_DIR)) > 0
if DS_REAL: ds_files = list_images(DOWNSTREAM_DIR)
else:
    ds_files = [g for g, _ in splits["test"]]
    print("WARNING: DOWNSTREAM_DIR not set / empty -> using the KLA TEST-split ground-truth images. This is NOT a real SEM test set.")
rs = np.random.RandomState(DS_SEED); order = rs.permutation(len(ds_files))[:DS_N_IMAGES]; ds_files = [ds_files[i] for i in sorted(order)]
DS_CLEAN = [prep(load_gray(f), DS_SIZE) for f in ds_files]
print(f"downstream images: {len(DS_CLEAN)} ({'REAL inspection images from DOWNSTREAM_DIR' if DS_REAL else 'FALLBACK, not real SEM'}) | size {DS_SIZE}x{DS_SIZE}")

downstream images: 150 (FALLBACK, not real SEM) | size 256x256


In [98]:
def _soft(mask, s=0.8): return np.clip(ndimage.gaussian_filter(mask.astype(np.float32), s), 0, 1)

def make_defect(kind, rng, S):
    sc = S / 256.0; m = np.zeros((S, S), np.float32); pad = int(14 * sc) + 2
    cy, cx = rng.randint(pad, S - pad, 2); yy, xx = np.mgrid[:S, :S]
    if kind == "particle":
        ry, rx = rng.uniform(3, 9, 2) * sc; th = rng.uniform(0, np.pi); u = (xx - cx) * np.cos(th) + (yy - cy) * np.sin(th); v = -(xx - cx) * np.sin(th) + (yy - cy) * np.cos(th)
        m = (u ** 2 / max(rx, 1) ** 2 + v ** 2 / max(ry, 1) ** 2 <= 1).astype(np.float32); sign = rng.choice([-1, 1])
    elif kind == "void":
        r = rng.uniform(3, 8) * sc; m = ((xx - cx) ** 2 + (yy - cy) ** 2 <= max(r, 1) ** 2).astype(np.float32); sign = -1
    else:                                                       # bridge: a short bright line
        L = rng.uniform(12, 30) * sc; w = rng.uniform(1.5, 3.5) * sc; th = rng.uniform(0, np.pi)
        u = (xx - cx) * np.cos(th) + (yy - cy) * np.sin(th); v = -(xx - cx) * np.sin(th) + (yy - cy) * np.cos(th)
        m = ((np.abs(u) <= L / 2) & (np.abs(v) <= max(w / 2, 0.7))).astype(np.float32); sign = 1
    return m, sign, (int(cy), int(cx))

def insert_defects(clean, rng, n_range, S):
    img, masks, specs, centers = clean.copy(), [], [], []
    n = rng.randint(n_range[0], n_range[1] + 1)
    for _ in range(n):
        for _try in range(50):
            kind = rng.choice(["particle", "bridge", "void"]); m, sign, c = make_defect(kind, rng, S)
            if m.sum() < 3 or any((c[0] - o[0]) ** 2 + (c[1] - o[1]) ** 2 < (24 * S / 256) ** 2 for o in centers): continue
            local = clean[m > 0].mean(); sg = sign if kind == "particle" else sign
            if kind == "particle": sg = -1 if local > 0.7 else (1 if local < 0.3 else sign)          # adaptive sign: keep the contrast visible
            if kind == "void" and local < 0.25: sg = 1
            amp = rng.uniform(0.15, 0.35); img = np.clip(img + sg * amp * _soft(m), 0, 1)
            masks.append(m > 0.5); centers.append(c); specs.append(dict(kind=kind, center=c, sign=int(sg), amp=float(amp), area=int(m.sum()))); break
    return img.astype(np.float32), masks, specs

fit_ds = fit_degradation(splits["train"], n_images=min(400, len(splits["train"])))
LOOKS, SIGMA = fit_ds["looks"], fit_ds["sigma"]
print(f"degradation used for the downstream test (fitted on TRAIN pairs): looks={LOOKS:.2f} sigma={SIGMA:.4f} (R2 of signal dependence {fit_ds['r2_signal_dependence']:.2f})")
rng = np.random.RandomState(DS_SEED); DS = []
for i, cl in enumerate(DS_CLEAN):
    img_def, masks, specs = insert_defects(cl, rng, DS_DEFECTS_PER_IMAGE, DS_SIZE)
    DS.append(dict(clean=cl, defect=img_def, masks=masks, specs=specs,
                   noisy_def=degrade(img_def, LOOKS, SIGMA, seed=DS_SEED * 1000 + 2 * i), noisy_ref=degrade(cl, LOOKS, SIGMA, seed=DS_SEED * 1000 + 2 * i + 1)))
N_DEF = sum(len(d["masks"]) for d in DS); kinds = pd.Series([s["kind"] for d in DS for s in d["specs"]]).value_counts().to_dict()
print(f"{len(DS)} images, {N_DEF} defects inserted: {kinds}")

degradation used for the downstream test (fitted on TRAIN pairs): looks=9.26 sigma=0.0425 (R2 of signal dependence 1.00)
150 images, 447 defects inserted: {np.str_('bridge'): 158, np.str_('void'): 156, np.str_('particle'): 133}


In [99]:
def detect(d, k, cfg=DET):
    """components of the thresholded, smoothed, MAD-normalised difference map -> list of boolean masks"""
    s = ndimage.gaussian_filter(d, cfg["smooth_sigma"]); mad = 1.4826 * np.median(np.abs(s - np.median(s))) + 1e-9
    lab, n = ndimage.label(np.abs(s) / mad > k); comps = []
    for j in range(1, n + 1):
        m = lab == j
        if m.sum() >= cfg["min_area"]: comps.append(m)
    return comps

KS = np.geomspace(DET["k_min"], DET["k_max"], DET["n_k"])
def score_image(dmap, masks):
    """-> hits[n_k] (number of GT defects detected), fps[n_k] (false-alarm components)"""
    dil = [ndimage.binary_dilation(m, iterations=2) for m in masks]; hits, fps = np.zeros(len(KS)), np.zeros(len(KS))
    for ki, k in enumerate(KS):
        comps = detect(dmap, k); used = [False] * len(comps); h = 0
        for dm in dil:
            ov = [ci for ci, c in enumerate(comps) if (c & dm).any()]
            if ov: h += 1
            for ci in ov: used[ci] = True
        hits[ki] = h; fps[ki] = used.count(False)
    return hits, fps

def restore(model, noisy):
    with torch.no_grad():
        o = model(torch.from_numpy(noisy)[None, None].to(dev)); o = o[0] if isinstance(o, (tuple, list)) else o
    return o.clamp(0, 1)[0, 0].cpu().numpy()

def run_arm(restorer):
    H, Fp = [], []
    for d in DS:
        dm = restorer(d["noisy_def"]) - restorer(d["noisy_ref"]); h, f = score_image(dm, d["masks"]); H.append(h); Fp.append(f)
    return np.array(H).T, np.array(Fp).T                      # [n_k, n_images]

def bil(noisy): return F.interpolate(torch.from_numpy(noisy)[None, None], size=(DS_SIZE, DS_SIZE), mode="bilinear", align_corners=False)[0, 0].numpy()

DS_RES = {}                                                    # arm -> dict(hits[n_seed, n_k, n_img], fps[...])
t0 = time.time(); DS_RES["bilinear"] = {k: v[None] for k, v in zip(("hits", "fps"), run_arm(bil))}
for arm in DS_ARMS:
    seeds_ok = finished_seeds(C.RUNS, arm); H, Fp = [], []
    for sd in seeds_ok:
        ck = torch.load(os.path.join(C.RUNS, arm, f"seed{sd}", "best.pth"), map_location=dev)
        m = build_model(ck["cfg"]).to(dev); m.load_state_dict(ck["model"]); m.eval()
        h, f = run_arm(lambda x: restore(m, x)); H.append(h); Fp.append(f)
    if H: DS_RES[arm] = dict(hits=np.stack(H), fps=np.stack(Fp)); print(f"  {arm}: {len(H)} seeds scored")
    else: print(f"  {arm}: no finished runs - skipped")
print(f"downstream scoring took {(time.time()-t0)/60:.1f} min")
NGT = np.array([len(d["masks"]) for d in DS])

  baseline: 8 seeds scored
  film_control: 8 seeds scored
  full: 8 seeds scored
  scale_only: 4 seeds scored
  Restormer-small: 4 seeds scored
downstream scoring took 58.6 min


In [100]:
def froc_from(hits, fps, idx):
    """hits,fps: [n_k, n_img] (mean over seeds) -> recall[n_k], far[n_k] on the images in idx"""
    return hits[:, idx].sum(1) / max(NGT[idx].sum(), 1), fps[:, idx].mean(1)

FAR_GRID = np.linspace(0.05, 2.0, 40)
def recall_at(far, rec, targets):
    o = np.argsort(far); f, r = far[o], np.maximum.accumulate(rec[o]); return [float(np.interp(t, f, r, left=0.0)) for t in targets]
def arm_metrics(hits, fps, idx):
    rec, far = froc_from(hits.mean(0), fps.mean(0), idx); at = recall_at(far, rec, FAR_TARGETS)
    return at + [float(np.mean(recall_at(far, rec, FAR_GRID)))]
COLS = [f"recall@FAR{t:g}" for t in FAR_TARGETS] + ["meanRecall[0.05-2]"]

rng_b = np.random.RandomState(DS_SEED + 7); BOOT = [rng_b.randint(0, len(DS), len(DS)) for _ in range(DS_BOOT)]      # shared resamples
FULL = np.arange(len(DS)); DS_TAB, BOOT_M = [], {}
for arm, r in DS_RES.items():
    pt = arm_metrics(r["hits"], r["fps"], FULL); bm = np.array([arm_metrics(r["hits"], r["fps"], b) for b in BOOT]); BOOT_M[arm] = bm
    row = dict(arm=arm, n_seeds=r["hits"].shape[0])
    for j, c in enumerate(COLS): row[c] = pt[j]; row[c + "_lo"], row[c + "_hi"] = np.percentile(bm[:, j], [2.5, 97.5])
    DS_TAB.append(row)
DS_TAB = pd.DataFrame(DS_TAB); print(f"{N_DEF} defects in {len(DS)} images | detector: {DET}")
print(DS_TAB[["arm", "n_seeds"] + COLS].round(3).to_string(index=False)); DS_TAB.to_csv(os.path.join(C.TABLES, "downstream_froc_summary.csv"), index=False)

DS_DIFF = []
for a, b in [("full", "baseline"), ("full", "film_control"), ("full", "scale_only"), ("scale_only", "film_control"), ("Restormer-small", "full"), ("baseline", "bilinear")]:
    if a in BOOT_M and b in BOOT_M:
        dd = BOOT_M[a] - BOOT_M[b]; j = COLS.index("recall@FAR1")
        DS_DIFF.append(dict(contrast=f"{a} - {b}", metric="recall@FAR1", diff=float(dd[:, j].mean()), lo=float(np.percentile(dd[:, j], 2.5)), hi=float(np.percentile(dd[:, j], 97.5)),
                            ci_excludes_0=bool(np.percentile(dd[:, j], 2.5) > 0 or np.percentile(dd[:, j], 97.5) < 0)))
DS_DIFF = pd.DataFrame(DS_DIFF); print("\nPaired bootstrap differences (recall at 1 false alarm / image):\n", DS_DIFF.round(3).to_string(index=False) if len(DS_DIFF) else "none")
DS_DIFF.to_csv(os.path.join(C.TABLES, "downstream_paired_differences.csv"), index=False)

# per-seed spread (seed variance is NOT in the bootstrap CI) and strata by defect type
for arm, r in DS_RES.items():
    if r["hits"].shape[0] > 1:
        ps = [arm_metrics(r["hits"][s:s + 1], r["fps"][s:s + 1], FULL)[1] for s in range(r["hits"].shape[0])]; print(f"  {arm}: recall@FAR1 per seed = {np.round(ps, 3)}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for arm, r in DS_RES.items():
    rec, far = froc_from(r["hits"].mean(0), r["fps"].mean(0), FULL); o = np.argsort(far); ax[0].plot(far[o], rec[o], label=arm, lw=1.5)
ax[0].set_xlim(0, 3); ax[0].set_xlabel("false alarms per image"); ax[0].set_ylabel("recall"); ax[0].set_title(f"FROC ({N_DEF} defects, {len(DS)} images)"); ax[0].legend(fontsize=7)
j = COLS.index("recall@FAR1"); arms_ = list(DS_TAB.arm)
ax[1].bar(arms_, DS_TAB["recall@FAR1"], yerr=[DS_TAB["recall@FAR1"] - DS_TAB["recall@FAR1_lo"], DS_TAB["recall@FAR1_hi"] - DS_TAB["recall@FAR1"]], capsize=3, color="C0")
ax[1].set_ylabel("recall at 1 FA / image (95% bootstrap CI)"); ax[1].tick_params(axis="x", rotation=40)
fig.tight_layout(); savefig(fig, "fig_downstream_froc", {"n_defects": int(N_DEF), "real_images": DS_REAL, "detector": DET}); plt.show()

# one example: reference, test die, difference maps of two arms
ex = int(np.argmax([len(d["masks"]) for d in DS])); d = DS[ex]; pan = [("clean+defects", d["defect"]), ("bilinear diff", np.abs(bil(d["noisy_def"]) - bil(d["noisy_ref"])))]
for arm in [a for a in ("baseline", "full") if a in DS_RES]:
    sd = finished_seeds(C.RUNS, arm)[0]; ck = torch.load(os.path.join(C.RUNS, arm, f"seed{sd}", "best.pth"), map_location=dev); m = build_model(ck["cfg"]).to(dev); m.load_state_dict(ck["model"]); m.eval()
    pan.append((f"{arm} diff", np.abs(restore(m, d["noisy_def"]) - restore(m, d["noisy_ref"]))))
fig, ax = plt.subplots(1, len(pan), figsize=(3.2 * len(pan), 3.4))
for a_, (t, im) in zip(np.atleast_1d(ax), pan):
    a_.imshow(im, cmap="gray"); a_.set_title(t, fontsize=8); a_.axis("off")
    for mk in d["masks"]: a_.contour(mk, levels=[0.5], colors="r", linewidths=0.6)
fig.tight_layout(); savefig(fig, "fig_downstream_example", {"image_index": ex}); plt.show()

def sha1(p):
    h = hashlib.sha1(); h.update(open(p, "rb").read()); return h.hexdigest()
MAN = dict(dataset=DS_DATASET_INFO, real_inspection_images=DS_REAL, n_images=len(DS), n_defects=int(N_DEF), defect_types=kinds, image_size=DS_SIZE, degradation=dict(looks=LOOKS, sigma=SIGMA, fitted_on="train pairs"),
           detector=dict(DET, kind="die-to-die difference, MAD-normalised, smoothed, connected components", match="component overlaps defect mask dilated by 2 px"), far_targets=list(FAR_TARGETS), bootstrap=DS_BOOT, seed=DS_SEED,
           files=[dict(path=os.path.basename(f), sha1=sha1(f)) for f in ds_files], defects=[dict(image=os.path.basename(f), defects=d["specs"]) for f, d in zip(ds_files, DS)])
json.dump(MAN, open(os.path.join(C.TABLES, "downstream_manifest.json"), "w"), indent=1, default=str); print("manifest saved: downstream_manifest.json (file hashes + every defect spec + detector parameters)")

447 defects in 150 images | detector: {'smooth_sigma': 1.0, 'min_area': 6, 'k_min': 1.2, 'k_max': 12.0, 'n_k': 40}
            arm  n_seeds  recall@FAR0.5  recall@FAR1  recall@FAR2  meanRecall[0.05-2]
       bilinear        1          0.120        0.171        0.238               0.161
       baseline        8          0.447        0.535        0.625               0.510
   film_control        8          0.451        0.541        0.636               0.516
           full        8          0.459        0.534        0.629               0.516
     scale_only        4          0.459        0.539        0.635               0.522
Restormer-small        4          0.442        0.520        0.611               0.500

Paired bootstrap differences (recall at 1 false alarm / image):
                  contrast      metric   diff     lo    hi  ci_excludes_0
          full - baseline recall@FAR1  0.004 -0.014 0.020          False
      full - film_control recall@FAR1 -0.005 -0.025 0.013          Fals

## Part 11 - Efficiency: is the module worth its cost?

In [101]:
from torch.utils.flop_counter import FlopCounterMode
def flops_g(m, hw=(64, 64)):
    x = torch.rand(1, 1, *hw);
    with FlopCounterMode(display=False) as fc:
        with torch.no_grad(): m(x)
    return fc.get_total_flops() / 1e9
def latency(m, device, hw=(128, 128), n=10):
    m = m.to(device).eval(); x = torch.rand(1, 1, *hw, device=device)
    with torch.no_grad():
        for _ in range(3): m(x)
        if device == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(n): m(x)
        if device == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / n * 1000
rows = []
for a in ["baseline", "film_control", "full", "scale_only", "NAFNet-small", "Restormer-small", "SwinIR-small", "Restormer-FiLM-control", "Restormer-LDMH-full"]:
    m = build_model(dict(CONFIGS[a])).eval(); r = dict(arm=a, params=sum(p.numel() for p in m.parameters()), GFLOPs_64x64_in=round(flops_g(m), 3), cpu_ms_128=round(latency(m, "cpu"), 1))
    if torch.cuda.is_available(): r["gpu_ms_128"] = round(latency(m, "cuda"), 2)
    rows.append(r)
EFF = pd.DataFrame(rows); print(EFF.to_string(index=False)); EFF.to_csv(os.path.join(C.TABLES, "efficiency.csv"), index=False)

                   arm  params  GFLOPs_64x64_in  cpu_ms_128  gpu_ms_128
              baseline   86945            0.974       145.4       20.93
          film_control  116138            1.211       160.5       23.99
                  full  116138            1.211       167.3       20.80
            scale_only  115490            1.206       165.7       23.07
          NAFNet-small  112817            0.182        32.2        5.32
       Restormer-small  111917            0.967       109.7       12.89
          SwinIR-small  100813            0.993       134.2       16.16
Restormer-FiLM-control  141110            1.203       178.2       14.38
   Restormer-LDMH-full  141110            1.203       173.6       14.38


## Part 12 - Claims ledger: what may the paper say?
Every rule is applied mechanically to the numbers above. `SUPPORTED` / `NOT SUPPORTED` / `NOT RUN` (the needed runs are missing). Only the **allowed wording** may go into the paper; if a rule fails, the **forbidden wording** must not appear.

In [102]:
def eff(a, b, root=None, dom=None):
    return ALL_EFFECTS.get((dom or os.path.basename(root or C.RUNS), a, b))
def sig_pos(e):  return e is not None and e["n_a"] >= 2 and e["n_b"] >= 2 and e["p_welch"] < 0.05 and e["ci_lo"] > 0
def sig_nz(e):   return e is not None and e["n_a"] >= 2 and e["n_b"] >= 2 and e["p_welch"] < 0.05
def _col(df, arm, col):
    s = df[df.arm == arm]; return s

def have(arm, root=None, n=2): return len(finished_seeds(root or C.RUNS, arm)) >= n
LEDGER = []
def rule(rid, claim, verdict, evidence, allowed, forbidden): LEDGER.append(dict(rule=rid, claim=claim, verdict=verdict, evidence=evidence, allowed_wording=allowed, forbidden_wording=forbidden))
nrun = lambda ok: "SUPPORTED" if ok else "NOT SUPPORTED"

# R1 collapse is real (clamp collapses under stress, sigmoid does not)
if have("full_clamp_stress") and have("full_stress"):
    c = COLL[COLL.arm == "full_clamp_stress"]; s = COLL[COLL.arm == "full_stress"]
    ok = bool(c.collapsed.sum() > 0 and s.collapsed.sum() == 0)
    rule("R1", "Hard-clamped beta collapses; sigmoid beta does not", nrun(ok), f"clamp_stress collapsed {int(c.collapsed.sum())}/{len(c)} seeds; sigmoid_stress {int(s.collapsed.sum())}/{len(s)}",
         "The clamp parameterisation is a failure mode (zero gradient at the ceiling); the sigmoid reparameterisation prevents it under stress.",
         "The proposed loss prevents collapse in general." if not ok else "")
else: rule("R1", "Hard-clamped beta collapses; sigmoid beta does not", "NOT RUN", "need full_clamp_stress and full_stress (Session 2)", "-", "any statement that collapse was demonstrated in the full network")
# R2 balance / diversity needed
if have("full_no_bal") and have("full_no_div") and have("full"):
    cb, cd, cf = (COLL[COLL.arm == a] for a in ("full_no_bal", "full_no_div", "full"))
    nb_ok = bool(cb.collapsed.sum() > 0 and cf.collapsed.sum() == 0); nd_ok = bool(cd.collapsed.sum() > 0 and cf.collapsed.sum() == 0)
    rule("R2a", "Balance penalty is needed to avoid a dead component", nrun(nb_ok), f"no_bal collapsed {int(cb.collapsed.sum())}/{len(cb)}; full {int(cf.collapsed.sum())}/{len(cf)}",
         "Removing the balance penalty produces a dead component." if nb_ok else "The balance penalty did not matter in our setting (report as an ablation, not as a necessary component).",
         "" if nb_ok else "The balance penalty is necessary.")
    rule("R2b", "Diversity penalty is needed to keep betas apart", nrun(nd_ok), f"no_div collapsed {int(cd.collapsed.sum())}/{len(cd)}; full {int(cf.collapsed.sum())}/{len(cf)}",
         "Removing the diversity penalty ties the betas." if nd_ok else "The diversity penalty did not matter in our setting.", "" if nd_ok else "The diversity penalty is necessary.")
else: rule("R2", "Balance / diversity penalties are needed", "NOT RUN", "need full_no_bal, full_no_div (Session 1)", "-", "any necessity claim for the penalties")
# R3 mixture beyond scale
e = eff("full", "scale_only")
if e: ok = sig_pos(e); rule("R3", "The mixture of shapes adds beyond a per-pixel scale", nrun(ok), f"full - scale_only = {e['diff']:+.3f} dB [{e['ci_lo']:+.3f}, {e['ci_hi']:+.3f}], seed-level p={e['p_welch']:.3g}",
    "A learned mixture of shapes improves over a scale-only model." if ok else "Most of the gain is explained by modelling the signal-dependent noise scale; the learned shape mixture adds no detectable benefit beyond it.",
    "" if ok else "The mixture of generalised-Gaussian shapes is responsible for the improvement. / distribution-adaptive shape learning.")
else: rule("R3", "The mixture of shapes adds beyond a per-pixel scale", "NOT RUN", "need scale_only (Session 2)", "-", "any claim that shape mixture matters")
# R4 beats capacity-matched control
e = eff("full", "film_control")
if e: ok = sig_pos(e); rule("R4", "The module beats a capacity-matched control", nrun(ok), f"full - film_control = {e['diff']:+.3f} dB [{e['ci_lo']:+.3f}, {e['ci_hi']:+.3f}], seed-level p={e['p_welch']:.3g}; image-level p={e['p_wilcoxon']:.2g} (not the decision rule)",
    "A small but statistically reliable gain over the capacity-matched control on the compact backbone." if ok else "No reliable gain over the capacity-matched control.", "" if ok else "Any claim of improvement over a fair baseline.")
# R5 strong backbone
for a in ("Restormer-LDMH-full", "Restormer-LDMH-outnll", "Restormer-LDMH-deep"):
    e = eff(a, "Restormer-FiLM-control")
    if e: ok = sig_pos(e); rule("R5:" + a, f"{a} helps a strong backbone", nrun(ok), f"{a} - Restormer-FiLM-control = {e['diff']:+.3f} dB [{e['ci_lo']:+.3f}, {e['ci_hi']:+.3f}], p={e['p_welch']:.3g}",
        "The module transfers to Restormer." if ok else "The module is backbone-specific (does not transfer to Restormer).", "" if ok else "The module is architecture-agnostic / general plug-in.")
if not any(r["rule"].startswith("R5") for r in LEDGER): rule("R5", "Helps a strong backbone", "NOT RUN", "need Restormer-* arms (Session 4)", "-", "any transfer claim")
# R6 domain 2
e1, e2 = eff("full", "film_control", RUNS_D2), eff("full", "baseline", RUNS_D2)
if e1: ok = sig_pos(e1); rule("R7", "Gain on a second noise domain (Poisson-Gaussian)", nrun(ok), f"domain 2: full - film_control = {e1['diff']:+.3f} dB [{e1['ci_lo']:+.3f}, {e1['ci_hi']:+.3f}], p={e1['p_welch']:.3g}",
    "The gain also appears under Poisson-Gaussian noise." if ok else "The gain is specific to the speckle domain / not reproduced in a second noise family.", "" if ok else "Generalises across noise domains.")
else: rule("R7", "Gain on a second noise domain", "NOT RUN", "need domain-2 runs (Session 3)", "-", "any cross-domain claim")
# R8 downstream
if len(DS_DIFF) and (DS_DIFF.contrast == "full - film_control").any():
    r_ = DS_DIFF[DS_DIFF.contrast == "full - film_control"].iloc[0]; ok = bool(r_.lo > 0) and DS_REAL
    why = "" if DS_REAL else " (images are a FALLBACK, not real SEM: cannot be claimed)"
    rule("R6", "Improves defect detection (recall at 1 FA/image)", nrun(ok), f"full - film_control = {r_['diff']:+.3f} [{r_.lo:+.3f}, {r_.hi:+.3f}] over {N_DEF} defects{why}",
         "Restoration gains translate into higher recall at fixed false-alarm rate on a real-image synthetic-defect benchmark." if ok else "Restoration quality gains did not measurably translate into detection (state this plainly).",
         "" if ok else "Improves defect inspection / manufacturing yield.")
else: rule("R6", "Improves defect detection", "NOT RUN", "need full and film_control runs and Part 10", "-", "any downstream claim")
# R9 compact-model efficiency statement
ns = eff("NAFNet-small", "full")
rule("R9", "Compact-model framing", "SUPPORTED" if have("full") else "NOT RUN", "see efficiency table (Part 11) and comparison with Restormer/SwinIR small",
     "A 116k-parameter model that closes a modest part of the gap to larger backbones at a fraction of their cost (quote the efficiency table).", "State of the art / outperforms Restormer or SwinIR.")

LG = pd.DataFrame(LEDGER); print(LG[["rule", "claim", "verdict", "evidence"]].to_string(index=False))
json.dump(dict(ledger=LEDGER, effects=[{**v, "key": list(k)} for k, v in ALL_EFFECTS.items()], real_downstream=DS_REAL, smoke=SMOKE), open(os.path.join(C.TABLES, "claims_ledger.json"), "w"), indent=1, default=str)
md = ["# Claims ledger\n"] + [f"- **{r['rule']}** {r['claim']}: **{r['verdict']}** - {r['evidence']}\n  - allowed: {r['allowed_wording']}\n  - forbidden: {r['forbidden_wording'] or '-'}" for r in LEDGER]
open(os.path.join(C.TABLES, "claims_ledger.md"), "w").write("\n".join(md)); print("\nsaved claims_ledger.json / .md")
if SMOKE: print("\nSMOKE MODE: all numbers are meaningless; only the code path was tested.")

                  rule                                               claim       verdict                                                                                                                evidence
                    R1  Hard-clamped beta collapses; sigmoid beta does not       NOT RUN                                                                      need full_clamp_stress and full_stress (Session 2)
                   R2a Balance penalty is needed to avoid a dead component NOT SUPPORTED                                                                                          no_bal collapsed 0/4; full 0/8
                   R2b     Diversity penalty is needed to keep betas apart NOT SUPPORTED                                                                                          no_div collapsed 0/4; full 0/8
                    R3 The mixture of shapes adds beyond a per-pixel scale     SUPPORTED                                                    full - scale_only = +0.1

## Narrow, honest framing (fill from the ledger; delete every sentence whose rule is not SUPPORTED)

**Title (suggested):** *Signal-dependent noise in semiconductor inspection images: a diagnosis of mixture-head collapse and a compact restoration model*

**Abstract skeleton (<= 150 words).** (1) Problem: restoration of degraded inspection images. (2) We show the residual's apparent shape heterogeneity is explained by a signal-dependent noise scale (hetero-Gaussian null reproduces it). (3) We identify and fix a failure mode (hard-clamped shape parameter -> zero gradient -> tied components) with a sigmoid reparameterisation [only if R1 supported]. (4) A 116k-parameter model gives +X dB over a capacity-matched control (R4) and +Y dB over baseline; [the mixture adds / does not add beyond a scale-only model (R3)]. (5) [Second domain (R7) / backbone transfer (R5)]: say what the ledger says. (6) Detection benchmark: N defects, detector, recall change with CI (R6).

**Contributions.** (a) the signal-dependent-noise diagnosis with null models; (b) the collapse analysis (R1, R2); (c) the controlled comparison incl. capacity-matched and scale-only controls (R3, R4); (d) a reproducible downstream protocol with manifest (R6).

**Limitations paragraph (always).** Single training dataset; seed count; synthetic defects; die-to-die detector is a simple, stated detector; backbone-specific unless R5 is supported.

**Response-letter map.** Comment 1 -> seed-level statistics (Parts 9A-9E); 2 -> downstream manifest + real-image set; 3 -> same-protocol ablations (Part 9A); 4 -> Part 10 + Part 11; 5 -> external backbones + Part 9E + second domain; 6 -> figures regenerated from JSON (Part 5 cells), caption numbers read from saved JSON.

In [103]:
save_now("(final)")
print("\nDeliverables in", C.OUT, ":")
for sub in ("tables", "figures"):
    fs = sorted(os.listdir(os.path.join(C.OUT, sub))); print(f"  {sub}/: {len(fs)} files", fs[:12], "..." if len(fs) > 12 else "")
print("Download results_latest.zip (runs + tables + figures + manifests). It is also the input of the next session.")

saved /kaggle/working/results_latest.zip (77.0 MB) (final)

Deliverables in /kaggle/working/results_pilot :
  tables/: 48 files ['all_effects.csv', 'all_runs_summary.csv', 'architectures.csv', 'arms_summary_domain1.csv', 'carinthia_downstream_manifest.json', 'carinthia_s_manifest.json', 'carinthia_s_paired.csv', 'carinthia_s_selection.csv', 'carinthia_s_summary.csv', 'claim_checks.json', 'claims_ledger.json', 'claims_ledger.md'] ...
  figures/: 110 files ['fig_ablation_test_hp_err.json', 'fig_ablation_test_hp_err.pdf', 'fig_ablation_test_hp_err.png', 'fig_ablation_test_psnr.json', 'fig_ablation_test_psnr.pdf', 'fig_ablation_test_psnr.png', 'fig_ablation_test_ssim.json', 'fig_ablation_test_ssim.pdf', 'fig_ablation_test_ssim.png', 'fig_beta_vs_structure_hetero_null.json', 'fig_beta_vs_structure_hetero_null.pdf', 'fig_beta_vs_structure_hetero_null.png'] ...
Download results_latest.zip (runs + tables + figures + manifests). It is also the input of the next session.


In [104]:
import os
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
display(FileLink("results_latest.zip"))

/kaggle/working/results_latest.zip